# Episode 1 — The AUD rates landscape

Companion notebook for the video. Every number that appears on screen or in the narration is produced by running this notebook and read from `build/outputs.json`.

> **Data.** `data/rba_cash_rate.csv` holds the cash rate target, the cash rate (AONIA) and the Cash Rate Total Return Index from RBA Statistical Table F1. *Source: Reserve Bank of Australia 2026.* The RBA publishes these series free of charge at https://www.rba.gov.au/statistics/tables/ ; this course does not imply any endorsement by the RBA. Other F1 columns (ASX bank bill rates, third-party OIS rates) are not included because they are third-party data.
>
> Educational material only, not investment advice.

1. Setup · 2. Three rates · 3. The cash rate since 2011 · 4. AONIA vs the target · 5. Dates: T+1 and Modified Following · 6. Which product uses which rate · 7. Export

**Running in Google Colab?** Run the next cells first: they install QuantLib (version 1.43, the one used in the video) and write the data file this notebook reads. Then run the rest of the notebook in order.

In [ ]:
# Colab doesn't include QuantLib. Install it (about 30 seconds).
# The video used QuantLib 1.43; drop '==1.43' for the latest.
!pip install QuantLib==1.43

In [ ]:
#@title Data: writes `data/rba_cash_rate.csv` (run me first) { display-mode: "form" }
# Source: Reserve Bank of Australia 2026 (Statistical Table F1: cash rate target, cash rate, total return index).
# The RBA publishes these data free of charge at rba.gov.au. No RBA endorsement is implied.
from pathlib import Path
Path('data/rba_cash_rate.csv').parent.mkdir(parents=True, exist_ok=True)
Path('data/rba_cash_rate.csv').write_text("""date,cash_rate_target,cash_rate,total_return_index
2011-01-04,4.75,4.75,100.000000
2011-01-05,4.75,4.75,100.013014
2011-01-06,4.75,4.75,100.026029
2011-01-07,4.75,4.75,100.039046
2011-01-10,4.75,4.75,100.078103
2011-01-11,4.75,4.75,100.091126
2011-01-12,4.75,4.75,100.104152
2011-01-13,4.75,4.75,100.117179
2011-01-14,4.75,4.75,100.130208
2011-01-17,4.75,4.75,100.169300
2011-01-18,4.75,4.75,100.182336
2011-01-19,4.75,4.75,100.195373
2011-01-20,4.75,4.75,100.208412
2011-01-21,4.75,4.75,100.221453
2011-01-24,4.75,4.75,100.260581
2011-01-25,4.75,4.75,100.273628
2011-01-27,4.75,4.75,100.299727
2011-01-28,4.75,4.75,100.312780
2011-01-31,4.75,4.75,100.351943
2011-02-01,4.75,4.75,100.365002
2011-02-02,4.75,4.75,100.378064
2011-02-03,4.75,4.75,100.391126
2011-02-04,4.75,4.75,100.404191
2011-02-07,4.75,4.75,100.443390
2011-02-08,4.75,4.75,100.456461
2011-02-09,4.75,4.75,100.469534
2011-02-10,4.75,4.75,100.482609
2011-02-11,4.75,4.75,100.495686
2011-02-14,4.75,4.75,100.534920
2011-02-15,4.75,4.75,100.548004
2011-02-16,4.75,4.75,100.561089
2011-02-17,4.75,4.75,100.574175
2011-02-18,4.75,4.75,100.587264
2011-02-21,4.75,4.75,100.626534
2011-02-22,4.75,4.75,100.639629
2011-02-23,4.75,4.75,100.652726
2011-02-24,4.75,4.75,100.665825
2011-02-25,4.75,4.75,100.678925
2011-02-28,4.75,4.75,100.718232
2011-03-01,4.75,4.75,100.731339
2011-03-02,4.75,4.75,100.744448
2011-03-03,4.75,4.75,100.757558
2011-03-04,4.75,4.75,100.770670
2011-03-07,4.75,4.75,100.810012
2011-03-08,4.75,4.75,100.823132
2011-03-09,4.75,4.75,100.836252
2011-03-10,4.75,4.75,100.849375
2011-03-11,4.75,4.75,100.862499
2011-03-14,4.75,4.75,100.901877
2011-03-15,4.75,4.75,100.915008
2011-03-16,4.75,4.75,100.928141
2011-03-17,4.75,4.75,100.941275
2011-03-18,4.75,4.75,100.954411
2011-03-21,4.75,4.75,100.993825
2011-03-22,4.75,4.75,101.006968
2011-03-23,4.75,4.75,101.020113
2011-03-24,4.75,4.75,101.033259
2011-03-25,4.75,4.75,101.046408
2011-03-28,4.75,4.75,101.085857
2011-03-29,4.75,4.75,101.099012
2011-03-30,4.75,4.75,101.112169
2011-03-31,4.75,4.75,101.125327
2011-04-01,4.75,4.75,101.138487
2011-04-04,4.75,4.75,101.177973
2011-04-05,4.75,4.75,101.191140
2011-04-06,4.75,4.75,101.204309
2011-04-07,4.75,4.75,101.217479
2011-04-08,4.75,4.75,101.230651
2011-04-11,4.75,4.75,101.270173
2011-04-12,4.75,4.75,101.283352
2011-04-13,4.75,4.75,101.296533
2011-04-14,4.75,4.75,101.309715
2011-04-15,4.75,4.75,101.322899
2011-04-18,4.75,4.75,101.362457
2011-04-19,4.75,4.75,101.375648
2011-04-20,4.75,4.75,101.388840
2011-04-21,4.75,4.75,101.402035
2011-04-27,4.75,4.75,101.481212
2011-04-28,4.75,4.75,101.494418
2011-04-29,4.75,4.75,101.507626
2011-05-02,4.75,4.75,101.547256
2011-05-03,4.75,4.75,101.560471
2011-05-04,4.75,4.75,101.573688
2011-05-05,4.75,4.75,101.586906
2011-05-06,4.75,4.75,101.600127
2011-05-09,4.75,4.75,101.639792
2011-05-10,4.75,4.75,101.653020
2011-05-11,4.75,4.75,101.666248
2011-05-12,4.75,4.75,101.679479
2011-05-13,4.75,4.75,101.692711
2011-05-16,4.75,4.75,101.732413
2011-05-17,4.75,4.75,101.745652
2011-05-18,4.75,4.75,101.758893
2011-05-19,4.75,4.75,101.772136
2011-05-20,4.75,4.75,101.785380
2011-05-23,4.75,4.75,101.825118
2011-05-24,4.75,4.75,101.838369
2011-05-25,4.75,4.75,101.851622
2011-05-26,4.75,4.75,101.864877
2011-05-27,4.75,4.75,101.878133
2011-05-30,4.75,4.75,101.917908
2011-05-31,4.75,4.75,101.931171
2011-06-01,4.75,4.75,101.944436
2011-06-02,4.75,4.75,101.957703
2011-06-03,4.75,4.75,101.970971
2011-06-06,4.75,4.75,102.010782
2011-06-07,4.75,4.75,102.024057
2011-06-08,4.75,4.75,102.037334
2011-06-09,4.75,4.75,102.050613
2011-06-10,4.75,4.75,102.063894
2011-06-14,4.75,4.75,102.117023
2011-06-15,4.75,4.75,102.130312
2011-06-16,4.75,4.75,102.143603
2011-06-17,4.75,4.75,102.156896
2011-06-20,4.75,4.75,102.196779
2011-06-21,4.75,4.75,102.210078
2011-06-22,4.75,4.75,102.223380
2011-06-23,4.75,4.75,102.236683
2011-06-24,4.75,4.75,102.249988
2011-06-27,4.75,4.75,102.289907
2011-06-28,4.75,4.75,102.303219
2011-06-29,4.75,4.75,102.316532
2011-06-30,4.75,4.75,102.329847
2011-07-01,4.75,4.75,102.343164
2011-07-04,4.75,4.75,102.383120
2011-07-05,4.75,4.75,102.396444
2011-07-06,4.75,4.75,102.409770
2011-07-07,4.75,4.75,102.423097
2011-07-08,4.75,4.75,102.436426
2011-07-11,4.75,4.75,102.476418
2011-07-12,4.75,4.75,102.489754
2011-07-13,4.75,4.75,102.503092
2011-07-14,4.75,4.75,102.516431
2011-07-15,4.75,4.75,102.529772
2011-07-18,4.75,4.75,102.569801
2011-07-19,4.75,4.75,102.583149
2011-07-20,4.75,4.75,102.596499
2011-07-21,4.75,4.75,102.609851
2011-07-22,4.75,4.75,102.623204
2011-07-25,4.75,4.75,102.663269
2011-07-26,4.75,4.75,102.676630
2011-07-27,4.75,4.75,102.689992
2011-07-28,4.75,4.75,102.703355
2011-07-29,4.75,4.75,102.716721
2011-08-01,4.75,4.75,102.756823
2011-08-02,4.75,4.75,102.770195
2011-08-03,4.75,4.75,102.783569
2011-08-04,4.75,4.75,102.796945
2011-08-05,4.75,4.75,102.810323
2011-08-08,4.75,4.75,102.850461
2011-08-09,4.75,4.75,102.863846
2011-08-10,4.75,4.75,102.877232
2011-08-11,4.75,4.75,102.890620
2011-08-12,4.75,4.75,102.904010
2011-08-15,4.75,4.75,102.944185
2011-08-16,4.75,4.75,102.957582
2011-08-17,4.75,4.75,102.970981
2011-08-18,4.75,4.75,102.984381
2011-08-19,4.75,4.75,102.997783
2011-08-22,4.75,4.75,103.037994
2011-08-23,4.75,4.75,103.051404
2011-08-24,4.75,4.75,103.064814
2011-08-25,4.75,4.75,103.078227
2011-08-26,4.75,4.75,103.091641
2011-08-29,4.75,4.75,103.131889
2011-08-30,4.75,4.75,103.145311
2011-08-31,4.75,4.75,103.158734
2011-09-01,4.75,4.75,103.172158
2011-09-02,4.75,4.75,103.185585
2011-09-05,4.75,4.75,103.225870
2011-09-06,4.75,4.75,103.239303
2011-09-07,4.75,4.75,103.252738
2011-09-08,4.75,4.75,103.266175
2011-09-09,4.75,4.75,103.279614
2011-09-12,4.75,4.75,103.319936
2011-09-13,4.75,4.75,103.333381
2011-09-14,4.75,4.75,103.346829
2011-09-15,4.75,4.75,103.360278
2011-09-16,4.75,4.75,103.373729
2011-09-19,4.75,4.75,103.414087
2011-09-20,4.75,4.75,103.427545
2011-09-21,4.75,4.75,103.441005
2011-09-22,4.75,4.75,103.454467
2011-09-23,4.75,4.75,103.467930
2011-09-26,4.75,4.75,103.508325
2011-09-27,4.75,4.75,103.521795
2011-09-28,4.75,4.75,103.535267
2011-09-29,4.75,4.75,103.548741
2011-09-30,4.75,4.75,103.562216
2011-10-03,4.75,4.75,103.602648
2011-10-04,4.75,4.75,103.616131
2011-10-05,4.75,4.75,103.629615
2011-10-06,4.75,4.75,103.643101
2011-10-07,4.75,4.75,103.656589
2011-10-10,4.75,4.75,103.697058
2011-10-11,4.75,4.75,103.710552
2011-10-12,4.75,4.75,103.724049
2011-10-13,4.75,4.75,103.737547
2011-10-14,4.75,4.75,103.751047
2011-10-17,4.75,4.75,103.791553
2011-10-18,4.75,4.75,103.805060
2011-10-19,4.75,4.75,103.818569
2011-10-20,4.75,4.75,103.832080
2011-10-21,4.75,4.75,103.845592
2011-10-24,4.75,4.75,103.886134
2011-10-25,4.75,4.75,103.899654
2011-10-26,4.75,4.75,103.913175
2011-10-27,4.75,4.75,103.926698
2011-10-28,4.75,4.75,103.940223
2011-10-31,4.75,4.75,103.980802
2011-11-01,4.75,4.75,103.994334
2011-11-02,4.50,4.50,104.007867
2011-11-03,4.50,4.50,104.020690
2011-11-04,4.50,4.50,104.033515
2011-11-07,4.50,4.50,104.071993
2011-11-08,4.50,4.50,104.084824
2011-11-09,4.50,4.50,104.097656
2011-11-10,4.50,4.50,104.110490
2011-11-11,4.50,4.50,104.123326
2011-11-14,4.50,4.50,104.161837
2011-11-15,4.50,4.50,104.174679
2011-11-16,4.50,4.50,104.187522
2011-11-17,4.50,4.50,104.200367
2011-11-18,4.50,4.50,104.213214
2011-11-21,4.50,4.50,104.251759
2011-11-22,4.50,4.50,104.264611
2011-11-23,4.50,4.50,104.277466
2011-11-24,4.50,4.50,104.290322
2011-11-25,4.50,4.50,104.303180
2011-11-28,4.50,4.50,104.341758
2011-11-29,4.50,4.50,104.354622
2011-11-30,4.50,4.50,104.367487
2011-12-01,4.50,4.50,104.380355
2011-12-02,4.50,4.50,104.393223
2011-12-05,4.50,4.50,104.431835
2011-12-06,4.50,4.50,104.444710
2011-12-07,4.25,4.25,104.457587
2011-12-08,4.25,4.25,104.469749
2011-12-09,4.25,4.25,104.481914
2011-12-12,4.25,4.25,104.518411
2011-12-13,4.25,4.25,104.530581
2011-12-14,4.25,4.25,104.542752
2011-12-15,4.25,4.25,104.554925
2011-12-16,4.25,4.25,104.567099
2011-12-19,4.25,4.25,104.603626
2011-12-20,4.25,4.25,104.615806
2011-12-21,4.25,4.25,104.627987
2011-12-22,4.25,4.25,104.640170
2011-12-23,4.25,4.25,104.652354
2011-12-28,4.25,4.25,104.713282
2011-12-29,4.25,4.25,104.725474
2011-12-30,4.25,4.25,104.737668
2012-01-03,4.25,4.25,104.786450
2012-01-04,4.25,4.25,104.798652
2012-01-05,4.25,4.25,104.810854
2012-01-06,4.25,4.25,104.823058
2012-01-09,4.25,4.25,104.859674
2012-01-10,4.25,4.25,104.871884
2012-01-11,4.25,4.25,104.884095
2012-01-12,4.25,4.25,104.896308
2012-01-13,4.25,4.25,104.908522
2012-01-16,4.25,4.25,104.945168
2012-01-17,4.25,4.25,104.957387
2012-01-18,4.25,4.25,104.969609
2012-01-19,4.25,4.25,104.981831
2012-01-20,4.25,4.25,104.994055
2012-01-23,4.25,4.25,105.030731
2012-01-24,4.25,4.25,105.042961
2012-01-25,4.25,4.25,105.055192
2012-01-27,4.25,4.25,105.079656
2012-01-30,4.25,4.25,105.116362
2012-01-31,4.25,4.25,105.128602
2012-02-01,4.25,4.25,105.140843
2012-02-02,4.25,4.25,105.153085
2012-02-03,4.25,4.25,105.165329
2012-02-06,4.25,4.25,105.202065
2012-02-07,4.25,4.25,105.214315
2012-02-08,4.25,4.25,105.226566
2012-02-09,4.25,4.25,105.238818
2012-02-10,4.25,4.25,105.251072
2012-02-13,4.25,4.25,105.287838
2012-02-14,4.25,4.25,105.300097
2012-02-15,4.25,4.25,105.312358
2012-02-16,4.25,4.25,105.324621
2012-02-17,4.25,4.25,105.336884
2012-02-20,4.25,4.25,105.373680
2012-02-21,4.25,4.25,105.385950
2012-02-22,4.25,4.25,105.398221
2012-02-23,4.25,4.25,105.410493
2012-02-24,4.25,4.25,105.422767
2012-02-27,4.25,4.25,105.459593
2012-02-28,4.25,4.25,105.471872
2012-02-29,4.25,4.25,105.484153
2012-03-01,4.25,4.25,105.496436
2012-03-02,4.25,4.25,105.508719
2012-03-05,4.25,4.25,105.545575
2012-03-06,4.25,4.25,105.557865
2012-03-07,4.25,4.25,105.570156
2012-03-08,4.25,4.25,105.582448
2012-03-09,4.25,4.25,105.594742
2012-03-12,4.25,4.25,105.631628
2012-03-13,4.25,4.25,105.643927
2012-03-14,4.25,4.25,105.656228
2012-03-15,4.25,4.25,105.668531
2012-03-16,4.25,4.25,105.680835
2012-03-19,4.25,4.25,105.717751
2012-03-20,4.25,4.25,105.730060
2012-03-21,4.25,4.25,105.742371
2012-03-22,4.25,4.25,105.754684
2012-03-23,4.25,4.25,105.766998
2012-03-26,4.25,4.25,105.803944
2012-03-27,4.25,4.25,105.816263
2012-03-28,4.25,4.25,105.828584
2012-03-29,4.25,4.25,105.840907
2012-03-30,4.25,4.25,105.853231
2012-04-02,4.25,4.25,105.890207
2012-04-03,4.25,4.25,105.902537
2012-04-04,4.25,4.25,105.914868
2012-04-05,4.25,4.25,105.927200
2012-04-10,4.25,4.25,105.988870
2012-04-11,4.25,4.25,106.001211
2012-04-12,4.25,4.25,106.013554
2012-04-13,4.25,4.25,106.025898
2012-04-16,4.25,4.25,106.062934
2012-04-17,4.25,4.25,106.075284
2012-04-18,4.25,4.25,106.087635
2012-04-19,4.25,4.25,106.099988
2012-04-20,4.25,4.25,106.112342
2012-04-23,4.25,4.25,106.149409
2012-04-24,4.25,4.25,106.161769
2012-04-26,4.25,4.25,106.186491
2012-04-27,4.25,4.25,106.198856
2012-04-30,4.25,4.25,106.235952
2012-05-01,4.25,4.25,106.248322
2012-05-02,3.75,3.75,106.260694
2012-05-03,3.75,3.75,106.271611
2012-05-04,3.75,3.75,106.282529
2012-05-07,3.75,3.75,106.315288
2012-05-08,3.75,3.75,106.326210
2012-05-09,3.75,3.75,106.337134
2012-05-10,3.75,3.75,106.348059
2012-05-11,3.75,3.75,106.358985
2012-05-14,3.75,3.75,106.391767
2012-05-15,3.75,3.75,106.402698
2012-05-16,3.75,3.75,106.413630
2012-05-17,3.75,3.75,106.424563
2012-05-18,3.75,3.75,106.435497
2012-05-21,3.75,3.75,106.468302
2012-05-22,3.75,3.75,106.479241
2012-05-23,3.75,3.75,106.490180
2012-05-24,3.75,3.75,106.501121
2012-05-25,3.75,3.75,106.512063
2012-05-28,3.75,3.75,106.544892
2012-05-29,3.75,3.75,106.555838
2012-05-30,3.75,3.75,106.566786
2012-05-31,3.75,3.75,106.577735
2012-06-01,3.75,3.75,106.588684
2012-06-04,3.75,3.75,106.621537
2012-06-05,3.75,3.75,106.632491
2012-06-06,3.50,3.50,106.643447
2012-06-07,3.50,3.50,106.653673
2012-06-08,3.50,3.50,106.663900
2012-06-12,3.50,3.50,106.704812
2012-06-13,3.50,3.50,106.715044
2012-06-14,3.50,3.50,106.725277
2012-06-15,3.50,3.50,106.735511
2012-06-18,3.50,3.50,106.766216
2012-06-19,3.50,3.50,106.776454
2012-06-20,3.50,3.50,106.786692
2012-06-21,3.50,3.50,106.796932
2012-06-22,3.50,3.50,106.807173
2012-06-25,3.50,3.50,106.837898
2012-06-26,3.50,3.50,106.848143
2012-06-27,3.50,3.50,106.858389
2012-06-28,3.50,3.50,106.868635
2012-06-29,3.50,3.50,106.878883
2012-07-02,3.50,3.50,106.909629
2012-07-03,3.50,3.50,106.919881
2012-07-04,3.50,3.50,106.930133
2012-07-05,3.50,3.50,106.940387
2012-07-06,3.50,3.50,106.950641
2012-07-09,3.50,3.50,106.981408
2012-07-10,3.50,3.50,106.991667
2012-07-11,3.50,3.50,107.001926
2012-07-12,3.50,3.50,107.012186
2012-07-13,3.50,3.50,107.022448
2012-07-16,3.50,3.50,107.053235
2012-07-17,3.50,3.50,107.063501
2012-07-18,3.50,3.50,107.073767
2012-07-19,3.50,3.50,107.084034
2012-07-20,3.50,3.50,107.094303
2012-07-23,3.50,3.50,107.125111
2012-07-24,3.50,3.50,107.135383
2012-07-25,3.50,3.50,107.145656
2012-07-26,3.50,3.50,107.155930
2012-07-27,3.50,3.50,107.166206
2012-07-30,3.50,3.50,107.197034
2012-07-31,3.50,3.50,107.207313
2012-08-01,3.50,3.50,107.217594
2012-08-02,3.50,3.50,107.227875
2012-08-03,3.50,3.50,107.238157
2012-08-06,3.50,3.50,107.269006
2012-08-07,3.50,3.50,107.279292
2012-08-08,3.50,3.50,107.289579
2012-08-09,3.50,3.50,107.299867
2012-08-10,3.50,3.50,107.310156
2012-08-13,3.50,3.50,107.341026
2012-08-14,3.50,3.50,107.351319
2012-08-15,3.50,3.50,107.361613
2012-08-16,3.50,3.50,107.371908
2012-08-17,3.50,3.50,107.382204
2012-08-20,3.50,3.50,107.413095
2012-08-21,3.50,3.50,107.423395
2012-08-22,3.50,3.50,107.433696
2012-08-23,3.50,3.50,107.443998
2012-08-24,3.50,3.50,107.454300
2012-08-27,3.50,3.50,107.485212
2012-08-28,3.50,3.50,107.495519
2012-08-29,3.50,3.50,107.505827
2012-08-30,3.50,3.50,107.516135
2012-08-31,3.50,3.50,107.526445
2012-09-03,3.50,3.50,107.557377
2012-09-04,3.50,3.50,107.567691
2012-09-05,3.50,3.50,107.578006
2012-09-06,3.50,3.50,107.588321
2012-09-07,3.50,3.50,107.598638
2012-09-10,3.50,3.50,107.629591
2012-09-11,3.50,3.50,107.639912
2012-09-12,3.50,3.50,107.650233
2012-09-13,3.50,3.50,107.660556
2012-09-14,3.50,3.50,107.670880
2012-09-17,3.50,3.50,107.701854
2012-09-18,3.50,3.50,107.712181
2012-09-19,3.50,3.50,107.722510
2012-09-20,3.50,3.50,107.732839
2012-09-21,3.50,3.50,107.743170
2012-09-24,3.50,3.50,107.774164
2012-09-25,3.50,3.50,107.784499
2012-09-26,3.50,3.50,107.794834
2012-09-27,3.50,3.50,107.805171
2012-09-28,3.50,3.50,107.815508
2012-10-01,3.50,3.50,107.846524
2012-10-02,3.50,3.50,107.856865
2012-10-03,3.25,3.25,107.867208
2012-10-04,3.25,3.25,107.876812
2012-10-05,3.25,3.25,107.886418
2012-10-08,3.25,3.25,107.915237
2012-10-09,3.25,3.25,107.924846
2012-10-10,3.25,3.25,107.934455
2012-10-11,3.25,3.25,107.944066
2012-10-12,3.25,3.25,107.953677
2012-10-15,3.25,3.25,107.982514
2012-10-16,3.25,3.25,107.992129
2012-10-17,3.25,3.25,108.001745
2012-10-18,3.25,3.25,108.011362
2012-10-19,3.25,3.25,108.020979
2012-10-22,3.25,3.25,108.049834
2012-10-23,3.25,3.25,108.059455
2012-10-24,3.25,3.25,108.069077
2012-10-25,3.25,3.25,108.078699
2012-10-26,3.25,3.25,108.088323
2012-10-29,3.25,3.25,108.117196
2012-10-30,3.25,3.25,108.126822
2012-10-31,3.25,3.25,108.136450
2012-11-01,3.25,3.25,108.146079
2012-11-02,3.25,3.25,108.155708
2012-11-05,3.25,3.25,108.184599
2012-11-06,3.25,3.25,108.194232
2012-11-07,3.25,3.25,108.203866
2012-11-08,3.25,3.25,108.213500
2012-11-09,3.25,3.25,108.223136
2012-11-12,3.25,3.25,108.252045
2012-11-13,3.25,3.25,108.261684
2012-11-14,3.25,3.25,108.271323
2012-11-15,3.25,3.25,108.280964
2012-11-16,3.25,3.25,108.290605
2012-11-19,3.25,3.25,108.319532
2012-11-20,3.25,3.25,108.329177
2012-11-21,3.25,3.25,108.338823
2012-11-22,3.25,3.25,108.348469
2012-11-23,3.25,3.25,108.358117
2012-11-26,3.25,3.25,108.387062
2012-11-27,3.25,3.25,108.396713
2012-11-28,3.25,3.25,108.406365
2012-11-29,3.25,3.25,108.416017
2012-11-30,3.25,3.25,108.425671
2012-12-03,3.25,3.25,108.454634
2012-12-04,3.25,3.25,108.464291
2012-12-05,3.00,3.00,108.473948
2012-12-06,3.00,3.00,108.482864
2012-12-07,3.00,3.00,108.491780
2012-12-10,3.00,3.00,108.518532
2012-12-11,3.00,3.00,108.527451
2012-12-12,3.00,3.00,108.536371
2012-12-13,3.00,3.00,108.545292
2012-12-14,3.00,3.00,108.554214
2012-12-17,3.00,3.00,108.580980
2012-12-18,3.00,3.00,108.589905
2012-12-19,3.00,3.00,108.598830
2012-12-20,3.00,3.00,108.607756
2012-12-21,3.00,3.00,108.616683
2012-12-24,3.00,3.00,108.643465
2012-12-27,3.00,3.00,108.670254
2012-12-28,3.00,3.00,108.679185
2012-12-31,3.00,3.00,108.705983
2013-01-02,3.00,3.00,108.723853
2013-01-03,3.00,3.00,108.732789
2013-01-04,3.00,3.00,108.741726
2013-01-07,3.00,3.00,108.768539
2013-01-08,3.00,3.00,108.777479
2013-01-09,3.00,3.00,108.786419
2013-01-10,3.00,3.00,108.795361
2013-01-11,3.00,3.00,108.804303
2013-01-14,3.00,3.00,108.831131
2013-01-15,3.00,3.00,108.840076
2013-01-16,3.00,3.00,108.849022
2013-01-17,3.00,3.00,108.857968
2013-01-18,3.00,3.00,108.866916
2013-01-21,3.00,3.00,108.893760
2013-01-22,3.00,3.00,108.902710
2013-01-23,3.00,3.00,108.911661
2013-01-24,3.00,3.00,108.920612
2013-01-25,3.00,3.00,108.929565
2013-01-29,3.00,3.00,108.965377
2013-01-30,3.00,3.00,108.974333
2013-01-31,3.00,3.00,108.983290
2013-02-01,3.00,3.00,108.992247
2013-02-04,3.00,3.00,109.019122
2013-02-05,3.00,3.00,109.028083
2013-02-06,3.00,3.00,109.037044
2013-02-07,3.00,3.00,109.046006
2013-02-08,3.00,3.00,109.054969
2013-02-11,3.00,3.00,109.081859
2013-02-12,3.00,3.00,109.090824
2013-02-13,3.00,3.00,109.099791
2013-02-14,3.00,3.00,109.108758
2013-02-15,3.00,3.00,109.117726
2013-02-18,3.00,3.00,109.144632
2013-02-19,3.00,3.00,109.153602
2013-02-20,3.00,3.00,109.162574
2013-02-21,3.00,3.00,109.171546
2013-02-22,3.00,3.00,109.180519
2013-02-25,3.00,3.00,109.207440
2013-02-26,3.00,3.00,109.216416
2013-02-27,3.00,3.00,109.225393
2013-02-28,3.00,3.00,109.234370
2013-03-01,3.00,3.00,109.243349
2013-03-04,3.00,3.00,109.270285
2013-03-05,3.00,3.00,109.279266
2013-03-06,3.00,3.00,109.288248
2013-03-07,3.00,3.00,109.297231
2013-03-08,3.00,3.00,109.306214
2013-03-11,3.00,3.00,109.333166
2013-03-12,3.00,3.00,109.342153
2013-03-13,3.00,3.00,109.351140
2013-03-14,3.00,3.00,109.360128
2013-03-15,3.00,3.00,109.369116
2013-03-18,3.00,3.00,109.396084
2013-03-19,3.00,3.00,109.405075
2013-03-20,3.00,3.00,109.414067
2013-03-21,3.00,3.00,109.423060
2013-03-22,3.00,3.00,109.432054
2013-03-25,3.00,3.00,109.459037
2013-03-26,3.00,3.00,109.468034
2013-03-27,3.00,3.00,109.477031
2013-03-28,3.00,3.00,109.486029
2013-04-02,3.00,3.00,109.531024
2013-04-03,3.00,3.00,109.540026
2013-04-04,3.00,3.00,109.549029
2013-04-05,3.00,3.00,109.558034
2013-04-08,3.00,3.00,109.585048
2013-04-09,3.00,3.00,109.594055
2013-04-10,3.00,3.00,109.603063
2013-04-11,3.00,3.00,109.612071
2013-04-12,3.00,3.00,109.621080
2013-04-15,3.00,3.00,109.648110
2013-04-16,3.00,3.00,109.657122
2013-04-17,3.00,3.00,109.666135
2013-04-18,3.00,3.00,109.675149
2013-04-19,3.00,3.00,109.684163
2013-04-22,3.00,3.00,109.711209
2013-04-23,3.00,3.00,109.720226
2013-04-24,3.00,3.00,109.729244
2013-04-26,3.00,3.00,109.747282
2013-04-29,3.00,3.00,109.774343
2013-04-30,3.00,3.00,109.783365
2013-05-01,3.00,3.00,109.792389
2013-05-02,3.00,3.00,109.801413
2013-05-03,3.00,3.00,109.810437
2013-05-06,3.00,3.00,109.837514
2013-05-07,3.00,3.00,109.846542
2013-05-08,2.75,2.75,109.855570
2013-05-09,2.75,2.75,109.863847
2013-05-10,2.75,2.75,109.872124
2013-05-13,2.75,2.75,109.896958
2013-05-14,2.75,2.75,109.905238
2013-05-15,2.75,2.75,109.913519
2013-05-16,2.75,2.75,109.921800
2013-05-17,2.75,2.75,109.930082
2013-05-20,2.75,2.75,109.954929
2013-05-21,2.75,2.75,109.963213
2013-05-22,2.75,2.75,109.971498
2013-05-23,2.75,2.75,109.979784
2013-05-24,2.75,2.75,109.988070
2013-05-27,2.75,2.75,110.012930
2013-05-28,2.75,2.75,110.021219
2013-05-29,2.75,2.75,110.029508
2013-05-30,2.75,2.75,110.037798
2013-05-31,2.75,2.75,110.046089
2013-06-03,2.75,2.75,110.070962
2013-06-04,2.75,2.75,110.079255
2013-06-05,2.75,2.75,110.087549
2013-06-06,2.75,2.75,110.095843
2013-06-07,2.75,2.75,110.104138
2013-06-11,2.75,2.75,110.137320
2013-06-12,2.75,2.75,110.145618
2013-06-13,2.75,2.75,110.153917
2013-06-14,2.75,2.75,110.162216
2013-06-17,2.75,2.75,110.187115
2013-06-18,2.75,2.75,110.195417
2013-06-19,2.75,2.75,110.203720
2013-06-20,2.75,2.75,110.212023
2013-06-21,2.75,2.75,110.220326
2013-06-24,2.75,2.75,110.245239
2013-06-25,2.75,2.75,110.253545
2013-06-26,2.75,2.75,110.261852
2013-06-27,2.75,2.75,110.270159
2013-06-28,2.75,2.75,110.278467
2013-07-01,2.75,2.75,110.303393
2013-07-02,2.75,2.75,110.311704
2013-07-03,2.75,2.75,110.320015
2013-07-04,2.75,2.75,110.328327
2013-07-05,2.75,2.75,110.336639
2013-07-08,2.75,2.75,110.361578
2013-07-09,2.75,2.75,110.369893
2013-07-10,2.75,2.75,110.378209
2013-07-11,2.75,2.75,110.386525
2013-07-12,2.75,2.75,110.394842
2013-07-15,2.75,2.75,110.419794
2013-07-16,2.75,2.75,110.428113
2013-07-17,2.75,2.75,110.436433
2013-07-18,2.75,2.75,110.444754
2013-07-19,2.75,2.75,110.453075
2013-07-22,2.75,2.75,110.478040
2013-07-23,2.75,2.75,110.486364
2013-07-24,2.75,2.75,110.494688
2013-07-25,2.75,2.75,110.503013
2013-07-26,2.75,2.75,110.511339
2013-07-29,2.75,2.75,110.536318
2013-07-30,2.75,2.75,110.544646
2013-07-31,2.75,2.75,110.552974
2013-08-01,2.75,2.75,110.561304
2013-08-02,2.75,2.75,110.569634
2013-08-05,2.75,2.75,110.594625
2013-08-06,2.75,2.75,110.602958
2013-08-07,2.50,2.50,110.611291
2013-08-08,2.50,2.50,110.618867
2013-08-09,2.50,2.50,110.626444
2013-08-12,2.50,2.50,110.649175
2013-08-13,2.50,2.50,110.656754
2013-08-14,2.50,2.50,110.664333
2013-08-15,2.50,2.50,110.671913
2013-08-16,2.50,2.50,110.679493
2013-08-19,2.50,2.50,110.702235
2013-08-20,2.50,2.50,110.709818
2013-08-21,2.50,2.50,110.717401
2013-08-22,2.50,2.50,110.724984
2013-08-23,2.50,2.50,110.732568
2013-08-26,2.50,2.50,110.755321
2013-08-27,2.50,2.50,110.762907
2013-08-28,2.50,2.50,110.770494
2013-08-29,2.50,2.50,110.778081
2013-08-30,2.50,2.50,110.785668
2013-09-02,2.50,2.50,110.808432
2013-09-03,2.50,2.50,110.816022
2013-09-04,2.50,2.50,110.823612
2013-09-05,2.50,2.50,110.831203
2013-09-06,2.50,2.50,110.838794
2013-09-09,2.50,2.50,110.861569
2013-09-10,2.50,2.50,110.869162
2013-09-11,2.50,2.50,110.876756
2013-09-12,2.50,2.50,110.884351
2013-09-13,2.50,2.50,110.891945
2013-09-16,2.50,2.50,110.914731
2013-09-17,2.50,2.50,110.922328
2013-09-18,2.50,2.50,110.929926
2013-09-19,2.50,2.50,110.937524
2013-09-20,2.50,2.50,110.945122
2013-09-23,2.50,2.50,110.967919
2013-09-24,2.50,2.50,110.975520
2013-09-25,2.50,2.50,110.983121
2013-09-26,2.50,2.50,110.990722
2013-09-27,2.50,2.50,110.998324
2013-09-30,2.50,2.50,111.021132
2013-10-01,2.50,2.50,111.028736
2013-10-02,2.50,2.50,111.036341
2013-10-03,2.50,2.50,111.043946
2013-10-04,2.50,2.50,111.051552
2013-10-07,2.50,2.50,111.074371
2013-10-08,2.50,2.50,111.081979
2013-10-09,2.50,2.50,111.089587
2013-10-10,2.50,2.50,111.097196
2013-10-11,2.50,2.50,111.104805
2013-10-14,2.50,2.50,111.127635
2013-10-15,2.50,2.50,111.135247
2013-10-16,2.50,2.50,111.142859
2013-10-17,2.50,2.50,111.150471
2013-10-18,2.50,2.50,111.158084
2013-10-21,2.50,2.50,111.180925
2013-10-22,2.50,2.50,111.188540
2013-10-23,2.50,2.50,111.196156
2013-10-24,2.50,2.50,111.203772
2013-10-25,2.50,2.50,111.211388
2013-10-28,2.50,2.50,111.234240
2013-10-29,2.50,2.50,111.241859
2013-10-30,2.50,2.50,111.249478
2013-10-31,2.50,2.50,111.257098
2013-11-01,2.50,2.50,111.264718
2013-11-04,2.50,2.50,111.287581
2013-11-05,2.50,2.50,111.295203
2013-11-06,2.50,2.50,111.302826
2013-11-07,2.50,2.50,111.310450
2013-11-08,2.50,2.50,111.318074
2013-11-11,2.50,2.50,111.340947
2013-11-12,2.50,2.50,111.348574
2013-11-13,2.50,2.50,111.356200
2013-11-14,2.50,2.50,111.363827
2013-11-15,2.50,2.50,111.371455
2013-11-18,2.50,2.50,111.394340
2013-11-19,2.50,2.50,111.401969
2013-11-20,2.50,2.50,111.409600
2013-11-21,2.50,2.50,111.417230
2013-11-22,2.50,2.50,111.424862
2013-11-25,2.50,2.50,111.447757
2013-11-26,2.50,2.50,111.455391
2013-11-27,2.50,2.50,111.463025
2013-11-28,2.50,2.50,111.470659
2013-11-29,2.50,2.50,111.478294
2013-12-02,2.50,2.50,111.501200
2013-12-03,2.50,2.50,111.508838
2013-12-04,2.50,2.50,111.516475
2013-12-05,2.50,2.50,111.524113
2013-12-06,2.50,2.50,111.531752
2013-12-09,2.50,2.50,111.554669
2013-12-10,2.50,2.50,111.562310
2013-12-11,2.50,2.50,111.569951
2013-12-12,2.50,2.50,111.577593
2013-12-13,2.50,2.50,111.585235
2013-12-16,2.50,2.50,111.608164
2013-12-17,2.50,2.50,111.615808
2013-12-18,2.50,2.50,111.623453
2013-12-19,2.50,2.50,111.631099
2013-12-20,2.50,2.50,111.638745
2013-12-23,2.50,2.50,111.661684
2013-12-24,2.50,2.50,111.669332
2013-12-27,2.50,2.50,111.692278
2013-12-30,2.50,2.50,111.715228
2013-12-31,2.50,2.50,111.722880
2014-01-02,2.50,2.50,111.738185
2014-01-03,2.50,2.50,111.745838
2014-01-06,2.50,2.50,111.768799
2014-01-07,2.50,2.50,111.776455
2014-01-08,2.50,2.50,111.784111
2014-01-09,2.50,2.50,111.791767
2014-01-10,2.50,2.50,111.799424
2014-01-13,2.50,2.50,111.822397
2014-01-14,2.50,2.50,111.830056
2014-01-15,2.50,2.50,111.837715
2014-01-16,2.50,2.50,111.845375
2014-01-17,2.50,2.50,111.853036
2014-01-20,2.50,2.50,111.876019
2014-01-21,2.50,2.50,111.883682
2014-01-22,2.50,2.50,111.891346
2014-01-23,2.50,2.50,111.899009
2014-01-24,2.50,2.50,111.906674
2014-01-28,2.50,2.50,111.937333
2014-01-29,2.50,2.50,111.945000
2014-01-30,2.50,2.50,111.952667
2014-01-31,2.50,2.50,111.960335
2014-02-03,2.50,2.50,111.983341
2014-02-04,2.50,2.50,111.991011
2014-02-05,2.50,2.50,111.998682
2014-02-06,2.50,2.50,112.006353
2014-02-07,2.50,2.50,112.014024
2014-02-10,2.50,2.50,112.037041
2014-02-11,2.50,2.50,112.044715
2014-02-12,2.50,2.50,112.052389
2014-02-13,2.50,2.50,112.060064
2014-02-14,2.50,2.50,112.067739
2014-02-17,2.50,2.50,112.090767
2014-02-18,2.50,2.50,112.098444
2014-02-19,2.50,2.50,112.106122
2014-02-20,2.50,2.50,112.113801
2014-02-21,2.50,2.50,112.121480
2014-02-24,2.50,2.50,112.144518
2014-02-25,2.50,2.50,112.152200
2014-02-26,2.50,2.50,112.159881
2014-02-27,2.50,2.50,112.167563
2014-02-28,2.50,2.50,112.175246
2014-03-03,2.50,2.50,112.198296
2014-03-04,2.50,2.50,112.205981
2014-03-05,2.50,2.50,112.213666
2014-03-06,2.50,2.50,112.221352
2014-03-07,2.50,2.50,112.229038
2014-03-10,2.50,2.50,112.252099
2014-03-11,2.50,2.50,112.259788
2014-03-12,2.50,2.50,112.267477
2014-03-13,2.50,2.50,112.275166
2014-03-14,2.50,2.50,112.282856
2014-03-17,2.50,2.50,112.305928
2014-03-18,2.50,2.50,112.313620
2014-03-19,2.50,2.50,112.321313
2014-03-20,2.50,2.50,112.329006
2014-03-21,2.50,2.50,112.336700
2014-03-24,2.50,2.50,112.359783
2014-03-25,2.50,2.50,112.367479
2014-03-26,2.50,2.50,112.375175
2014-03-27,2.50,2.50,112.382872
2014-03-28,2.50,2.50,112.390569
2014-03-31,2.50,2.50,112.413663
2014-04-01,2.50,2.50,112.421363
2014-04-02,2.50,2.50,112.429063
2014-04-03,2.50,2.50,112.436764
2014-04-04,2.50,2.50,112.444465
2014-04-07,2.50,2.50,112.467570
2014-04-08,2.50,2.50,112.475273
2014-04-09,2.50,2.50,112.482977
2014-04-10,2.50,2.50,112.490681
2014-04-11,2.50,2.50,112.498386
2014-04-14,2.50,2.50,112.521502
2014-04-15,2.50,2.50,112.529209
2014-04-16,2.50,2.50,112.536917
2014-04-17,2.50,2.50,112.544625
2014-04-22,2.50,2.50,112.583167
2014-04-23,2.50,2.50,112.590878
2014-04-24,2.50,2.50,112.598590
2014-04-28,2.50,2.50,112.629439
2014-04-29,2.50,2.50,112.637153
2014-04-30,2.50,2.50,112.644868
2014-05-01,2.50,2.50,112.652584
2014-05-02,2.50,2.50,112.660300
2014-05-05,2.50,2.50,112.683449
2014-05-06,2.50,2.50,112.691167
2014-05-07,2.50,2.50,112.698886
2014-05-08,2.50,2.50,112.706605
2014-05-09,2.50,2.50,112.714324
2014-05-12,2.50,2.50,112.737485
2014-05-13,2.50,2.50,112.745207
2014-05-14,2.50,2.50,112.752929
2014-05-15,2.50,2.50,112.760652
2014-05-16,2.50,2.50,112.768375
2014-05-19,2.50,2.50,112.791547
2014-05-20,2.50,2.50,112.799272
2014-05-21,2.50,2.50,112.806998
2014-05-22,2.50,2.50,112.814725
2014-05-23,2.50,2.50,112.822452
2014-05-26,2.50,2.50,112.845634
2014-05-27,2.50,2.50,112.853363
2014-05-28,2.50,2.50,112.861093
2014-05-29,2.50,2.50,112.868823
2014-05-30,2.50,2.50,112.876554
2014-06-02,2.50,2.50,112.899748
2014-06-03,2.50,2.50,112.907481
2014-06-04,2.50,2.50,112.915214
2014-06-05,2.50,2.50,112.922948
2014-06-06,2.50,2.50,112.930683
2014-06-10,2.50,2.50,112.961622
2014-06-11,2.50,2.50,112.969360
2014-06-12,2.50,2.50,112.977097
2014-06-13,2.50,2.50,112.984835
2014-06-16,2.50,2.50,113.008051
2014-06-17,2.50,2.50,113.015792
2014-06-18,2.50,2.50,113.023532
2014-06-19,2.50,2.50,113.031274
2014-06-20,2.50,2.50,113.039016
2014-06-23,2.50,2.50,113.062243
2014-06-24,2.50,2.50,113.069987
2014-06-25,2.50,2.50,113.077731
2014-06-26,2.50,2.50,113.085476
2014-06-27,2.50,2.50,113.093222
2014-06-30,2.50,2.50,113.116460
2014-07-01,2.50,2.50,113.124208
2014-07-02,2.50,2.50,113.131956
2014-07-03,2.50,2.50,113.139705
2014-07-04,2.50,2.50,113.147454
2014-07-07,2.50,2.50,113.170704
2014-07-08,2.50,2.50,113.178455
2014-07-09,2.50,2.50,113.186207
2014-07-10,2.50,2.50,113.193960
2014-07-11,2.50,2.50,113.201713
2014-07-14,2.50,2.50,113.224973
2014-07-15,2.50,2.50,113.232728
2014-07-16,2.50,2.50,113.240484
2014-07-17,2.50,2.50,113.248240
2014-07-18,2.50,2.50,113.255997
2014-07-21,2.50,2.50,113.279269
2014-07-22,2.50,2.50,113.287028
2014-07-23,2.50,2.50,113.294787
2014-07-24,2.50,2.50,113.302547
2014-07-25,2.50,2.50,113.310307
2014-07-28,2.50,2.50,113.333590
2014-07-29,2.50,2.50,113.341353
2014-07-30,2.50,2.50,113.349116
2014-07-31,2.50,2.50,113.356880
2014-08-01,2.50,2.50,113.364644
2014-08-04,2.50,2.50,113.387938
2014-08-05,2.50,2.50,113.395704
2014-08-06,2.50,2.50,113.403471
2014-08-07,2.50,2.50,113.411238
2014-08-08,2.50,2.50,113.419006
2014-08-11,2.50,2.50,113.442312
2014-08-12,2.50,2.50,113.450082
2014-08-13,2.50,2.50,113.457852
2014-08-14,2.50,2.50,113.465623
2014-08-15,2.50,2.50,113.473395
2014-08-18,2.50,2.50,113.496711
2014-08-19,2.50,2.50,113.504485
2014-08-20,2.50,2.50,113.512259
2014-08-21,2.50,2.50,113.520034
2014-08-22,2.50,2.50,113.527809
2014-08-25,2.50,2.50,113.551137
2014-08-26,2.50,2.50,113.558915
2014-08-27,2.50,2.50,113.566693
2014-08-28,2.50,2.50,113.574471
2014-08-29,2.50,2.50,113.582250
2014-09-01,2.50,2.50,113.605589
2014-09-02,2.50,2.50,113.613370
2014-09-03,2.50,2.50,113.621152
2014-09-04,2.50,2.50,113.628934
2014-09-05,2.50,2.50,113.636717
2014-09-08,2.50,2.50,113.660067
2014-09-09,2.50,2.50,113.667852
2014-09-10,2.50,2.50,113.675637
2014-09-11,2.50,2.50,113.683423
2014-09-12,2.50,2.50,113.691210
2014-09-15,2.50,2.50,113.714571
2014-09-16,2.50,2.50,113.722360
2014-09-17,2.50,2.50,113.730149
2014-09-18,2.50,2.50,113.737939
2014-09-19,2.50,2.50,113.745729
2014-09-22,2.50,2.50,113.769102
2014-09-23,2.50,2.50,113.776894
2014-09-24,2.50,2.50,113.784687
2014-09-25,2.50,2.50,113.792480
2014-09-26,2.50,2.50,113.800274
2014-09-29,2.50,2.50,113.823658
2014-09-30,2.50,2.50,113.831454
2014-10-01,2.50,2.50,113.839251
2014-10-02,2.50,2.50,113.847048
2014-10-03,2.50,2.50,113.854846
2014-10-06,2.50,2.50,113.878241
2014-10-07,2.50,2.50,113.886040
2014-10-08,2.50,2.50,113.893841
2014-10-09,2.50,2.50,113.901642
2014-10-10,2.50,2.50,113.909443
2014-10-13,2.50,2.50,113.932849
2014-10-14,2.50,2.50,113.940653
2014-10-15,2.50,2.50,113.948457
2014-10-16,2.50,2.50,113.956262
2014-10-17,2.50,2.50,113.964067
2014-10-20,2.50,2.50,113.987484
2014-10-21,2.50,2.50,113.995292
2014-10-22,2.50,2.50,114.003100
2014-10-23,2.50,2.50,114.010908
2014-10-24,2.50,2.50,114.018717
2014-10-27,2.50,2.50,114.042145
2014-10-28,2.50,2.50,114.049957
2014-10-29,2.50,2.50,114.057768
2014-10-30,2.50,2.50,114.065580
2014-10-31,2.50,2.50,114.073393
2014-11-03,2.50,2.50,114.096833
2014-11-04,2.50,2.50,114.104648
2014-11-05,2.50,2.50,114.112463
2014-11-06,2.50,2.50,114.120279
2014-11-07,2.50,2.50,114.128095
2014-11-10,2.50,2.50,114.151546
2014-11-11,2.50,2.50,114.159365
2014-11-12,2.50,2.50,114.167184
2014-11-13,2.50,2.50,114.175004
2014-11-14,2.50,2.50,114.182824
2014-11-17,2.50,2.50,114.206286
2014-11-18,2.50,2.50,114.214109
2014-11-19,2.50,2.50,114.221931
2014-11-20,2.50,2.50,114.229755
2014-11-21,2.50,2.50,114.237579
2014-11-24,2.50,2.50,114.261052
2014-11-25,2.50,2.50,114.268878
2014-11-26,2.50,2.50,114.276705
2014-11-27,2.50,2.50,114.284532
2014-11-28,2.50,2.50,114.292360
2014-12-01,2.50,2.50,114.315845
2014-12-02,2.50,2.50,114.323675
2014-12-03,2.50,2.50,114.331505
2014-12-04,2.50,2.50,114.339336
2014-12-05,2.50,2.50,114.347167
2014-12-08,2.50,2.50,114.370663
2014-12-09,2.50,2.50,114.378497
2014-12-10,2.50,2.50,114.386331
2014-12-11,2.50,2.50,114.394166
2014-12-12,2.50,2.50,114.402001
2014-12-15,2.50,2.50,114.425508
2014-12-16,2.50,2.50,114.433346
2014-12-17,2.50,2.50,114.441183
2014-12-18,2.50,2.50,114.449022
2014-12-19,2.50,2.50,114.456861
2014-12-22,2.50,2.50,114.480379
2014-12-23,2.50,2.50,114.488221
2014-12-24,2.50,2.50,114.496062
2014-12-29,2.50,2.50,114.535273
2014-12-30,2.50,2.50,114.543118
2014-12-31,2.50,2.50,114.550963
2015-01-02,2.50,2.50,114.566655
2015-01-05,2.50,2.50,114.590196
2015-01-06,2.50,2.50,114.598045
2015-01-07,2.50,2.50,114.605894
2015-01-08,2.50,2.50,114.613744
2015-01-09,2.50,2.50,114.621594
2015-01-12,2.50,2.50,114.645147
2015-01-13,2.50,2.50,114.652999
2015-01-14,2.50,2.50,114.660852
2015-01-15,2.50,2.50,114.668705
2015-01-16,2.50,2.50,114.676560
2015-01-19,2.50,2.50,114.700123
2015-01-20,2.50,2.50,114.707979
2015-01-21,2.50,2.50,114.715836
2015-01-22,2.50,2.50,114.723693
2015-01-23,2.50,2.50,114.731551
2015-01-27,2.50,2.50,114.762984
2015-01-28,2.50,2.50,114.770845
2015-01-29,2.50,2.50,114.778706
2015-01-30,2.50,2.50,114.786567
2015-02-02,2.50,2.50,114.810154
2015-02-03,2.50,2.50,114.818017
2015-02-04,2.25,2.25,114.825882
2015-02-05,2.25,2.25,114.832960
2015-02-06,2.25,2.25,114.840039
2015-02-09,2.25,2.25,114.861276
2015-02-10,2.25,2.25,114.868357
2015-02-11,2.25,2.25,114.875438
2015-02-12,2.25,2.25,114.882519
2015-02-13,2.25,2.25,114.889601
2015-02-16,2.25,2.25,114.910848
2015-02-17,2.25,2.25,114.917931
2015-02-18,2.25,2.25,114.925015
2015-02-19,2.25,2.25,114.932100
2015-02-20,2.25,2.25,114.939184
2015-02-23,2.25,2.25,114.960440
2015-02-24,2.25,2.25,114.967527
2015-02-25,2.25,2.25,114.974614
2015-02-26,2.25,2.25,114.981701
2015-02-27,2.25,2.25,114.988789
2015-03-02,2.25,2.25,115.010054
2015-03-03,2.25,2.25,115.017144
2015-03-04,2.25,2.25,115.024234
2015-03-05,2.25,2.25,115.031325
2015-03-06,2.25,2.25,115.038416
2015-03-09,2.25,2.25,115.059690
2015-03-10,2.25,2.25,115.066783
2015-03-11,2.25,2.25,115.073876
2015-03-12,2.25,2.25,115.080969
2015-03-13,2.25,2.25,115.088063
2015-03-16,2.25,2.25,115.109347
2015-03-17,2.25,2.25,115.116443
2015-03-18,2.25,2.25,115.123539
2015-03-19,2.25,2.25,115.130635
2015-03-20,2.25,2.25,115.137732
2015-03-23,2.25,2.25,115.159025
2015-03-24,2.25,2.25,115.166124
2015-03-25,2.25,2.25,115.173223
2015-03-26,2.25,2.25,115.180323
2015-03-27,2.25,2.25,115.187423
2015-03-30,2.25,2.25,115.208725
2015-03-31,2.25,2.25,115.215827
2015-04-01,2.25,2.25,115.222929
2015-04-02,2.25,2.25,115.230032
2015-04-07,2.25,2.25,115.265548
2015-04-08,2.25,2.25,115.272653
2015-04-09,2.25,2.25,115.279759
2015-04-10,2.25,2.25,115.286866
2015-04-13,2.25,2.25,115.308186
2015-04-14,2.25,2.25,115.315294
2015-04-15,2.25,2.25,115.322402
2015-04-16,2.25,2.25,115.329511
2015-04-17,2.25,2.25,115.336621
2015-04-20,2.25,2.25,115.357950
2015-04-21,2.25,2.25,115.365061
2015-04-22,2.25,2.25,115.372173
2015-04-23,2.25,2.25,115.379285
2015-04-24,2.25,2.25,115.386397
2015-04-27,2.25,2.25,115.407736
2015-04-28,2.25,2.25,115.414850
2015-04-29,2.25,2.25,115.421964
2015-04-30,2.25,2.25,115.429079
2015-05-01,2.25,2.25,115.436195
2015-05-04,2.25,2.25,115.457543
2015-05-05,2.25,2.25,115.464660
2015-05-06,2.00,2.00,115.471778
2015-05-07,2.00,2.00,115.478105
2015-05-08,2.00,2.00,115.484432
2015-05-11,2.00,2.00,115.503416
2015-05-12,2.00,2.00,115.509745
2015-05-13,2.00,2.00,115.516074
2015-05-14,2.00,2.00,115.522404
2015-05-15,2.00,2.00,115.528734
2015-05-18,2.00,2.00,115.547725
2015-05-19,2.00,2.00,115.554056
2015-05-20,2.00,2.00,115.560388
2015-05-21,2.00,2.00,115.566720
2015-05-22,2.00,2.00,115.573053
2015-05-25,2.00,2.00,115.592051
2015-05-26,2.00,2.00,115.598385
2015-05-27,2.00,2.00,115.604719
2015-05-28,2.00,2.00,115.611053
2015-05-29,2.00,2.00,115.617388
2015-06-01,2.00,2.00,115.636394
2015-06-02,2.00,2.00,115.642730
2015-06-03,2.00,2.00,115.649067
2015-06-04,2.00,2.00,115.655404
2015-06-05,2.00,2.00,115.661741
2015-06-09,2.00,2.00,115.687091
2015-06-10,2.00,2.00,115.693430
2015-06-11,2.00,2.00,115.699770
2015-06-12,2.00,2.00,115.706110
2015-06-15,2.00,2.00,115.725130
2015-06-16,2.00,2.00,115.731471
2015-06-17,2.00,2.00,115.737812
2015-06-18,2.00,2.00,115.744154
2015-06-19,2.00,2.00,115.750496
2015-06-22,2.00,2.00,115.769524
2015-06-23,2.00,2.00,115.775867
2015-06-24,2.00,2.00,115.782211
2015-06-25,2.00,2.00,115.788555
2015-06-26,2.00,2.00,115.794900
2015-06-29,2.00,2.00,115.813935
2015-06-30,2.00,2.00,115.820281
2015-07-01,2.00,2.00,115.826627
2015-07-02,2.00,2.00,115.832974
2015-07-03,2.00,2.00,115.839321
2015-07-06,2.00,2.00,115.858363
2015-07-07,2.00,2.00,115.864711
2015-07-08,2.00,2.00,115.871060
2015-07-09,2.00,2.00,115.877409
2015-07-10,2.00,2.00,115.883758
2015-07-13,2.00,2.00,115.902808
2015-07-14,2.00,2.00,115.909159
2015-07-15,2.00,2.00,115.915510
2015-07-16,2.00,2.00,115.921861
2015-07-17,2.00,2.00,115.928213
2015-07-20,2.00,2.00,115.947270
2015-07-21,2.00,2.00,115.953623
2015-07-22,2.00,2.00,115.959977
2015-07-23,2.00,2.00,115.966331
2015-07-24,2.00,2.00,115.972685
2015-07-27,2.00,2.00,115.991749
2015-07-28,2.00,2.00,115.998105
2015-07-29,2.00,2.00,116.004461
2015-07-30,2.00,2.00,116.010817
2015-07-31,2.00,2.00,116.017174
2015-08-03,2.00,2.00,116.036245
2015-08-04,2.00,2.00,116.042604
2015-08-05,2.00,2.00,116.048962
2015-08-06,2.00,2.00,116.055321
2015-08-07,2.00,2.00,116.061680
2015-08-10,2.00,2.00,116.080759
2015-08-11,2.00,2.00,116.087119
2015-08-12,2.00,2.00,116.093480
2015-08-13,2.00,2.00,116.099842
2015-08-14,2.00,2.00,116.106203
2015-08-17,2.00,2.00,116.125289
2015-08-18,2.00,2.00,116.131652
2015-08-19,2.00,2.00,116.138016
2015-08-20,2.00,2.00,116.144379
2015-08-21,2.00,2.00,116.150743
2015-08-24,2.00,2.00,116.169837
2015-08-25,2.00,2.00,116.176202
2015-08-26,2.00,2.00,116.182568
2015-08-27,2.00,2.00,116.188934
2015-08-28,2.00,2.00,116.195301
2015-08-31,2.00,2.00,116.214401
2015-09-01,2.00,2.00,116.220769
2015-09-02,2.00,2.00,116.227137
2015-09-03,2.00,2.00,116.233506
2015-09-04,2.00,2.00,116.239875
2015-09-07,2.00,2.00,116.258983
2015-09-08,2.00,2.00,116.265353
2015-09-09,2.00,2.00,116.271724
2015-09-10,2.00,2.00,116.278095
2015-09-11,2.00,2.00,116.284466
2015-09-14,2.00,2.00,116.303582
2015-09-15,2.00,2.00,116.309954
2015-09-16,2.00,2.00,116.316328
2015-09-17,2.00,2.00,116.322701
2015-09-18,2.00,2.00,116.329075
2015-09-21,2.00,2.00,116.348198
2015-09-22,2.00,2.00,116.354573
2015-09-23,2.00,2.00,116.360948
2015-09-24,2.00,2.00,116.367324
2015-09-25,2.00,2.00,116.373701
2015-09-28,2.00,2.00,116.392831
2015-09-29,2.00,2.00,116.399208
2015-09-30,2.00,2.00,116.405586
2015-10-01,2.00,2.00,116.411965
2015-10-02,2.00,2.00,116.418343
2015-10-05,2.00,2.00,116.437481
2015-10-06,2.00,2.00,116.443861
2015-10-07,2.00,2.00,116.450241
2015-10-08,2.00,2.00,116.456622
2015-10-09,2.00,2.00,116.463003
2015-10-12,2.00,2.00,116.482148
2015-10-13,2.00,2.00,116.488530
2015-10-14,2.00,2.00,116.494913
2015-10-15,2.00,2.00,116.501297
2015-10-16,2.00,2.00,116.507680
2015-10-19,2.00,2.00,116.526832
2015-10-20,2.00,2.00,116.533217
2015-10-21,2.00,2.00,116.539603
2015-10-22,2.00,2.00,116.545988
2015-10-23,2.00,2.00,116.552374
2015-10-26,2.00,2.00,116.571534
2015-10-27,2.00,2.00,116.577921
2015-10-28,2.00,2.00,116.584309
2015-10-29,2.00,2.00,116.590697
2015-10-30,2.00,2.00,116.597086
2015-11-02,2.00,2.00,116.616252
2015-11-03,2.00,2.00,116.622642
2015-11-04,2.00,2.00,116.629033
2015-11-05,2.00,2.00,116.635423
2015-11-06,2.00,2.00,116.641814
2015-11-09,2.00,2.00,116.660988
2015-11-10,2.00,2.00,116.667381
2015-11-11,2.00,2.00,116.673773
2015-11-12,2.00,2.00,116.680166
2015-11-13,2.00,2.00,116.686560
2015-11-16,2.00,2.00,116.705741
2015-11-17,2.00,2.00,116.712136
2015-11-18,2.00,2.00,116.718531
2015-11-19,2.00,2.00,116.724927
2015-11-20,2.00,2.00,116.731323
2015-11-23,2.00,2.00,116.750511
2015-11-24,2.00,2.00,116.756909
2015-11-25,2.00,2.00,116.763306
2015-11-26,2.00,2.00,116.769704
2015-11-27,2.00,2.00,116.776103
2015-11-30,2.00,2.00,116.795299
2015-12-01,2.00,2.00,116.801698
2015-12-02,2.00,2.00,116.808099
2015-12-03,2.00,2.00,116.814499
2015-12-04,2.00,2.00,116.820900
2015-12-07,2.00,2.00,116.840103
2015-12-08,2.00,2.00,116.846505
2015-12-09,2.00,2.00,116.852908
2015-12-10,2.00,2.00,116.859311
2015-12-11,2.00,2.00,116.865714
2015-12-14,2.00,2.00,116.884925
2015-12-15,2.00,2.00,116.891330
2015-12-16,2.00,2.00,116.897735
2015-12-17,2.00,2.00,116.904140
2015-12-18,2.00,2.00,116.910546
2015-12-21,2.00,2.00,116.929764
2015-12-22,2.00,2.00,116.936171
2015-12-23,2.00,2.00,116.942578
2015-12-24,2.00,2.00,116.948986
2015-12-29,2.00,2.00,116.981027
2015-12-30,2.00,2.00,116.987437
2015-12-31,2.00,2.00,116.993847
2016-01-04,2.00,2.00,117.019490
2016-01-05,2.00,2.00,117.025902
2016-01-06,2.00,2.00,117.032314
2016-01-07,2.00,2.00,117.038727
2016-01-08,2.00,2.00,117.045140
2016-01-11,2.00,2.00,117.064380
2016-01-12,2.00,2.00,117.070795
2016-01-13,2.00,2.00,117.077210
2016-01-14,2.00,2.00,117.083625
2016-01-15,2.00,2.00,117.090040
2016-01-18,2.00,2.00,117.109288
2016-01-19,2.00,2.00,117.115705
2016-01-20,2.00,2.00,117.122122
2016-01-21,2.00,2.00,117.128540
2016-01-22,2.00,2.00,117.134958
2016-01-25,2.00,2.00,117.154213
2016-01-27,2.00,2.00,117.167052
2016-01-28,2.00,2.00,117.173472
2016-01-29,2.00,2.00,117.179892
2016-02-01,2.00,2.00,117.199155
2016-02-02,2.00,2.00,117.205577
2016-02-03,2.00,2.00,117.211999
2016-02-04,2.00,2.00,117.218421
2016-02-05,2.00,2.00,117.224844
2016-02-08,2.00,2.00,117.244114
2016-02-09,2.00,2.00,117.250539
2016-02-10,2.00,2.00,117.256963
2016-02-11,2.00,2.00,117.263388
2016-02-12,2.00,2.00,117.269814
2016-02-15,2.00,2.00,117.289091
2016-02-16,2.00,2.00,117.295518
2016-02-17,2.00,2.00,117.301945
2016-02-18,2.00,2.00,117.308372
2016-02-19,2.00,2.00,117.314800
2016-02-22,2.00,2.00,117.334085
2016-02-23,2.00,2.00,117.340514
2016-02-24,2.00,2.00,117.346944
2016-02-25,2.00,2.00,117.353374
2016-02-26,2.00,2.00,117.359804
2016-02-29,2.00,2.00,117.379096
2016-03-01,2.00,2.00,117.385528
2016-03-02,2.00,2.00,117.391960
2016-03-03,2.00,2.00,117.398392
2016-03-04,2.00,2.00,117.404825
2016-03-07,2.00,2.00,117.424124
2016-03-08,2.00,2.00,117.430559
2016-03-09,2.00,2.00,117.436993
2016-03-10,2.00,2.00,117.443428
2016-03-11,2.00,2.00,117.449863
2016-03-14,2.00,2.00,117.469170
2016-03-15,2.00,2.00,117.475607
2016-03-16,2.00,2.00,117.482044
2016-03-17,2.00,2.00,117.488481
2016-03-18,2.00,2.00,117.494919
2016-03-21,2.00,2.00,117.514233
2016-03-22,2.00,2.00,117.520672
2016-03-23,2.00,2.00,117.527112
2016-03-24,2.00,2.00,117.533552
2016-03-29,2.00,2.00,117.565753
2016-03-30,2.00,2.00,117.572195
2016-03-31,2.00,2.00,117.578637
2016-04-01,2.00,2.00,117.585080
2016-04-04,2.00,2.00,117.604409
2016-04-05,2.00,2.00,117.610853
2016-04-06,2.00,2.00,117.617297
2016-04-07,2.00,2.00,117.623742
2016-04-08,2.00,2.00,117.630187
2016-04-11,2.00,2.00,117.649524
2016-04-12,2.00,2.00,117.655970
2016-04-13,2.00,2.00,117.662417
2016-04-14,2.00,2.00,117.668864
2016-04-15,2.00,2.00,117.675312
2016-04-18,2.00,2.00,117.694656
2016-04-19,2.00,2.00,117.701105
2016-04-20,2.00,2.00,117.707554
2016-04-21,2.00,2.00,117.714004
2016-04-22,2.00,2.00,117.720454
2016-04-26,2.00,2.00,117.746256
2016-04-27,2.00,2.00,117.752708
2016-04-28,2.00,2.00,117.759160
2016-04-29,2.00,2.00,117.765612
2016-05-02,2.00,2.00,117.784971
2016-05-03,2.00,2.00,117.791425
2016-05-04,1.75,1.75,117.797879
2016-05-05,1.75,1.75,117.803527
2016-05-06,1.75,1.75,117.809175
2016-05-09,1.75,1.75,117.826120
2016-05-10,1.75,1.75,117.831770
2016-05-11,1.75,1.75,117.837419
2016-05-12,1.75,1.75,117.843069
2016-05-13,1.75,1.75,117.848719
2016-05-16,1.75,1.75,117.865670
2016-05-17,1.75,1.75,117.871321
2016-05-18,1.75,1.75,117.876972
2016-05-19,1.75,1.75,117.882624
2016-05-20,1.75,1.75,117.888276
2016-05-23,1.75,1.75,117.905232
2016-05-24,1.75,1.75,117.910885
2016-05-25,1.75,1.75,117.916539
2016-05-26,1.75,1.75,117.922192
2016-05-27,1.75,1.75,117.927846
2016-05-30,1.75,1.75,117.944808
2016-05-31,1.75,1.75,117.950463
2016-06-01,1.75,1.75,117.956118
2016-06-02,1.75,1.75,117.961774
2016-06-03,1.75,1.75,117.967429
2016-06-06,1.75,1.75,117.984397
2016-06-07,1.75,1.75,117.990054
2016-06-08,1.75,1.75,117.995711
2016-06-09,1.75,1.75,118.001368
2016-06-10,1.75,1.75,118.007026
2016-06-14,1.75,1.75,118.029657
2016-06-15,1.75,1.75,118.035316
2016-06-16,1.75,1.75,118.040976
2016-06-17,1.75,1.75,118.046635
2016-06-20,1.75,1.75,118.063614
2016-06-21,1.75,1.75,118.069275
2016-06-22,1.75,1.75,118.074936
2016-06-23,1.75,1.75,118.080597
2016-06-24,1.75,1.75,118.086258
2016-06-27,1.75,1.75,118.103243
2016-06-28,1.75,1.75,118.108906
2016-06-29,1.75,1.75,118.114569
2016-06-30,1.75,1.75,118.120232
2016-07-01,1.75,1.75,118.125895
2016-07-04,1.75,1.75,118.142886
2016-07-05,1.75,1.75,118.148550
2016-07-06,1.75,1.75,118.154215
2016-07-07,1.75,1.75,118.159880
2016-07-08,1.75,1.75,118.165545
2016-07-11,1.75,1.75,118.182541
2016-07-12,1.75,1.75,118.188208
2016-07-13,1.75,1.75,118.193874
2016-07-14,1.75,1.75,118.199541
2016-07-15,1.75,1.75,118.205208
2016-07-18,1.75,1.75,118.222210
2016-07-19,1.75,1.75,118.227878
2016-07-20,1.75,1.75,118.233547
2016-07-21,1.75,1.75,118.239216
2016-07-22,1.75,1.75,118.244885
2016-07-25,1.75,1.75,118.261892
2016-07-26,1.75,1.75,118.267562
2016-07-27,1.75,1.75,118.273233
2016-07-28,1.75,1.75,118.278903
2016-07-29,1.75,1.75,118.284574
2016-08-01,1.75,1.75,118.301588
2016-08-02,1.75,1.75,118.307260
2016-08-03,1.50,1.50,118.312932
2016-08-04,1.50,1.50,118.317794
2016-08-05,1.50,1.50,118.322657
2016-08-08,1.50,1.50,118.337244
2016-08-09,1.50,1.50,118.342108
2016-08-10,1.50,1.50,118.346971
2016-08-11,1.50,1.50,118.351835
2016-08-12,1.50,1.50,118.356698
2016-08-15,1.50,1.50,118.371290
2016-08-16,1.50,1.50,118.376155
2016-08-17,1.50,1.50,118.381020
2016-08-18,1.50,1.50,118.385885
2016-08-19,1.50,1.50,118.390750
2016-08-22,1.50,1.50,118.405346
2016-08-23,1.50,1.50,118.410212
2016-08-24,1.50,1.50,118.415078
2016-08-25,1.50,1.50,118.419944
2016-08-26,1.50,1.50,118.424811
2016-08-29,1.50,1.50,118.439411
2016-08-30,1.50,1.50,118.444279
2016-08-31,1.50,1.50,118.449146
2016-09-01,1.50,1.50,118.454014
2016-09-02,1.50,1.50,118.458882
2016-09-05,1.50,1.50,118.473486
2016-09-06,1.50,1.50,118.478355
2016-09-07,1.50,1.50,118.483224
2016-09-08,1.50,1.50,118.488093
2016-09-09,1.50,1.50,118.492963
2016-09-12,1.50,1.50,118.507571
2016-09-13,1.50,1.50,118.512442
2016-09-14,1.50,1.50,118.517312
2016-09-15,1.50,1.50,118.522183
2016-09-16,1.50,1.50,118.527053
2016-09-19,1.50,1.50,118.541666
2016-09-20,1.50,1.50,118.546538
2016-09-21,1.50,1.50,118.551410
2016-09-22,1.50,1.50,118.556282
2016-09-23,1.50,1.50,118.561154
2016-09-26,1.50,1.50,118.575771
2016-09-27,1.50,1.50,118.580644
2016-09-28,1.50,1.50,118.585517
2016-09-29,1.50,1.50,118.590390
2016-09-30,1.50,1.50,118.595264
2016-10-03,1.50,1.50,118.609885
2016-10-04,1.50,1.50,118.614760
2016-10-05,1.50,1.50,118.619634
2016-10-06,1.50,1.50,118.624509
2016-10-07,1.50,1.50,118.629384
2016-10-10,1.50,1.50,118.644010
2016-10-11,1.50,1.50,118.648885
2016-10-12,1.50,1.50,118.653761
2016-10-13,1.50,1.50,118.658638
2016-10-14,1.50,1.50,118.663514
2016-10-17,1.50,1.50,118.678144
2016-10-18,1.50,1.50,118.683021
2016-10-19,1.50,1.50,118.687898
2016-10-20,1.50,1.50,118.692776
2016-10-21,1.50,1.50,118.697654
2016-10-24,1.50,1.50,118.712288
2016-10-25,1.50,1.50,118.717166
2016-10-26,1.50,1.50,118.722045
2016-10-27,1.50,1.50,118.726924
2016-10-28,1.50,1.50,118.731803
2016-10-31,1.50,1.50,118.746441
2016-11-01,1.50,1.50,118.751321
2016-11-02,1.50,1.50,118.756202
2016-11-03,1.50,1.50,118.761082
2016-11-04,1.50,1.50,118.765962
2016-11-07,1.50,1.50,118.780605
2016-11-08,1.50,1.50,118.785486
2016-11-09,1.50,1.50,118.790368
2016-11-10,1.50,1.50,118.795250
2016-11-11,1.50,1.50,118.800132
2016-11-14,1.50,1.50,118.814778
2016-11-15,1.50,1.50,118.819661
2016-11-16,1.50,1.50,118.824544
2016-11-17,1.50,1.50,118.829427
2016-11-18,1.50,1.50,118.834311
2016-11-21,1.50,1.50,118.848961
2016-11-22,1.50,1.50,118.853846
2016-11-23,1.50,1.50,118.858730
2016-11-24,1.50,1.50,118.863615
2016-11-25,1.50,1.50,118.868499
2016-11-28,1.50,1.50,118.883154
2016-11-29,1.50,1.50,118.888040
2016-11-30,1.50,1.50,118.892926
2016-12-01,1.50,1.50,118.897812
2016-12-02,1.50,1.50,118.902698
2016-12-05,1.50,1.50,118.917357
2016-12-06,1.50,1.50,118.922244
2016-12-07,1.50,1.50,118.927132
2016-12-08,1.50,1.50,118.932019
2016-12-09,1.50,1.50,118.936907
2016-12-12,1.50,1.50,118.951570
2016-12-13,1.50,1.50,118.956459
2016-12-14,1.50,1.50,118.961347
2016-12-15,1.50,1.50,118.966236
2016-12-16,1.50,1.50,118.971125
2016-12-19,1.50,1.50,118.985793
2016-12-20,1.50,1.50,118.990682
2016-12-21,1.50,1.50,118.995573
2016-12-22,1.50,1.50,119.000463
2016-12-23,1.50,1.50,119.005353
2016-12-28,1.50,1.50,119.029806
2016-12-29,1.50,1.50,119.034698
2016-12-30,1.50,1.50,119.039590
2017-01-03,1.50,1.50,119.059158
2017-01-04,1.50,1.50,119.064051
2017-01-05,1.50,1.50,119.068944
2017-01-06,1.50,1.50,119.073837
2017-01-09,1.50,1.50,119.088517
2017-01-10,1.50,1.50,119.093411
2017-01-11,1.50,1.50,119.098306
2017-01-12,1.50,1.50,119.103200
2017-01-13,1.50,1.50,119.108095
2017-01-16,1.50,1.50,119.122779
2017-01-17,1.50,1.50,119.127675
2017-01-18,1.50,1.50,119.132570
2017-01-19,1.50,1.50,119.137466
2017-01-20,1.50,1.50,119.142362
2017-01-23,1.50,1.50,119.157051
2017-01-24,1.50,1.50,119.161948
2017-01-25,1.50,1.50,119.166845
2017-01-27,1.50,1.50,119.176640
2017-01-30,1.50,1.50,119.191333
2017-01-31,1.50,1.50,119.196231
2017-02-01,1.50,1.50,119.201129
2017-02-02,1.50,1.50,119.206028
2017-02-03,1.50,1.50,119.210927
2017-02-06,1.50,1.50,119.225624
2017-02-07,1.50,1.50,119.230524
2017-02-08,1.50,1.50,119.235424
2017-02-09,1.50,1.50,119.240324
2017-02-10,1.50,1.50,119.245224
2017-02-13,1.50,1.50,119.259926
2017-02-14,1.50,1.50,119.264827
2017-02-15,1.50,1.50,119.269728
2017-02-16,1.50,1.50,119.274629
2017-02-17,1.50,1.50,119.279531
2017-02-20,1.50,1.50,119.294237
2017-02-21,1.50,1.50,119.299139
2017-02-22,1.50,1.50,119.304042
2017-02-23,1.50,1.50,119.308945
2017-02-24,1.50,1.50,119.313848
2017-02-27,1.50,1.50,119.328558
2017-02-28,1.50,1.50,119.333462
2017-03-01,1.50,1.50,119.338366
2017-03-02,1.50,1.50,119.343270
2017-03-03,1.50,1.50,119.348175
2017-03-06,1.50,1.50,119.362889
2017-03-07,1.50,1.50,119.367794
2017-03-08,1.50,1.50,119.372700
2017-03-09,1.50,1.50,119.377606
2017-03-10,1.50,1.50,119.382512
2017-03-13,1.50,1.50,119.397230
2017-03-14,1.50,1.50,119.402137
2017-03-15,1.50,1.50,119.407044
2017-03-16,1.50,1.50,119.411951
2017-03-17,1.50,1.50,119.416858
2017-03-20,1.50,1.50,119.431581
2017-03-21,1.50,1.50,119.436489
2017-03-22,1.50,1.50,119.441397
2017-03-23,1.50,1.50,119.446306
2017-03-24,1.50,1.50,119.451215
2017-03-27,1.50,1.50,119.465941
2017-03-28,1.50,1.50,119.470851
2017-03-29,1.50,1.50,119.475761
2017-03-30,1.50,1.50,119.480671
2017-03-31,1.50,1.50,119.485581
2017-04-03,1.50,1.50,119.500312
2017-04-04,1.50,1.50,119.505223
2017-04-05,1.50,1.50,119.510134
2017-04-06,1.50,1.50,119.515045
2017-04-07,1.50,1.50,119.519957
2017-04-10,1.50,1.50,119.534692
2017-04-11,1.50,1.50,119.539605
2017-04-12,1.50,1.50,119.544517
2017-04-13,1.50,1.50,119.549430
2017-04-18,1.50,1.50,119.573995
2017-04-19,1.50,1.50,119.578909
2017-04-20,1.50,1.50,119.583823
2017-04-21,1.50,1.50,119.588738
2017-04-24,1.50,1.50,119.603482
2017-04-26,1.50,1.50,119.613312
2017-04-27,1.50,1.50,119.618228
2017-04-28,1.50,1.50,119.623143
2017-05-01,1.50,1.50,119.637891
2017-05-02,1.50,1.50,119.642808
2017-05-03,1.50,1.50,119.647725
2017-05-04,1.50,1.50,119.652642
2017-05-05,1.50,1.50,119.657559
2017-05-08,1.50,1.50,119.672311
2017-05-09,1.50,1.50,119.677229
2017-05-10,1.50,1.50,119.682148
2017-05-11,1.50,1.50,119.687066
2017-05-12,1.50,1.50,119.691985
2017-05-15,1.50,1.50,119.706741
2017-05-16,1.50,1.50,119.711661
2017-05-17,1.50,1.50,119.716580
2017-05-18,1.50,1.50,119.721500
2017-05-19,1.50,1.50,119.726420
2017-05-22,1.50,1.50,119.741181
2017-05-23,1.50,1.50,119.746102
2017-05-24,1.50,1.50,119.751023
2017-05-25,1.50,1.50,119.755944
2017-05-26,1.50,1.50,119.760866
2017-05-29,1.50,1.50,119.775631
2017-05-30,1.50,1.50,119.780553
2017-05-31,1.50,1.50,119.785476
2017-06-01,1.50,1.50,119.790398
2017-06-02,1.50,1.50,119.795321
2017-06-05,1.50,1.50,119.810091
2017-06-06,1.50,1.50,119.815014
2017-06-07,1.50,1.50,119.819938
2017-06-08,1.50,1.50,119.824862
2017-06-09,1.50,1.50,119.829787
2017-06-13,1.50,1.50,119.849485
2017-06-14,1.50,1.50,119.854410
2017-06-15,1.50,1.50,119.859335
2017-06-16,1.50,1.50,119.864261
2017-06-19,1.50,1.50,119.879039
2017-06-20,1.50,1.50,119.883966
2017-06-21,1.50,1.50,119.888892
2017-06-22,1.50,1.50,119.893819
2017-06-23,1.50,1.50,119.898746
2017-06-26,1.50,1.50,119.913528
2017-06-27,1.50,1.50,119.918456
2017-06-28,1.50,1.50,119.923384
2017-06-29,1.50,1.50,119.928313
2017-06-30,1.50,1.50,119.933241
2017-07-03,1.50,1.50,119.948028
2017-07-04,1.50,1.50,119.952957
2017-07-05,1.50,1.50,119.957887
2017-07-06,1.50,1.50,119.962816
2017-07-07,1.50,1.50,119.967746
2017-07-10,1.50,1.50,119.982537
2017-07-11,1.50,1.50,119.987468
2017-07-12,1.50,1.50,119.992399
2017-07-13,1.50,1.50,119.997330
2017-07-14,1.50,1.50,120.002261
2017-07-17,1.50,1.50,120.017056
2017-07-18,1.50,1.50,120.021988
2017-07-19,1.50,1.50,120.026921
2017-07-20,1.50,1.50,120.031853
2017-07-21,1.50,1.50,120.036786
2017-07-24,1.50,1.50,120.051585
2017-07-25,1.50,1.50,120.056519
2017-07-26,1.50,1.50,120.061453
2017-07-27,1.50,1.50,120.066387
2017-07-28,1.50,1.50,120.071321
2017-07-31,1.50,1.50,120.086124
2017-08-01,1.50,1.50,120.091059
2017-08-02,1.50,1.50,120.095995
2017-08-03,1.50,1.50,120.100930
2017-08-04,1.50,1.50,120.105866
2017-08-07,1.50,1.50,120.120673
2017-08-08,1.50,1.50,120.125610
2017-08-09,1.50,1.50,120.130546
2017-08-10,1.50,1.50,120.135483
2017-08-11,1.50,1.50,120.140420
2017-08-14,1.50,1.50,120.155232
2017-08-15,1.50,1.50,120.160170
2017-08-16,1.50,1.50,120.165108
2017-08-17,1.50,1.50,120.170046
2017-08-18,1.50,1.50,120.174985
2017-08-21,1.50,1.50,120.189801
2017-08-22,1.50,1.50,120.194740
2017-08-23,1.50,1.50,120.199680
2017-08-24,1.50,1.50,120.204620
2017-08-25,1.50,1.50,120.209559
2017-08-28,1.50,1.50,120.224380
2017-08-29,1.50,1.50,120.229321
2017-08-30,1.50,1.50,120.234261
2017-08-31,1.50,1.50,120.239203
2017-09-01,1.50,1.50,120.244144
2017-09-04,1.50,1.50,120.258969
2017-09-05,1.50,1.50,120.263911
2017-09-06,1.50,1.50,120.268853
2017-09-07,1.50,1.50,120.273796
2017-09-08,1.50,1.50,120.278738
2017-09-11,1.50,1.50,120.293567
2017-09-12,1.50,1.50,120.298511
2017-09-13,1.50,1.50,120.303455
2017-09-14,1.50,1.50,120.308399
2017-09-15,1.50,1.50,120.313343
2017-09-18,1.50,1.50,120.328176
2017-09-19,1.50,1.50,120.333121
2017-09-20,1.50,1.50,120.338066
2017-09-21,1.50,1.50,120.343012
2017-09-22,1.50,1.50,120.347957
2017-09-25,1.50,1.50,120.362795
2017-09-26,1.50,1.50,120.367741
2017-09-27,1.50,1.50,120.372688
2017-09-28,1.50,1.50,120.377634
2017-09-29,1.50,1.50,120.382581
2017-10-02,1.50,1.50,120.397423
2017-10-03,1.50,1.50,120.402371
2017-10-04,1.50,1.50,120.407319
2017-10-05,1.50,1.50,120.412267
2017-10-06,1.50,1.50,120.417216
2017-10-09,1.50,1.50,120.432062
2017-10-10,1.50,1.50,120.437011
2017-10-11,1.50,1.50,120.441960
2017-10-12,1.50,1.50,120.446910
2017-10-13,1.50,1.50,120.451860
2017-10-16,1.50,1.50,120.466710
2017-10-17,1.50,1.50,120.471661
2017-10-18,1.50,1.50,120.476612
2017-10-19,1.50,1.50,120.481563
2017-10-20,1.50,1.50,120.486514
2017-10-23,1.50,1.50,120.501369
2017-10-24,1.50,1.50,120.506321
2017-10-25,1.50,1.50,120.511273
2017-10-26,1.50,1.50,120.516226
2017-10-27,1.50,1.50,120.521178
2017-10-30,1.50,1.50,120.536037
2017-10-31,1.50,1.50,120.540991
2017-11-01,1.50,1.50,120.545944
2017-11-02,1.50,1.50,120.550898
2017-11-03,1.50,1.50,120.555852
2017-11-06,1.50,1.50,120.570715
2017-11-07,1.50,1.50,120.575670
2017-11-08,1.50,1.50,120.580626
2017-11-09,1.50,1.50,120.585581
2017-11-10,1.50,1.50,120.590537
2017-11-13,1.50,1.50,120.605404
2017-11-14,1.50,1.50,120.610360
2017-11-15,1.50,1.50,120.615317
2017-11-16,1.50,1.50,120.620274
2017-11-17,1.50,1.50,120.625231
2017-11-20,1.50,1.50,120.640102
2017-11-21,1.50,1.50,120.645060
2017-11-22,1.50,1.50,120.650018
2017-11-23,1.50,1.50,120.654976
2017-11-24,1.50,1.50,120.659935
2017-11-27,1.50,1.50,120.674811
2017-11-28,1.50,1.50,120.679770
2017-11-29,1.50,1.50,120.684729
2017-11-30,1.50,1.50,120.689689
2017-12-01,1.50,1.50,120.694649
2017-12-04,1.50,1.50,120.709529
2017-12-05,1.50,1.50,120.714490
2017-12-06,1.50,1.50,120.719450
2017-12-07,1.50,1.50,120.724412
2017-12-08,1.50,1.50,120.729373
2017-12-11,1.50,1.50,120.744257
2017-12-12,1.50,1.50,120.749219
2017-12-13,1.50,1.50,120.754182
2017-12-14,1.50,1.50,120.759144
2017-12-15,1.50,1.50,120.764107
2017-12-18,1.50,1.50,120.778996
2017-12-19,1.50,1.50,120.783959
2017-12-20,1.50,1.50,120.788923
2017-12-21,1.50,1.50,120.793887
2017-12-22,1.50,1.50,120.798851
2017-12-27,1.50,1.50,120.823673
2017-12-28,1.50,1.50,120.828638
2017-12-29,1.50,1.50,120.833603
2018-01-02,1.50,1.50,120.853467
2018-01-03,1.50,1.50,120.858433
2018-01-04,1.50,1.50,120.863400
2018-01-05,1.50,1.50,120.868367
2018-01-08,1.50,1.50,120.883268
2018-01-09,1.50,1.50,120.888236
2018-01-10,1.50,1.50,120.893204
2018-01-11,1.50,1.50,120.898172
2018-01-12,1.50,1.50,120.903141
2018-01-15,1.50,1.50,120.918047
2018-01-16,1.50,1.50,120.923016
2018-01-17,1.50,1.50,120.927985
2018-01-18,1.50,1.50,120.932955
2018-01-19,1.50,1.50,120.937925
2018-01-22,1.50,1.50,120.952835
2018-01-23,1.50,1.50,120.957806
2018-01-24,1.50,1.50,120.962777
2018-01-25,1.50,1.50,120.967748
2018-01-29,1.50,1.50,120.987633
2018-01-30,1.50,1.50,120.992605
2018-01-31,1.50,1.50,120.997577
2018-02-01,1.50,1.50,121.002550
2018-02-02,1.50,1.50,121.007522
2018-02-05,1.50,1.50,121.022441
2018-02-06,1.50,1.50,121.027415
2018-02-07,1.50,1.50,121.032388
2018-02-08,1.50,1.50,121.037362
2018-02-09,1.50,1.50,121.042336
2018-02-12,1.50,1.50,121.057260
2018-02-13,1.50,1.50,121.062234
2018-02-14,1.50,1.50,121.067210
2018-02-15,1.50,1.50,121.072185
2018-02-16,1.50,1.50,121.077161
2018-02-19,1.50,1.50,121.092088
2018-02-20,1.50,1.50,121.097064
2018-02-21,1.50,1.50,121.102041
2018-02-22,1.50,1.50,121.107018
2018-02-23,1.50,1.50,121.111995
2018-02-26,1.50,1.50,121.126926
2018-02-27,1.50,1.50,121.131904
2018-02-28,1.50,1.50,121.136882
2018-03-01,1.50,1.50,121.141860
2018-03-02,1.50,1.50,121.146839
2018-03-05,1.50,1.50,121.161775
2018-03-06,1.50,1.50,121.166754
2018-03-07,1.50,1.50,121.171733
2018-03-08,1.50,1.50,121.176713
2018-03-09,1.50,1.50,121.181693
2018-03-12,1.50,1.50,121.196633
2018-03-13,1.50,1.50,121.201614
2018-03-14,1.50,1.50,121.206595
2018-03-15,1.50,1.50,121.211576
2018-03-16,1.50,1.50,121.216557
2018-03-19,1.50,1.50,121.231502
2018-03-20,1.50,1.50,121.236484
2018-03-21,1.50,1.50,121.241466
2018-03-22,1.50,1.50,121.246449
2018-03-23,1.50,1.50,121.251431
2018-03-26,1.50,1.50,121.266380
2018-03-27,1.50,1.50,121.271364
2018-03-28,1.50,1.50,121.276347
2018-03-29,1.50,1.50,121.281331
2018-04-03,1.50,1.50,121.306252
2018-04-04,1.50,1.50,121.311237
2018-04-05,1.50,1.50,121.316223
2018-04-06,1.50,1.50,121.321208
2018-04-09,1.50,1.50,121.336166
2018-04-10,1.50,1.50,121.341152
2018-04-11,1.50,1.50,121.346139
2018-04-12,1.50,1.50,121.351126
2018-04-13,1.50,1.50,121.356113
2018-04-16,1.50,1.50,121.371074
2018-04-17,1.50,1.50,121.376062
2018-04-18,1.50,1.50,121.381050
2018-04-19,1.50,1.50,121.386039
2018-04-20,1.50,1.50,121.391027
2018-04-23,1.50,1.50,121.405993
2018-04-24,1.50,1.50,121.410982
2018-04-26,1.50,1.50,121.420961
2018-04-27,1.50,1.50,121.425951
2018-04-30,1.50,1.50,121.440922
2018-05-01,1.50,1.50,121.445912
2018-05-02,1.50,1.50,121.450903
2018-05-03,1.50,1.50,121.455894
2018-05-04,1.50,1.50,121.460886
2018-05-07,1.50,1.50,121.475860
2018-05-08,1.50,1.50,121.480852
2018-05-09,1.50,1.50,121.485845
2018-05-10,1.50,1.50,121.490837
2018-05-11,1.50,1.50,121.495830
2018-05-14,1.50,1.50,121.510809
2018-05-15,1.50,1.50,121.515803
2018-05-16,1.50,1.50,121.520796
2018-05-17,1.50,1.50,121.525790
2018-05-18,1.50,1.50,121.530785
2018-05-21,1.50,1.50,121.545768
2018-05-22,1.50,1.50,121.550763
2018-05-23,1.50,1.50,121.555758
2018-05-24,1.50,1.50,121.560754
2018-05-25,1.50,1.50,121.565749
2018-05-28,1.50,1.50,121.580737
2018-05-29,1.50,1.50,121.585733
2018-05-30,1.50,1.50,121.590730
2018-05-31,1.50,1.50,121.595727
2018-06-01,1.50,1.50,121.600724
2018-06-04,1.50,1.50,121.615716
2018-06-05,1.50,1.50,121.620714
2018-06-06,1.50,1.50,121.625712
2018-06-07,1.50,1.50,121.630710
2018-06-08,1.50,1.50,121.635709
2018-06-12,1.50,1.50,121.655704
2018-06-13,1.50,1.50,121.660703
2018-06-14,1.50,1.50,121.665703
2018-06-15,1.50,1.50,121.670703
2018-06-18,1.50,1.50,121.685703
2018-06-19,1.50,1.50,121.690704
2018-06-20,1.50,1.50,121.695705
2018-06-21,1.50,1.50,121.700706
2018-06-22,1.50,1.50,121.705708
2018-06-25,1.50,1.50,121.720713
2018-06-26,1.50,1.50,121.725715
2018-06-27,1.50,1.50,121.730717
2018-06-28,1.50,1.50,121.735720
2018-06-29,1.50,1.50,121.740723
2018-07-02,1.50,1.50,121.755732
2018-07-03,1.50,1.50,121.760735
2018-07-04,1.50,1.50,121.765739
2018-07-05,1.50,1.50,121.770743
2018-07-06,1.50,1.50,121.775748
2018-07-09,1.50,1.50,121.790761
2018-07-10,1.50,1.50,121.795766
2018-07-11,1.50,1.50,121.800772
2018-07-12,1.50,1.50,121.805777
2018-07-13,1.50,1.50,121.810783
2018-07-16,1.50,1.50,121.825801
2018-07-17,1.50,1.50,121.830807
2018-07-18,1.50,1.50,121.835814
2018-07-19,1.50,1.50,121.840821
2018-07-20,1.50,1.50,121.845828
2018-07-23,1.50,1.50,121.860850
2018-07-24,1.50,1.50,121.865858
2018-07-25,1.50,1.50,121.870866
2018-07-26,1.50,1.50,121.875875
2018-07-27,1.50,1.50,121.880883
2018-07-30,1.50,1.50,121.895910
2018-07-31,1.50,1.50,121.900919
2018-08-01,1.50,1.50,121.905929
2018-08-02,1.50,1.50,121.910938
2018-08-03,1.50,1.50,121.915948
2018-08-06,1.50,1.50,121.930979
2018-08-07,1.50,1.50,121.935990
2018-08-08,1.50,1.50,121.941001
2018-08-09,1.50,1.50,121.946012
2018-08-10,1.50,1.50,121.951024
2018-08-13,1.50,1.50,121.966059
2018-08-14,1.50,1.50,121.971071
2018-08-15,1.50,1.50,121.976084
2018-08-16,1.50,1.50,121.981096
2018-08-17,1.50,1.50,121.986109
2018-08-20,1.50,1.50,122.001149
2018-08-21,1.50,1.50,122.006163
2018-08-22,1.50,1.50,122.011176
2018-08-23,1.50,1.50,122.016191
2018-08-24,1.50,1.50,122.021205
2018-08-27,1.50,1.50,122.036249
2018-08-28,1.50,1.50,122.041264
2018-08-29,1.50,1.50,122.046279
2018-08-30,1.50,1.50,122.051295
2018-08-31,1.50,1.50,122.056311
2018-09-03,1.50,1.50,122.071359
2018-09-04,1.50,1.50,122.076375
2018-09-05,1.50,1.50,122.081392
2018-09-06,1.50,1.50,122.086409
2018-09-07,1.50,1.50,122.091427
2018-09-10,1.50,1.50,122.106479
2018-09-11,1.50,1.50,122.111497
2018-09-12,1.50,1.50,122.116515
2018-09-13,1.50,1.50,122.121534
2018-09-14,1.50,1.50,122.126552
2018-09-17,1.50,1.50,122.141609
2018-09-18,1.50,1.50,122.146629
2018-09-19,1.50,1.50,122.151648
2018-09-20,1.50,1.50,122.156668
2018-09-21,1.50,1.50,122.161688
2018-09-24,1.50,1.50,122.176749
2018-09-25,1.50,1.50,122.181770
2018-09-26,1.50,1.50,122.186792
2018-09-27,1.50,1.50,122.191813
2018-09-28,1.50,1.50,122.196835
2018-10-01,1.50,1.50,122.211900
2018-10-02,1.50,1.50,122.216922
2018-10-03,1.50,1.50,122.221945
2018-10-04,1.50,1.50,122.226968
2018-10-05,1.50,1.50,122.231991
2018-10-08,1.50,1.50,122.247060
2018-10-09,1.50,1.50,122.252084
2018-10-10,1.50,1.50,122.257108
2018-10-11,1.50,1.50,122.262133
2018-10-12,1.50,1.50,122.267157
2018-10-15,1.50,1.50,122.282231
2018-10-16,1.50,1.50,122.287256
2018-10-17,1.50,1.50,122.292282
2018-10-18,1.50,1.50,122.297308
2018-10-19,1.50,1.50,122.302334
2018-10-22,1.50,1.50,122.317412
2018-10-23,1.50,1.50,122.322439
2018-10-24,1.50,1.50,122.327466
2018-10-25,1.50,1.50,122.332493
2018-10-26,1.50,1.50,122.337520
2018-10-29,1.50,1.50,122.352603
2018-10-30,1.50,1.50,122.357631
2018-10-31,1.50,1.50,122.362659
2018-11-01,1.50,1.50,122.367688
2018-11-02,1.50,1.50,122.372717
2018-11-05,1.50,1.50,122.387804
2018-11-06,1.50,1.50,122.392834
2018-11-07,1.50,1.50,122.397863
2018-11-08,1.50,1.50,122.402893
2018-11-09,1.50,1.50,122.407924
2018-11-12,1.50,1.50,122.423015
2018-11-13,1.50,1.50,122.428046
2018-11-14,1.50,1.50,122.433077
2018-11-15,1.50,1.50,122.438109
2018-11-16,1.50,1.50,122.443141
2018-11-19,1.50,1.50,122.458236
2018-11-20,1.50,1.50,122.463269
2018-11-21,1.50,1.50,122.468302
2018-11-22,1.50,1.50,122.473335
2018-11-23,1.50,1.50,122.478368
2018-11-26,1.50,1.50,122.493468
2018-11-27,1.50,1.50,122.498502
2018-11-28,1.50,1.50,122.503536
2018-11-29,1.50,1.50,122.508570
2018-11-30,1.50,1.50,122.513605
2018-12-03,1.50,1.50,122.528709
2018-12-04,1.50,1.50,122.533745
2018-12-05,1.50,1.50,122.538780
2018-12-06,1.50,1.50,122.543816
2018-12-07,1.50,1.50,122.548852
2018-12-10,1.50,1.50,122.563961
2018-12-11,1.50,1.50,122.568998
2018-12-12,1.50,1.50,122.574035
2018-12-13,1.50,1.50,122.579072
2018-12-14,1.50,1.50,122.584110
2018-12-17,1.50,1.50,122.599223
2018-12-18,1.50,1.50,122.604261
2018-12-19,1.50,1.50,122.609300
2018-12-20,1.50,1.50,122.614339
2018-12-21,1.50,1.50,122.619377
2018-12-24,1.50,1.50,122.634495
2018-12-27,1.50,1.50,122.649614
2018-12-28,1.50,1.50,122.654655
2018-12-31,1.50,1.50,122.669776
2019-01-02,1.50,1.50,122.679859
2019-01-03,1.50,1.50,122.684901
2019-01-04,1.50,1.50,122.689942
2019-01-07,1.50,1.50,122.705069
2019-01-08,1.50,1.50,122.710111
2019-01-09,1.50,1.50,122.715154
2019-01-10,1.50,1.50,122.720197
2019-01-11,1.50,1.50,122.725241
2019-01-14,1.50,1.50,122.740371
2019-01-15,1.50,1.50,122.745415
2019-01-16,1.50,1.50,122.750459
2019-01-17,1.50,1.50,122.755504
2019-01-18,1.50,1.50,122.760549
2019-01-21,1.50,1.50,122.775684
2019-01-22,1.50,1.50,122.780729
2019-01-23,1.50,1.50,122.785775
2019-01-24,1.50,1.50,122.790821
2019-01-25,1.50,1.50,122.795867
2019-01-29,1.50,1.50,122.816053
2019-01-30,1.50,1.50,122.821100
2019-01-31,1.50,1.50,122.826147
2019-02-01,1.50,1.50,122.831195
2019-02-04,1.50,1.50,122.846339
2019-02-05,1.50,1.50,122.851387
2019-02-06,1.50,1.50,122.856436
2019-02-07,1.50,1.50,122.861485
2019-02-08,1.50,1.50,122.866534
2019-02-11,1.50,1.50,122.881682
2019-02-12,1.50,1.50,122.886732
2019-02-13,1.50,1.50,122.891782
2019-02-14,1.50,1.50,122.896832
2019-02-15,1.50,1.50,122.901883
2019-02-18,1.50,1.50,122.917035
2019-02-19,1.50,1.50,122.922086
2019-02-20,1.50,1.50,122.927138
2019-02-21,1.50,1.50,122.932190
2019-02-22,1.50,1.50,122.937242
2019-02-25,1.50,1.50,122.952398
2019-02-26,1.50,1.50,122.957451
2019-02-27,1.50,1.50,122.962504
2019-02-28,1.50,1.50,122.967558
2019-03-01,1.50,1.50,122.972611
2019-03-04,1.50,1.50,122.987772
2019-03-05,1.50,1.50,122.992826
2019-03-06,1.50,1.50,122.997881
2019-03-07,1.50,1.50,123.002936
2019-03-08,1.50,1.50,123.007990
2019-03-11,1.50,1.50,123.023156
2019-03-12,1.50,1.50,123.028212
2019-03-13,1.50,1.50,123.033268
2019-03-14,1.50,1.50,123.038324
2019-03-15,1.50,1.50,123.043380
2019-03-18,1.50,1.50,123.058550
2019-03-19,1.50,1.50,123.063607
2019-03-20,1.50,1.50,123.068664
2019-03-21,1.50,1.50,123.073722
2019-03-22,1.50,1.50,123.078780
2019-03-25,1.50,1.50,123.093954
2019-03-26,1.50,1.50,123.099013
2019-03-27,1.50,1.50,123.104071
2019-03-28,1.50,1.50,123.109131
2019-03-29,1.50,1.50,123.114190
2019-04-01,1.50,1.50,123.129368
2019-04-02,1.50,1.50,123.134428
2019-04-03,1.50,1.50,123.139489
2019-04-04,1.50,1.50,123.144549
2019-04-05,1.50,1.50,123.149610
2019-04-08,1.50,1.50,123.164793
2019-04-09,1.50,1.50,123.169854
2019-04-10,1.50,1.50,123.174916
2019-04-11,1.50,1.50,123.179978
2019-04-12,1.50,1.50,123.185040
2019-04-15,1.50,1.50,123.200228
2019-04-16,1.50,1.50,123.205291
2019-04-17,1.50,1.50,123.210354
2019-04-18,1.50,1.50,123.215417
2019-04-23,1.50,1.50,123.240735
2019-04-24,1.50,1.50,123.245800
2019-04-26,1.50,1.50,123.255930
2019-04-29,1.50,1.50,123.271126
2019-04-30,1.50,1.50,123.276192
2019-05-01,1.50,1.50,123.281258
2019-05-02,1.50,1.50,123.286324
2019-05-03,1.50,1.50,123.291391
2019-05-06,1.50,1.50,123.306591
2019-05-07,1.50,1.50,123.311659
2019-05-08,1.50,1.50,123.316726
2019-05-09,1.50,1.50,123.321794
2019-05-10,1.50,1.50,123.326862
2019-05-13,1.50,1.50,123.342067
2019-05-14,1.50,1.50,123.347136
2019-05-15,1.50,1.50,123.352205
2019-05-16,1.50,1.50,123.357274
2019-05-17,1.50,1.50,123.362343
2019-05-20,1.50,1.50,123.377552
2019-05-21,1.50,1.50,123.382623
2019-05-22,1.50,1.50,123.387693
2019-05-23,1.50,1.50,123.392764
2019-05-24,1.50,1.50,123.397835
2019-05-27,1.50,1.50,123.413048
2019-05-28,1.50,1.50,123.418120
2019-05-29,1.50,1.50,123.423192
2019-05-30,1.50,1.50,123.428264
2019-05-31,1.50,1.50,123.433337
2019-06-03,1.50,1.50,123.448554
2019-06-04,1.50,1.50,123.453628
2019-06-05,1.25,1.25,123.458701
2019-06-06,1.25,1.25,123.462929
2019-06-07,1.25,1.25,123.467157
2019-06-11,1.25,1.25,123.484071
2019-06-12,1.25,1.25,123.488300
2019-06-13,1.25,1.25,123.492529
2019-06-14,1.25,1.25,123.496758
2019-06-17,1.25,1.25,123.509446
2019-06-18,1.25,1.25,123.513676
2019-06-19,1.25,1.25,123.517906
2019-06-20,1.25,1.25,123.522136
2019-06-21,1.25,1.25,123.526366
2019-06-24,1.25,1.25,123.539057
2019-06-25,1.25,1.25,123.543288
2019-06-26,1.25,1.25,123.547519
2019-06-27,1.25,1.25,123.551750
2019-06-28,1.25,1.25,123.555981
2019-07-01,1.25,1.25,123.568675
2019-07-02,1.25,1.25,123.572907
2019-07-03,1.00,1.00,123.577139
2019-07-04,1.00,1.00,123.580524
2019-07-05,1.00,1.00,123.583910
2019-07-08,1.00,1.00,123.594068
2019-07-09,1.00,1.00,123.597454
2019-07-10,1.00,1.00,123.600840
2019-07-11,1.00,1.00,123.604226
2019-07-12,1.00,1.00,123.607613
2019-07-15,1.00,1.00,123.617772
2019-07-16,1.00,1.00,123.621159
2019-07-17,1.00,1.00,123.624546
2019-07-18,1.00,1.00,123.627933
2019-07-19,1.00,1.00,123.631320
2019-07-22,1.00,1.00,123.641482
2019-07-23,1.00,1.00,123.644869
2019-07-24,1.00,1.00,123.648257
2019-07-25,1.00,1.00,123.651644
2019-07-26,1.00,1.00,123.655032
2019-07-29,1.00,1.00,123.665195
2019-07-30,1.00,1.00,123.668583
2019-07-31,1.00,1.00,123.671972
2019-08-01,1.00,1.00,123.675360
2019-08-02,1.00,1.00,123.678748
2019-08-05,1.00,1.00,123.688914
2019-08-06,1.00,1.00,123.692302
2019-08-07,1.00,1.00,123.695691
2019-08-08,1.00,1.00,123.699080
2019-08-09,1.00,1.00,123.702469
2019-08-12,1.00,1.00,123.712636
2019-08-13,1.00,1.00,123.716026
2019-08-14,1.00,1.00,123.719415
2019-08-15,1.00,1.00,123.722805
2019-08-16,1.00,1.00,123.726195
2019-08-19,1.00,1.00,123.736364
2019-08-20,1.00,1.00,123.739754
2019-08-21,1.00,1.00,123.743144
2019-08-22,1.00,1.00,123.746534
2019-08-23,1.00,1.00,123.749925
2019-08-26,1.00,1.00,123.760096
2019-08-27,1.00,1.00,123.763486
2019-08-28,1.00,1.00,123.766877
2019-08-29,1.00,1.00,123.770268
2019-08-30,1.00,1.00,123.773659
2019-09-02,1.00,1.00,123.783832
2019-09-03,1.00,1.00,123.787224
2019-09-04,1.00,1.00,123.790615
2019-09-05,1.00,1.00,123.794007
2019-09-06,1.00,1.00,123.797398
2019-09-09,1.00,1.00,123.807573
2019-09-10,1.00,1.00,123.810965
2019-09-11,1.00,1.00,123.814357
2019-09-12,1.00,1.00,123.817750
2019-09-13,1.00,1.00,123.821142
2019-09-16,1.00,1.00,123.831319
2019-09-17,1.00,1.00,123.834712
2019-09-18,1.00,1.00,123.838104
2019-09-19,1.00,1.00,123.841497
2019-09-20,1.00,1.00,123.844890
2019-09-23,1.00,1.00,123.855069
2019-09-24,1.00,1.00,123.858462
2019-09-25,1.00,1.00,123.861856
2019-09-26,1.00,1.00,123.865249
2019-09-27,1.00,1.00,123.868643
2019-09-30,1.00,1.00,123.878824
2019-10-01,1.00,1.00,123.882218
2019-10-02,0.75,0.75,123.885612
2019-10-03,0.75,0.75,123.888157
2019-10-04,0.75,0.75,123.890703
2019-10-07,0.75,0.75,123.898340
2019-10-08,0.75,0.75,123.900886
2019-10-09,0.75,0.75,123.903432
2019-10-10,0.75,0.75,123.905978
2019-10-11,0.75,0.75,123.908524
2019-10-14,0.75,0.75,123.916162
2019-10-15,0.75,0.75,123.918708
2019-10-16,0.75,0.75,123.921254
2019-10-17,0.75,0.75,123.923801
2019-10-18,0.75,0.75,123.926347
2019-10-21,0.75,0.75,123.933986
2019-10-22,0.75,0.75,123.936533
2019-10-23,0.75,0.75,123.939080
2019-10-24,0.75,0.75,123.941626
2019-10-25,0.75,0.75,123.944173
2019-10-28,0.75,0.75,123.951814
2019-10-29,0.75,0.75,123.954361
2019-10-30,0.75,0.75,123.956908
2019-10-31,0.75,0.75,123.959455
2019-11-01,0.75,0.75,123.962002
2019-11-04,0.75,0.75,123.969643
2019-11-05,0.75,0.75,123.972191
2019-11-06,0.75,0.75,123.974738
2019-11-07,0.75,0.75,123.977285
2019-11-08,0.75,0.75,123.979833
2019-11-11,0.75,0.75,123.987475
2019-11-12,0.75,0.75,123.990023
2019-11-13,0.75,0.75,123.992571
2019-11-14,0.75,0.75,123.995119
2019-11-15,0.75,0.75,123.997666
2019-11-18,0.75,0.75,124.005310
2019-11-19,0.75,0.75,124.007858
2019-11-20,0.75,0.75,124.010406
2019-11-21,0.75,0.75,124.012954
2019-11-22,0.75,0.75,124.015503
2019-11-25,0.75,0.75,124.023147
2019-11-26,0.75,0.75,124.025696
2019-11-27,0.75,0.75,124.028244
2019-11-28,0.75,0.75,124.030793
2019-11-29,0.75,0.75,124.033341
2019-12-02,0.75,0.75,124.040987
2019-12-03,0.75,0.75,124.043536
2019-12-04,0.75,0.75,124.046085
2019-12-05,0.75,0.75,124.048634
2019-12-06,0.75,0.75,124.051183
2019-12-09,0.75,0.75,124.058830
2019-12-10,0.75,0.75,124.061379
2019-12-11,0.75,0.75,124.063928
2019-12-12,0.75,0.75,124.066477
2019-12-13,0.75,0.75,124.069027
2019-12-16,0.75,0.75,124.076675
2019-12-17,0.75,0.75,124.079224
2019-12-18,0.75,0.75,124.081774
2019-12-19,0.75,0.75,124.084324
2019-12-20,0.75,0.75,124.086873
2019-12-23,0.75,0.75,124.094522
2019-12-24,0.75,0.75,124.097072
2019-12-27,0.75,0.75,124.104722
2019-12-30,0.75,0.75,124.112372
2019-12-31,0.75,0.75,124.114923
2020-01-02,0.75,0.75,124.120023
2020-01-03,0.75,0.75,124.122574
2020-01-06,0.75,0.75,124.130225
2020-01-07,0.75,0.75,124.132776
2020-01-08,0.75,0.75,124.135326
2020-01-09,0.75,0.75,124.137877
2020-01-10,0.75,0.75,124.140428
2020-01-13,0.75,0.75,124.148080
2020-01-14,0.75,0.75,124.150631
2020-01-15,0.75,0.75,124.153182
2020-01-16,0.75,0.75,124.155733
2020-01-17,0.75,0.75,124.158285
2020-01-20,0.75,0.75,124.165938
2020-01-21,0.75,0.75,124.168490
2020-01-22,0.75,0.75,124.171041
2020-01-23,0.75,0.75,124.173592
2020-01-24,0.75,0.75,124.176144
2020-01-28,0.75,0.75,124.186350
2020-01-29,0.75,0.75,124.188902
2020-01-30,0.75,0.75,124.191454
2020-01-31,0.75,0.75,124.194006
2020-02-03,0.75,0.75,124.201661
2020-02-04,0.75,0.75,124.204214
2020-02-05,0.75,0.75,124.206766
2020-02-06,0.75,0.75,124.209318
2020-02-07,0.75,0.75,124.211870
2020-02-10,0.75,0.75,124.219527
2020-02-11,0.75,0.75,124.222080
2020-02-12,0.75,0.75,124.224632
2020-02-13,0.75,0.75,124.227185
2020-02-14,0.75,0.75,124.229737
2020-02-17,0.75,0.75,124.237395
2020-02-18,0.75,0.75,124.239948
2020-02-19,0.75,0.75,124.242501
2020-02-20,0.75,0.75,124.245054
2020-02-21,0.75,0.75,124.247607
2020-02-24,0.75,0.75,124.255266
2020-02-25,0.75,0.75,124.257819
2020-02-26,0.75,0.75,124.260372
2020-02-27,0.75,0.75,124.262926
2020-02-28,0.75,0.75,124.265479
2020-03-02,0.75,0.75,124.273139
2020-03-03,0.75,0.75,124.275693
2020-03-04,0.50,0.50,124.278246
2020-03-05,0.50,0.50,124.279949
2020-03-06,0.50,0.50,124.281651
2020-03-09,0.50,0.50,124.286759
2020-03-10,0.50,0.50,124.288461
2020-03-11,0.50,0.50,124.290164
2020-03-12,0.50,0.50,124.291866
2020-03-13,0.50,0.50,124.293569
2020-03-16,0.50,0.50,124.298677
2020-03-17,0.50,0.50,124.300380
2020-03-18,0.50,0.50,124.302083
2020-03-19,0.50,0.50,124.303785
2020-03-20,0.25,0.25,124.305488
2020-03-23,0.25,0.25,124.308042
2020-03-24,0.25,0.25,124.308894
2020-03-25,0.25,0.24,124.309745
2020-03-26,0.25,0.23,124.310563
2020-03-27,0.25,0.22,124.311346
2020-03-30,0.25,0.23,124.313594
2020-03-31,0.25,0.20,124.314377
2020-04-01,0.25,0.20,124.315058
2020-04-02,0.25,0.19,124.315739
2020-04-03,0.25,0.18,124.316387
2020-04-06,0.25,0.18,124.318226
2020-04-07,0.25,0.17,124.318839
2020-04-08,0.25,0.17,124.319418
2020-04-09,0.25,0.17,124.319997
2020-04-14,0.25,0.17,124.322892
2020-04-15,0.25,0.16,124.323471
2020-04-16,0.25,0.15,124.324016
2020-04-17,0.25,0.15,124.324527
2020-04-20,0.25,0.15,124.326060
2020-04-21,0.25,0.14,124.326571
2020-04-22,0.25,0.14,124.327047
2020-04-23,0.25,0.14,124.327524
2020-04-24,0.25,0.14,124.328001
2020-04-27,0.25,0.13,124.329432
2020-04-28,0.25,0.13,124.329875
2020-04-29,0.25,0.13,124.330317
2020-04-30,0.25,0.14,124.330760
2020-05-01,0.25,0.14,124.331237
2020-05-04,0.25,0.14,124.332668
2020-05-05,0.25,0.14,124.333145
2020-05-06,0.25,0.14,124.333622
2020-05-07,0.25,0.14,124.334099
2020-05-08,0.25,0.14,124.334575
2020-05-11,0.25,0.14,124.336006
2020-05-12,0.25,0.14,124.336483
2020-05-13,0.25,0.14,124.336960
2020-05-14,0.25,0.13,124.337437
2020-05-15,0.25,0.13,124.337880
2020-05-18,0.25,0.13,124.339208
2020-05-19,0.25,0.13,124.339651
2020-05-20,0.25,0.13,124.340094
2020-05-21,0.25,0.13,124.340537
2020-05-22,0.25,0.13,124.340980
2020-05-25,0.25,0.13,124.342308
2020-05-26,0.25,0.13,124.342751
2020-05-27,0.25,0.13,124.343194
2020-05-28,0.25,0.14,124.343637
2020-05-29,0.25,0.14,124.344114
2020-06-01,0.25,0.15,124.345545
2020-06-02,0.25,0.15,124.346056
2020-06-03,0.25,0.14,124.346567
2020-06-04,0.25,0.14,124.347044
2020-06-05,0.25,0.14,124.347520
2020-06-09,0.25,0.14,124.349428
2020-06-10,0.25,0.14,124.349905
2020-06-11,0.25,0.14,124.350382
2020-06-12,0.25,0.14,124.350859
2020-06-15,0.25,0.13,124.352290
2020-06-16,0.25,0.13,124.352733
2020-06-17,0.25,0.13,124.353176
2020-06-18,0.25,0.13,124.353619
2020-06-19,0.25,0.13,124.354062
2020-06-22,0.25,0.13,124.355390
2020-06-23,0.25,0.13,124.355833
2020-06-24,0.25,0.13,124.356276
2020-06-25,0.25,0.13,124.356719
2020-06-26,0.25,0.13,124.357162
2020-06-29,0.25,0.13,124.358491
2020-06-30,0.25,0.14,124.358934
2020-07-01,0.25,0.14,124.359411
2020-07-02,0.25,0.14,124.359888
2020-07-03,0.25,0.13,124.360365
2020-07-06,0.25,0.13,124.361693
2020-07-07,0.25,0.13,124.362136
2020-07-08,0.25,0.13,124.362579
2020-07-09,0.25,0.13,124.363022
2020-07-10,0.25,0.13,124.363465
2020-07-13,0.25,0.13,124.364794
2020-07-14,0.25,0.13,124.365237
2020-07-15,0.25,0.13,124.365680
2020-07-16,0.25,0.13,124.366123
2020-07-17,0.25,0.13,124.366566
2020-07-20,0.25,0.13,124.367895
2020-07-21,0.25,0.13,124.368338
2020-07-22,0.25,0.13,124.368781
2020-07-23,0.25,0.13,124.369223
2020-07-24,0.25,0.13,124.369666
2020-07-27,0.25,0.13,124.370995
2020-07-28,0.25,0.13,124.371438
2020-07-29,0.25,0.13,124.371881
2020-07-30,0.25,0.13,124.372324
2020-07-31,0.25,0.13,124.372767
2020-08-03,0.25,0.13,124.374096
2020-08-04,0.25,0.13,124.374539
2020-08-05,0.25,0.14,124.374982
2020-08-06,0.25,0.14,124.375459
2020-08-07,0.25,0.14,124.375936
2020-08-10,0.25,0.13,124.377367
2020-08-11,0.25,0.13,124.377810
2020-08-12,0.25,0.13,124.378253
2020-08-13,0.25,0.13,124.378696
2020-08-14,0.25,0.13,124.379139
2020-08-17,0.25,0.13,124.380468
2020-08-18,0.25,0.13,124.380911
2020-08-19,0.25,0.13,124.381354
2020-08-20,0.25,0.13,124.381797
2020-08-21,0.25,0.13,124.382240
2020-08-24,0.25,0.13,124.383569
2020-08-25,0.25,0.13,124.384012
2020-08-26,0.25,0.13,124.384455
2020-08-27,0.25,0.13,124.384898
2020-08-28,0.25,0.13,124.385341
2020-08-31,0.25,0.13,124.386670
2020-09-01,0.25,0.13,124.387113
2020-09-02,0.25,0.13,124.387556
2020-09-03,0.25,0.13,124.387999
2020-09-04,0.25,0.13,124.388442
2020-09-07,0.25,0.13,124.389772
2020-09-08,0.25,0.13,124.390215
2020-09-09,0.25,0.13,124.390658
2020-09-10,0.25,0.13,124.391101
2020-09-11,0.25,0.13,124.391544
2020-09-14,0.25,0.13,124.392873
2020-09-15,0.25,0.13,124.393316
2020-09-16,0.25,0.13,124.393759
2020-09-17,0.25,0.13,124.394202
2020-09-18,0.25,0.13,124.394645
2020-09-21,0.25,0.13,124.395974
2020-09-22,0.25,0.13,124.396417
2020-09-23,0.25,0.13,124.396860
2020-09-24,0.25,0.13,124.397303
2020-09-25,0.25,0.13,124.397746
2020-09-28,0.25,0.13,124.399076
2020-09-29,0.25,0.13,124.399519
2020-09-30,0.25,0.13,124.399962
2020-10-01,0.25,0.13,124.400405
2020-10-02,0.25,0.13,124.400848
2020-10-05,0.25,0.13,124.402200
2020-10-06,0.25,0.13,124.402620
2020-10-07,0.25,0.13,124.403063
2020-10-08,0.25,0.13,124.403506
2020-10-09,0.25,0.13,124.403949
2020-10-12,0.25,0.13,124.405279
2020-10-13,0.25,0.13,124.405722
2020-10-14,0.25,0.13,124.406165
2020-10-15,0.25,0.13,124.406608
2020-10-16,0.25,0.13,124.407051
2020-10-19,0.25,0.13,124.408380
2020-10-20,0.25,0.13,124.408823
2020-10-21,0.25,0.13,124.409266
2020-10-22,0.25,0.13,124.409710
2020-10-23,0.25,0.13,124.410153
2020-10-26,0.25,0.13,124.411482
2020-10-27,0.25,0.13,124.411925
2020-10-28,0.25,0.13,124.412368
2020-10-29,0.25,0.13,124.412811
2020-10-30,0.25,0.13,124.413254
2020-11-02,0.25,0.13,124.414584
2020-11-03,0.25,0.13,124.415027
2020-11-04,0.10,0.04,124.415470
2020-11-05,0.10,0.04,124.415606
2020-11-06,0.10,0.04,124.415743
2020-11-09,0.10,0.05,124.416152
2020-11-10,0.10,0.05,124.416322
2020-11-11,0.10,0.05,124.416493
2020-11-12,0.10,0.05,124.416663
2020-11-13,0.10,0.05,124.416833
2020-11-16,0.10,0.05,124.417345
2020-11-17,0.10,0.05,124.417515
2020-11-18,0.10,0.05,124.417686
2020-11-19,0.10,0.05,124.417856
2020-11-20,0.10,0.05,124.418027
2020-11-23,0.10,0.05,124.418538
2020-11-24,0.10,0.05,124.418708
2020-11-25,0.10,0.05,124.418879
2020-11-26,0.10,0.05,124.419049
2020-11-27,0.10,0.05,124.419220
2020-11-30,0.10,0.05,124.419731
2020-12-01,0.10,0.05,124.419901
2020-12-02,0.10,0.04,124.420072
2020-12-03,0.10,0.04,124.420208
2020-12-04,0.10,0.04,124.420344
2020-12-07,0.10,0.04,124.420753
2020-12-08,0.10,0.04,124.420890
2020-12-09,0.10,0.04,124.421026
2020-12-10,0.10,0.04,124.421163
2020-12-11,0.10,0.04,124.421299
2020-12-14,0.10,0.04,124.421708
2020-12-15,0.10,0.04,124.421844
2020-12-16,0.10,0.04,124.421981
2020-12-17,0.10,0.04,124.422117
2020-12-18,0.10,0.04,124.422253
2020-12-21,0.10,0.04,124.422662
2020-12-22,0.10,0.04,124.422799
2020-12-23,0.10,0.04,124.422935
2020-12-24,0.10,0.04,124.423071
2020-12-29,0.10,0.04,124.423753
2020-12-30,0.10,0.04,124.423890
2020-12-31,0.10,0.04,124.424026
2021-01-04,0.10,0.04,124.424571
2021-01-05,0.10,0.04,124.424708
2021-01-06,0.10,0.03,124.424844
2021-01-07,0.10,0.03,124.424946
2021-01-08,0.10,0.03,124.425049
2021-01-11,0.10,0.03,124.425355
2021-01-12,0.10,0.03,124.425458
2021-01-13,0.10,0.03,124.425560
2021-01-14,0.10,0.03,124.425662
2021-01-15,0.10,0.03,124.425765
2021-01-18,0.10,0.03,124.426071
2021-01-19,0.10,0.03,124.426174
2021-01-20,0.10,0.03,124.426276
2021-01-21,0.10,0.03,124.426378
2021-01-22,0.10,0.03,124.426480
2021-01-25,0.10,0.03,124.426787
2021-01-27,0.10,0.03,124.426992
2021-01-28,0.10,0.03,124.427094
2021-01-29,0.10,0.03,124.427196
2021-02-01,0.10,0.03,124.427503
2021-02-02,0.10,0.03,124.427605
2021-02-03,0.10,0.03,124.427708
2021-02-04,0.10,0.03,124.427810
2021-02-05,0.10,0.03,124.427912
2021-02-08,0.10,0.03,124.428219
2021-02-09,0.10,0.03,124.428321
2021-02-10,0.10,0.03,124.428423
2021-02-11,0.10,0.03,124.428526
2021-02-12,0.10,0.03,124.428628
2021-02-15,0.10,0.03,124.428935
2021-02-16,0.10,0.03,124.429037
2021-02-17,0.10,0.03,124.429139
2021-02-18,0.10,0.03,124.429242
2021-02-19,0.10,0.03,124.429344
2021-02-22,0.10,0.03,124.429651
2021-02-23,0.10,0.03,124.429753
2021-02-24,0.10,0.03,124.429855
2021-02-25,0.10,0.03,124.429958
2021-02-26,0.10,0.03,124.430060
2021-03-01,0.10,0.03,124.430367
2021-03-02,0.10,0.03,124.430469
2021-03-03,0.10,0.03,124.430571
2021-03-04,0.10,0.03,124.430673
2021-03-05,0.10,0.03,124.430776
2021-03-08,0.10,0.03,124.431083
2021-03-09,0.10,0.03,124.431185
2021-03-10,0.10,0.03,124.431287
2021-03-11,0.10,0.03,124.431389
2021-03-12,0.10,0.03,124.431492
2021-03-15,0.10,0.03,124.431798
2021-03-16,0.10,0.03,124.431901
2021-03-17,0.10,0.03,124.432003
2021-03-18,0.10,0.03,124.432105
2021-03-19,0.10,0.03,124.432208
2021-03-22,0.10,0.03,124.432514
2021-03-23,0.10,0.03,124.432617
2021-03-24,0.10,0.03,124.432719
2021-03-25,0.10,0.03,124.432821
2021-03-26,0.10,0.03,124.432923
2021-03-29,0.10,0.03,124.433230
2021-03-30,0.10,0.03,124.433333
2021-03-31,0.10,0.03,124.433435
2021-04-01,0.10,0.03,124.433537
2021-04-06,0.10,0.03,124.434048
2021-04-07,0.10,0.03,124.434151
2021-04-08,0.10,0.03,124.434253
2021-04-09,0.10,0.03,124.434355
2021-04-12,0.10,0.03,124.434662
2021-04-13,0.10,0.03,124.434764
2021-04-14,0.10,0.03,124.434867
2021-04-15,0.10,0.03,124.434969
2021-04-16,0.10,0.03,124.435071
2021-04-19,0.10,0.03,124.435378
2021-04-20,0.10,0.03,124.435480
2021-04-21,0.10,0.03,124.435583
2021-04-22,0.10,0.03,124.435685
2021-04-23,0.10,0.03,124.435787
2021-04-26,0.10,0.03,124.436094
2021-04-27,0.10,0.03,124.436196
2021-04-28,0.10,0.03,124.436299
2021-04-29,0.10,0.03,124.436401
2021-04-30,0.10,0.03,124.436503
2021-05-03,0.10,0.03,124.436810
2021-05-04,0.10,0.03,124.436912
2021-05-05,0.10,0.03,124.437014
2021-05-06,0.10,0.03,124.437117
2021-05-07,0.10,0.03,124.437219
2021-05-10,0.10,0.03,124.437526
2021-05-11,0.10,0.03,124.437628
2021-05-12,0.10,0.03,124.437730
2021-05-13,0.10,0.03,124.437833
2021-05-14,0.10,0.03,124.437935
2021-05-17,0.10,0.03,124.438242
2021-05-18,0.10,0.03,124.438344
2021-05-19,0.10,0.03,124.438446
2021-05-20,0.10,0.03,124.438549
2021-05-21,0.10,0.03,124.438651
2021-05-24,0.10,0.03,124.438958
2021-05-25,0.10,0.03,124.439060
2021-05-26,0.10,0.03,124.439162
2021-05-27,0.10,0.03,124.439265
2021-05-28,0.10,0.03,124.439367
2021-05-31,0.10,0.03,124.439674
2021-06-01,0.10,0.03,124.439776
2021-06-02,0.10,0.03,124.439878
2021-06-03,0.10,0.03,124.439981
2021-06-04,0.10,0.03,124.440083
2021-06-07,0.10,0.03,124.440390
2021-06-08,0.10,0.03,124.440492
2021-06-09,0.10,0.03,124.440594
2021-06-10,0.10,0.03,124.440696
2021-06-11,0.10,0.03,124.440799
2021-06-15,0.10,0.03,124.441208
2021-06-16,0.10,0.03,124.441310
2021-06-17,0.10,0.03,124.441412
2021-06-18,0.10,0.03,124.441515
2021-06-21,0.10,0.03,124.441822
2021-06-22,0.10,0.03,124.441924
2021-06-23,0.10,0.03,124.442026
2021-06-24,0.10,0.03,124.442128
2021-06-25,0.10,0.03,124.442231
2021-06-28,0.10,0.03,124.442538
2021-06-29,0.10,0.03,124.442640
2021-06-30,0.10,0.03,124.442742
2021-07-01,0.10,0.03,124.442844
2021-07-02,0.10,0.03,124.442947
2021-07-05,0.10,0.03,124.443254
2021-07-06,0.10,0.03,124.443356
2021-07-07,0.10,0.03,124.443458
2021-07-08,0.10,0.03,124.443560
2021-07-09,0.10,0.03,124.443663
2021-07-12,0.10,0.03,124.443969
2021-07-13,0.10,0.03,124.444072
2021-07-14,0.10,0.03,124.444174
2021-07-15,0.10,0.03,124.444276
2021-07-16,0.10,0.03,124.444379
2021-07-19,0.10,0.03,124.444685
2021-07-20,0.10,0.03,124.444788
2021-07-21,0.10,0.03,124.444890
2021-07-22,0.10,0.03,124.444992
2021-07-23,0.10,0.03,124.445095
2021-07-26,0.10,0.03,124.445401
2021-07-27,0.10,0.03,124.445504
2021-07-28,0.10,0.03,124.445606
2021-07-29,0.10,0.03,124.445708
2021-07-30,0.10,0.03,124.445811
2021-08-02,0.10,0.03,124.446117
2021-08-03,0.10,0.03,124.446220
2021-08-04,0.10,0.03,124.446322
2021-08-05,0.10,0.03,124.446424
2021-08-06,0.10,0.03,124.446527
2021-08-09,0.10,0.03,124.446833
2021-08-10,0.10,0.03,124.446936
2021-08-11,0.10,0.03,124.447038
2021-08-12,0.10,0.03,124.447140
2021-08-13,0.10,0.03,124.447243
2021-08-16,0.10,0.03,124.447549
2021-08-17,0.10,0.03,124.447652
2021-08-18,0.10,0.03,124.447754
2021-08-19,0.10,0.03,124.447856
2021-08-20,0.10,0.03,124.447959
2021-08-23,0.10,0.03,124.448265
2021-08-24,0.10,0.03,124.448368
2021-08-25,0.10,0.03,124.448470
2021-08-26,0.10,0.03,124.448572
2021-08-27,0.10,0.03,124.448675
2021-08-30,0.10,0.03,124.448981
2021-08-31,0.10,0.03,124.449084
2021-09-01,0.10,0.03,124.449186
2021-09-02,0.10,0.03,124.449288
2021-09-03,0.10,0.03,124.449391
2021-09-06,0.10,0.03,124.449697
2021-09-07,0.10,0.03,124.449800
2021-09-08,0.10,0.03,124.449902
2021-09-09,0.10,0.03,124.450004
2021-09-10,0.10,0.03,124.450107
2021-09-13,0.10,0.03,124.450413
2021-09-14,0.10,0.03,124.450516
2021-09-15,0.10,0.03,124.450618
2021-09-16,0.10,0.03,124.450720
2021-09-17,0.10,0.03,124.450823
2021-09-20,0.10,0.03,124.451129
2021-09-21,0.10,0.03,124.451232
2021-09-22,0.10,0.03,124.451334
2021-09-23,0.10,0.03,124.451436
2021-09-24,0.10,0.03,124.451539
2021-09-27,0.10,0.03,124.451845
2021-09-28,0.10,0.03,124.451948
2021-09-29,0.10,0.03,124.452050
2021-09-30,0.10,0.03,124.452152
2021-10-01,0.10,0.03,124.452255
2021-10-04,0.10,0.03,124.452562
2021-10-05,0.10,0.03,124.452664
2021-10-06,0.10,0.03,124.452766
2021-10-07,0.10,0.03,124.452868
2021-10-08,0.10,0.03,124.452971
2021-10-11,0.10,0.03,124.453278
2021-10-12,0.10,0.03,124.453380
2021-10-13,0.10,0.03,124.453482
2021-10-14,0.10,0.03,124.453584
2021-10-15,0.10,0.03,124.453687
2021-10-18,0.10,0.03,124.453994
2021-10-19,0.10,0.03,124.454096
2021-10-20,0.10,0.03,124.454198
2021-10-21,0.10,0.03,124.454300
2021-10-22,0.10,0.03,124.454403
2021-10-25,0.10,0.03,124.454710
2021-10-26,0.10,0.03,124.454812
2021-10-27,0.10,0.03,124.454914
2021-10-28,0.10,0.03,124.455017
2021-10-29,0.10,0.03,124.455119
2021-11-01,0.10,0.03,124.455426
2021-11-02,0.10,0.03,124.455528
2021-11-03,0.10,0.03,124.455630
2021-11-04,0.10,0.03,124.455733
2021-11-05,0.10,0.03,124.455835
2021-11-08,0.10,0.03,124.456142
2021-11-09,0.10,0.03,124.456244
2021-11-10,0.10,0.03,124.456346
2021-11-11,0.10,0.03,124.456449
2021-11-12,0.10,0.04,124.456551
2021-11-15,0.10,0.04,124.456960
2021-11-16,0.10,0.04,124.457096
2021-11-17,0.10,0.04,124.457233
2021-11-18,0.10,0.04,124.457369
2021-11-19,0.10,0.04,124.457506
2021-11-22,0.10,0.04,124.457915
2021-11-23,0.10,0.04,124.458051
2021-11-24,0.10,0.04,124.458188
2021-11-25,0.10,0.04,124.458324
2021-11-26,0.10,0.04,124.458460
2021-11-29,0.10,0.04,124.458870
2021-11-30,0.10,0.04,124.459006
2021-12-01,0.10,0.04,124.459142
2021-12-02,0.10,0.04,124.459279
2021-12-03,0.10,0.04,124.459415
2021-12-06,0.10,0.04,124.459824
2021-12-07,0.10,0.04,124.459961
2021-12-08,0.10,0.04,124.460097
2021-12-09,0.10,0.04,124.460233
2021-12-10,0.10,0.04,124.460370
2021-12-13,0.10,0.04,124.460779
2021-12-14,0.10,0.04,124.460915
2021-12-15,0.10,0.04,124.461052
2021-12-16,0.10,0.04,124.461188
2021-12-17,0.10,0.04,124.461325
2021-12-20,0.10,0.04,124.461734
2021-12-21,0.10,0.04,124.461870
2021-12-22,0.10,0.04,124.462006
2021-12-23,0.10,0.04,124.462142
2021-12-24,0.10,0.04,124.462278
2021-12-29,0.10,0.04,124.462960
2021-12-30,0.10,0.04,124.463096
2021-12-31,0.10,0.04,124.463232
2022-01-04,0.10,0.04,124.463778
2022-01-05,0.10,0.04,124.463914
2022-01-06,0.10,0.04,124.464050
2022-01-07,0.10,0.04,124.464186
2022-01-10,0.10,0.04,124.464595
2022-01-11,0.10,0.04,124.464731
2022-01-12,0.10,0.04,124.464867
2022-01-13,0.10,0.05,124.465003
2022-01-14,0.10,0.05,124.465174
2022-01-17,0.10,0.05,124.465686
2022-01-18,0.10,0.05,124.465857
2022-01-19,0.10,0.05,124.466028
2022-01-20,0.10,0.05,124.466199
2022-01-21,0.10,0.05,124.466370
2022-01-24,0.10,0.05,124.466882
2022-01-25,0.10,0.05,124.467053
2022-01-27,0.10,0.05,124.467394
2022-01-28,0.10,0.05,124.467565
2022-01-31,0.10,0.05,124.468077
2022-02-01,0.10,0.05,124.468248
2022-02-02,0.10,0.05,124.468419
2022-02-03,0.10,0.05,124.468590
2022-02-04,0.10,0.05,124.468761
2022-02-07,0.10,0.05,124.469273
2022-02-08,0.10,0.05,124.469444
2022-02-09,0.10,0.05,124.469615
2022-02-10,0.10,0.05,124.469786
2022-02-11,0.10,0.05,124.469957
2022-02-14,0.10,0.05,124.470469
2022-02-15,0.10,0.05,124.470640
2022-02-16,0.10,0.05,124.470811
2022-02-17,0.10,0.05,124.470982
2022-02-18,0.10,0.05,124.471153
2022-02-21,0.10,0.05,124.471665
2022-02-22,0.10,0.05,124.471836
2022-02-23,0.10,0.05,124.472007
2022-02-24,0.10,0.05,124.472178
2022-02-25,0.10,0.05,124.472349
2022-02-28,0.10,0.05,124.472861
2022-03-01,0.10,0.05,124.473032
2022-03-02,0.10,0.05,124.473203
2022-03-03,0.10,0.05,124.473374
2022-03-04,0.10,0.05,124.473545
2022-03-07,0.10,0.05,124.474057
2022-03-08,0.10,0.05,124.474228
2022-03-09,0.10,0.05,124.474399
2022-03-10,0.10,0.05,124.474570
2022-03-11,0.10,0.05,124.474741
2022-03-14,0.10,0.05,124.475253
2022-03-15,0.10,0.05,124.475424
2022-03-16,0.10,0.05,124.475595
2022-03-17,0.10,0.05,124.475766
2022-03-18,0.10,0.05,124.475937
2022-03-21,0.10,0.05,124.476449
2022-03-22,0.10,0.05,124.476620
2022-03-23,0.10,0.05,124.476791
2022-03-24,0.10,0.05,124.476962
2022-03-25,0.10,0.05,124.477133
2022-03-28,0.10,0.05,124.477645
2022-03-29,0.10,0.05,124.477816
2022-03-30,0.10,0.05,124.477987
2022-03-31,0.10,0.09,124.478158
2022-04-01,0.10,0.07,124.478465
2022-04-04,0.10,0.06,124.479181
2022-04-05,0.10,0.06,124.479386
2022-04-06,0.10,0.06,124.479591
2022-04-07,0.10,0.06,124.479796
2022-04-08,0.10,0.06,124.480001
2022-04-11,0.10,0.06,124.480615
2022-04-12,0.10,0.06,124.480820
2022-04-13,0.10,0.06,124.481025
2022-04-14,0.10,0.06,124.481230
2022-04-19,0.10,0.06,124.482253
2022-04-20,0.10,0.06,124.482458
2022-04-21,0.10,0.06,124.482663
2022-04-22,0.10,0.06,124.482868
2022-04-26,0.10,0.06,124.483687
2022-04-27,0.10,0.06,124.483892
2022-04-28,0.10,0.06,124.484097
2022-04-29,0.10,0.06,124.484302
2022-05-02,0.10,0.06,124.484916
2022-05-03,0.10,0.07,124.485121
2022-05-04,0.35,0.31,124.485360
2022-05-05,0.35,0.31,124.486417
2022-05-06,0.35,0.31,124.487474
2022-05-09,0.35,0.31,124.490646
2022-05-10,0.35,0.31,124.491703
2022-05-11,0.35,0.31,124.492760
2022-05-12,0.35,0.31,124.493817
2022-05-13,0.35,0.31,124.494874
2022-05-16,0.35,0.31,124.498046
2022-05-17,0.35,0.31,124.499103
2022-05-18,0.35,0.31,124.500160
2022-05-19,0.35,0.31,124.501217
2022-05-20,0.35,0.31,124.502274
2022-05-23,0.35,0.31,124.505446
2022-05-24,0.35,0.31,124.506503
2022-05-25,0.35,0.31,124.507560
2022-05-26,0.35,0.31,124.508617
2022-05-27,0.35,0.31,124.509674
2022-05-30,0.35,0.31,124.512846
2022-05-31,0.35,0.31,124.513904
2022-06-01,0.35,0.31,124.514962
2022-06-02,0.35,0.31,124.516020
2022-06-03,0.35,0.31,124.517078
2022-06-06,0.35,0.31,124.520251
2022-06-07,0.35,0.31,124.521309
2022-06-08,0.85,0.81,124.522367
2022-06-09,0.85,0.82,124.525130
2022-06-10,0.85,0.81,124.527928
2022-06-14,0.85,0.81,124.538982
2022-06-15,0.85,0.81,124.541746
2022-06-16,0.85,0.81,124.544510
2022-06-17,0.85,0.81,124.547274
2022-06-20,0.85,0.81,124.555566
2022-06-21,0.85,0.81,124.558330
2022-06-22,0.85,0.81,124.561094
2022-06-23,0.85,0.81,124.563858
2022-06-24,0.85,0.81,124.566622
2022-06-27,0.85,0.81,124.574915
2022-06-28,0.85,0.81,124.577680
2022-06-29,0.85,0.81,124.580445
2022-06-30,0.85,0.81,124.583210
2022-07-01,0.85,0.81,124.585975
2022-07-04,0.85,0.81,124.594269
2022-07-05,0.85,0.81,124.597034
2022-07-06,1.35,1.31,124.599799
2022-07-07,1.35,1.31,124.604271
2022-07-08,1.35,1.31,124.608743
2022-07-11,1.35,1.31,124.622160
2022-07-12,1.35,1.31,124.626633
2022-07-13,1.35,1.31,124.631106
2022-07-14,1.35,1.31,124.635579
2022-07-15,1.35,1.31,124.640052
2022-07-18,1.35,1.31,124.653472
2022-07-19,1.35,1.31,124.657946
2022-07-20,1.35,1.31,124.662420
2022-07-21,1.35,1.31,124.666894
2022-07-22,1.35,1.31,124.671368
2022-07-25,1.35,1.31,124.684792
2022-07-26,1.35,1.31,124.689267
2022-07-27,1.35,1.31,124.693742
2022-07-28,1.35,1.31,124.698217
2022-07-29,1.35,1.31,124.702692
2022-08-01,1.35,1.31,124.716119
2022-08-02,1.35,1.31,124.720595
2022-08-03,1.85,1.81,124.725071
2022-08-04,1.85,1.81,124.731256
2022-08-05,1.85,1.81,124.737441
2022-08-08,1.85,1.81,124.755998
2022-08-09,1.85,1.81,124.762185
2022-08-10,1.85,1.81,124.768372
2022-08-11,1.85,1.81,124.774559
2022-08-12,1.85,1.81,124.780746
2022-08-15,1.85,1.81,124.799309
2022-08-16,1.85,1.81,124.805498
2022-08-17,1.85,1.81,124.811687
2022-08-18,1.85,1.81,124.817876
2022-08-19,1.85,1.81,124.824066
2022-08-22,1.85,1.81,124.842636
2022-08-23,1.85,1.81,124.848827
2022-08-24,1.85,1.81,124.855018
2022-08-25,1.85,1.81,124.861209
2022-08-26,1.85,1.81,124.867401
2022-08-29,1.85,1.81,124.885977
2022-08-30,1.85,1.81,124.892170
2022-08-31,1.85,1.81,124.898363
2022-09-01,1.85,1.81,124.904557
2022-09-02,1.85,1.81,124.910751
2022-09-05,1.85,1.81,124.929334
2022-09-06,1.85,1.81,124.935529
2022-09-07,2.35,2.31,124.941724
2022-09-08,2.35,2.31,124.949631
2022-09-09,2.35,2.31,124.957539
2022-09-12,2.35,2.31,124.981264
2022-09-13,2.35,2.31,124.989174
2022-09-14,2.35,2.31,124.997084
2022-09-15,2.35,2.31,125.004995
2022-09-16,2.35,2.31,125.012906
2022-09-19,2.35,2.31,125.036641
2022-09-20,2.35,2.31,125.044554
2022-09-21,2.35,2.31,125.052468
2022-09-23,2.35,2.31,125.068297
2022-09-26,2.35,2.31,125.092043
2022-09-27,2.35,2.31,125.099960
2022-09-28,2.35,2.31,125.107877
2022-09-29,2.35,2.31,125.115795
2022-09-30,2.35,2.31,125.123713
2022-10-03,2.35,2.31,125.147469
2022-10-04,2.35,2.31,125.155389
2022-10-05,2.60,2.56,125.163310
2022-10-06,2.60,2.56,125.172089
2022-10-07,2.60,2.56,125.180868
2022-10-10,2.60,2.56,125.207207
2022-10-11,2.60,2.56,125.215989
2022-10-12,2.60,2.56,125.224771
2022-10-13,2.60,2.56,125.233554
2022-10-14,2.60,2.56,125.242338
2022-10-17,2.60,2.56,125.268690
2022-10-18,2.60,2.56,125.277476
2022-10-19,2.60,2.56,125.286263
2022-10-20,2.60,2.56,125.295050
2022-10-21,2.60,2.56,125.303838
2022-10-24,2.60,2.56,125.330203
2022-10-25,2.60,2.56,125.338993
2022-10-26,2.60,2.56,125.347784
2022-10-27,2.60,2.56,125.356576
2022-10-28,2.60,2.56,125.365368
2022-10-31,2.60,2.56,125.391746
2022-11-01,2.60,2.56,125.400541
2022-11-02,2.85,2.81,125.409336
2022-11-03,2.85,2.81,125.418991
2022-11-04,2.85,2.81,125.428647
2022-11-07,2.85,2.81,125.457616
2022-11-08,2.85,2.81,125.467275
2022-11-09,2.85,2.81,125.476934
2022-11-10,2.85,2.81,125.486594
2022-11-11,2.85,2.81,125.496255
2022-11-14,2.85,2.81,125.525239
2022-11-15,2.85,2.81,125.534903
2022-11-16,2.85,2.81,125.544567
2022-11-17,2.85,2.81,125.554232
2022-11-18,2.85,2.81,125.563898
2022-11-21,2.85,2.81,125.592898
2022-11-22,2.85,2.81,125.602567
2022-11-23,2.85,2.81,125.612237
2022-11-24,2.85,2.81,125.621907
2022-11-25,2.85,2.81,125.631578
2022-11-28,2.85,2.81,125.660594
2022-11-29,2.85,2.81,125.670268
2022-11-30,2.85,2.82,125.679943
2022-12-01,2.85,2.81,125.689653
2022-12-02,2.85,2.82,125.699329
2022-12-05,2.85,2.81,125.728464
2022-12-06,2.85,2.81,125.738143
2022-12-07,3.10,3.06,125.747823
2022-12-08,3.10,3.06,125.758365
2022-12-09,3.10,3.06,125.768908
2022-12-12,3.10,3.06,125.800540
2022-12-13,3.10,3.06,125.811087
2022-12-14,3.10,3.06,125.821634
2022-12-15,3.10,3.06,125.832182
2022-12-16,3.10,3.06,125.842731
2022-12-19,3.10,3.06,125.874381
2022-12-20,3.10,3.06,125.884934
2022-12-21,3.10,3.06,125.895488
2022-12-22,3.10,3.06,125.906043
2022-12-23,3.10,3.06,125.916598
2022-12-28,3.10,3.06,125.969379
2022-12-29,3.10,3.06,125.979940
2022-12-30,3.10,3.07,125.990502
2023-01-03,3.10,3.07,126.032890
2023-01-04,3.10,3.07,126.043491
2023-01-05,3.10,3.07,126.054092
2023-01-06,3.10,3.07,126.064694
2023-01-09,3.10,3.07,126.096504
2023-01-10,3.10,3.07,126.107110
2023-01-11,3.10,3.07,126.117717
2023-01-12,3.10,3.07,126.128325
2023-01-13,3.10,3.07,126.138934
2023-01-16,3.10,3.07,126.170762
2023-01-17,3.10,3.07,126.181374
2023-01-18,3.10,3.07,126.191987
2023-01-19,3.10,3.07,126.202601
2023-01-20,3.10,3.07,126.213216
2023-01-23,3.10,3.07,126.245063
2023-01-24,3.10,3.07,126.255681
2023-01-25,3.10,3.07,126.266300
2023-01-27,3.10,3.07,126.287540
2023-01-30,3.10,3.07,126.319406
2023-01-31,3.10,3.07,126.330031
2023-02-01,3.10,3.07,126.340657
2023-02-02,3.10,3.07,126.351283
2023-02-03,3.10,3.07,126.361910
2023-02-06,3.10,3.07,126.393795
2023-02-07,3.10,3.07,126.404426
2023-02-08,3.35,3.32,126.415058
2023-02-09,3.35,3.32,126.426557
2023-02-10,3.35,3.32,126.438057
2023-02-13,3.35,3.32,126.472559
2023-02-14,3.35,3.32,126.484063
2023-02-15,3.35,3.32,126.495568
2023-02-16,3.35,3.32,126.507074
2023-02-17,3.35,3.32,126.518581
2023-02-20,3.35,3.32,126.553105
2023-02-21,3.35,3.32,126.564616
2023-02-22,3.35,3.33,126.576128
2023-02-23,3.35,3.32,126.587676
2023-02-24,3.35,3.32,126.599190
2023-02-27,3.35,3.32,126.633736
2023-02-28,3.35,3.32,126.645254
2023-03-01,3.35,3.32,126.656774
2023-03-02,3.35,3.32,126.668295
2023-03-03,3.35,3.32,126.679817
2023-03-06,3.35,3.32,126.714385
2023-03-07,3.35,3.32,126.725911
2023-03-08,3.60,3.57,126.737438
2023-03-09,3.60,3.57,126.749834
2023-03-10,3.60,3.57,126.762231
2023-03-13,3.60,3.57,126.799426
2023-03-14,3.60,3.57,126.811828
2023-03-15,3.60,3.57,126.824231
2023-03-16,3.60,3.57,126.836635
2023-03-17,3.60,3.57,126.849041
2023-03-20,3.60,3.57,126.886262
2023-03-21,3.60,3.57,126.898673
2023-03-22,3.60,3.57,126.911085
2023-03-23,3.60,3.57,126.923498
2023-03-24,3.60,3.57,126.935912
2023-03-27,3.60,3.57,126.973158
2023-03-28,3.60,3.57,126.985577
2023-03-29,3.60,3.57,126.997997
2023-03-30,3.60,3.57,127.010418
2023-03-31,3.60,3.57,127.022841
2023-04-03,3.60,3.57,127.060113
2023-04-04,3.60,3.57,127.072541
2023-04-05,3.60,3.57,127.084970
2023-04-06,3.60,3.57,127.097400
2023-04-11,3.60,3.57,127.159556
2023-04-12,3.60,3.57,127.171993
2023-04-13,3.60,3.57,127.184431
2023-04-14,3.60,3.57,127.196871
2023-04-17,3.60,3.57,127.234194
2023-04-18,3.60,3.57,127.246639
2023-04-19,3.60,3.57,127.259085
2023-04-20,3.60,3.57,127.271532
2023-04-21,3.60,3.57,127.283980
2023-04-24,3.60,3.57,127.321328
2023-04-26,3.60,3.57,127.346234
2023-04-27,3.60,3.57,127.358690
2023-04-28,3.60,3.57,127.371147
2023-05-01,3.60,3.57,127.408521
2023-05-02,3.60,3.57,127.420983
2023-05-03,3.85,3.82,127.433446
2023-05-04,3.85,3.82,127.446783
2023-05-05,3.85,3.82,127.460121
2023-05-08,3.85,3.82,127.500140
2023-05-09,3.85,3.82,127.513484
2023-05-10,3.85,3.82,127.526829
2023-05-11,3.85,3.82,127.540176
2023-05-12,3.85,3.82,127.553524
2023-05-15,3.85,3.82,127.593572
2023-05-16,3.85,3.82,127.606926
2023-05-17,3.85,3.82,127.620281
2023-05-18,3.85,3.82,127.633637
2023-05-19,3.85,3.82,127.646995
2023-05-22,3.85,3.82,127.687073
2023-05-23,3.85,3.82,127.700436
2023-05-24,3.85,3.82,127.713801
2023-05-25,3.85,3.82,127.727167
2023-05-26,3.85,3.82,127.740535
2023-05-29,3.85,3.82,127.780642
2023-05-30,3.85,3.82,127.794015
2023-05-31,3.85,3.82,127.807390
2023-06-01,3.85,3.82,127.820766
2023-06-02,3.85,3.82,127.834143
2023-06-05,3.85,3.82,127.874279
2023-06-06,3.85,3.82,127.887662
2023-06-07,4.10,4.07,127.901046
2023-06-08,4.10,4.07,127.915308
2023-06-09,4.10,4.07,127.929571
2023-06-13,4.10,4.07,127.986631
2023-06-14,4.10,4.07,128.000902
2023-06-15,4.10,4.07,128.015175
2023-06-16,4.10,4.07,128.029450
2023-06-19,4.10,4.07,128.072278
2023-06-20,4.10,4.07,128.086559
2023-06-21,4.10,4.07,128.100842
2023-06-22,4.10,4.07,128.115126
2023-06-23,4.10,4.07,128.129412
2023-06-26,4.10,4.07,128.172274
2023-06-27,4.10,4.07,128.186566
2023-06-28,4.10,4.07,128.200860
2023-06-29,4.10,4.07,128.215155
2023-06-30,4.10,4.07,128.229452
2023-07-03,4.10,4.07,128.272347
2023-07-04,4.10,4.07,128.286650
2023-07-05,4.10,4.07,128.300955
2023-07-06,4.10,4.07,128.315261
2023-07-07,4.10,4.07,128.329569
2023-07-10,4.10,4.07,128.372498
2023-07-11,4.10,4.07,128.386812
2023-07-12,4.10,4.07,128.401128
2023-07-13,4.10,4.07,128.415446
2023-07-14,4.10,4.07,128.429765
2023-07-17,4.10,4.07,128.472727
2023-07-18,4.10,4.07,128.487053
2023-07-19,4.10,4.07,128.501380
2023-07-20,4.10,4.07,128.515709
2023-07-21,4.10,4.07,128.530039
2023-07-24,4.10,4.07,128.573035
2023-07-25,4.10,4.07,128.587372
2023-07-26,4.10,4.07,128.601710
2023-07-27,4.10,4.07,128.616050
2023-07-28,4.10,4.07,128.630392
2023-07-31,4.10,4.07,128.673422
2023-08-01,4.10,4.07,128.687770
2023-08-02,4.10,4.07,128.702120
2023-08-03,4.10,4.07,128.716471
2023-08-04,4.10,4.07,128.730824
2023-08-07,4.10,4.07,128.773887
2023-08-08,4.10,4.07,128.788246
2023-08-09,4.10,4.07,128.802607
2023-08-10,4.10,4.07,128.816969
2023-08-11,4.10,4.07,128.831333
2023-08-14,4.10,4.07,128.874430
2023-08-15,4.10,4.07,128.888800
2023-08-16,4.10,4.07,128.903172
2023-08-17,4.10,4.07,128.917546
2023-08-18,4.10,4.07,128.931921
2023-08-21,4.10,4.07,128.975051
2023-08-22,4.10,4.07,128.989433
2023-08-23,4.10,4.07,129.003816
2023-08-24,4.10,4.07,129.018201
2023-08-25,4.10,4.07,129.032587
2023-08-28,4.10,4.07,129.075751
2023-08-29,4.10,4.07,129.090144
2023-08-30,4.10,4.07,129.104538
2023-08-31,4.10,4.07,129.118934
2023-09-01,4.10,4.07,129.133332
2023-09-04,4.10,4.07,129.176530
2023-09-05,4.10,4.07,129.190934
2023-09-06,4.10,4.07,129.205340
2023-09-07,4.10,4.07,129.219747
2023-09-08,4.10,4.07,129.234156
2023-09-11,4.10,4.07,129.277387
2023-09-12,4.10,4.07,129.291802
2023-09-13,4.10,4.07,129.306219
2023-09-14,4.10,4.07,129.320638
2023-09-15,4.10,4.07,129.335058
2023-09-18,4.10,4.07,129.378323
2023-09-19,4.10,4.07,129.392750
2023-09-20,4.10,4.07,129.407178
2023-09-21,4.10,4.07,129.421608
2023-09-22,4.10,4.07,129.436039
2023-09-25,4.10,4.07,129.479338
2023-09-26,4.10,4.07,129.493776
2023-09-27,4.10,4.07,129.508215
2023-09-28,4.10,4.07,129.522656
2023-09-29,4.10,4.07,129.537099
2023-10-02,4.10,4.07,129.580432
2023-10-03,4.10,4.07,129.594881
2023-10-04,4.10,4.07,129.609332
2023-10-05,4.10,4.07,129.623784
2023-10-06,4.10,4.07,129.638238
2023-10-09,4.10,4.07,129.681605
2023-10-10,4.10,4.07,129.696065
2023-10-11,4.10,4.07,129.710527
2023-10-12,4.10,4.07,129.724991
2023-10-13,4.10,4.07,129.739456
2023-10-16,4.10,4.07,129.782857
2023-10-17,4.10,4.07,129.797329
2023-10-18,4.10,4.07,129.811802
2023-10-19,4.10,4.07,129.826277
2023-10-20,4.10,4.07,129.840754
2023-10-23,4.10,4.07,129.884188
2023-10-24,4.10,4.07,129.898671
2023-10-25,4.10,4.07,129.913156
2023-10-26,4.10,4.07,129.927642
2023-10-27,4.10,4.07,129.942130
2023-10-30,4.10,4.07,129.985598
2023-10-31,4.10,4.07,130.000092
2023-11-01,4.10,4.07,130.014588
2023-11-02,4.10,4.07,130.029086
2023-11-03,4.10,4.07,130.043585
2023-11-06,4.10,4.07,130.087087
2023-11-07,4.10,4.07,130.101593
2023-11-08,4.35,4.32,130.116100
2023-11-09,4.35,4.32,130.131500
2023-11-10,4.35,4.32,130.146902
2023-11-13,4.35,4.32,130.193113
2023-11-14,4.35,4.32,130.208522
2023-11-15,4.35,4.32,130.223933
2023-11-16,4.35,4.32,130.239346
2023-11-17,4.35,4.32,130.254761
2023-11-20,4.35,4.32,130.301010
2023-11-21,4.35,4.32,130.316432
2023-11-22,4.35,4.32,130.331856
2023-11-23,4.35,4.32,130.347282
2023-11-24,4.35,4.32,130.362709
2023-11-27,4.35,4.32,130.408997
2023-11-28,4.35,4.32,130.424432
2023-11-29,4.35,4.32,130.439869
2023-11-30,4.35,4.32,130.455307
2023-12-01,4.35,4.32,130.470747
2023-12-04,4.35,4.32,130.517073
2023-12-05,4.35,4.32,130.532521
2023-12-06,4.35,4.32,130.547970
2023-12-07,4.35,4.32,130.563421
2023-12-08,4.35,4.32,130.578874
2023-12-11,4.35,4.32,130.625238
2023-12-12,4.35,4.32,130.640698
2023-12-13,4.35,4.32,130.656160
2023-12-14,4.35,4.32,130.671624
2023-12-15,4.35,4.32,130.687090
2023-12-18,4.35,4.32,130.733493
2023-12-19,4.35,4.32,130.748966
2023-12-20,4.35,4.32,130.764441
2023-12-21,4.35,4.32,130.779918
2023-12-22,4.35,4.32,130.795397
2023-12-27,4.35,4.32,130.872799
2023-12-28,4.35,4.32,130.888289
2023-12-29,4.35,4.32,130.903780
2024-01-02,4.35,4.32,130.965753
2024-01-03,4.35,4.32,130.981254
2024-01-04,4.35,4.32,130.996756
2024-01-05,4.35,4.32,131.012260
2024-01-08,4.35,4.32,131.058778
2024-01-09,4.35,4.32,131.074290
2024-01-10,4.35,4.32,131.089803
2024-01-11,4.35,4.32,131.105318
2024-01-12,4.35,4.32,131.120835
2024-01-15,4.35,4.32,131.167392
2024-01-16,4.35,4.32,131.182916
2024-01-17,4.35,4.32,131.198442
2024-01-18,4.35,4.32,131.213970
2024-01-19,4.35,4.32,131.229500
2024-01-22,4.35,4.32,131.276095
2024-01-23,4.35,4.32,131.291632
2024-01-24,4.35,4.32,131.307171
2024-01-25,4.35,4.32,131.322712
2024-01-29,4.35,4.32,131.384883
2024-01-30,4.35,4.32,131.400433
2024-01-31,4.35,4.32,131.415985
2024-02-01,4.35,4.32,131.431539
2024-02-02,4.35,4.32,131.447095
2024-02-05,4.35,4.32,131.493768
2024-02-06,4.35,4.32,131.509331
2024-02-07,4.35,4.32,131.524896
2024-02-08,4.35,4.32,131.540463
2024-02-09,4.35,4.32,131.556032
2024-02-12,4.35,4.32,131.602743
2024-02-13,4.35,4.32,131.618319
2024-02-14,4.35,4.32,131.633897
2024-02-15,4.35,4.32,131.649477
2024-02-16,4.35,4.32,131.665059
2024-02-19,4.35,4.32,131.711809
2024-02-20,4.35,4.32,131.727398
2024-02-21,4.35,4.32,131.742989
2024-02-22,4.35,4.32,131.758582
2024-02-23,4.35,4.32,131.774176
2024-02-26,4.35,4.32,131.820965
2024-02-27,4.35,4.32,131.836567
2024-02-28,4.35,4.32,131.852171
2024-02-29,4.35,4.32,131.867777
2024-03-01,4.35,4.32,131.883384
2024-03-04,4.35,4.32,131.930212
2024-03-05,4.35,4.32,131.945827
2024-03-06,4.35,4.32,131.961444
2024-03-07,4.35,4.32,131.977062
2024-03-08,4.35,4.32,131.992682
2024-03-11,4.35,4.32,132.039548
2024-03-12,4.35,4.32,132.055176
2024-03-13,4.35,4.32,132.070806
2024-03-14,4.35,4.32,132.086437
2024-03-15,4.35,4.32,132.102070
2024-03-18,4.35,4.32,132.148975
2024-03-19,4.35,4.32,132.164616
2024-03-20,4.35,4.32,132.180258
2024-03-21,4.35,4.32,132.195902
2024-03-22,4.35,4.32,132.211548
2024-03-25,4.35,4.32,132.258492
2024-03-26,4.35,4.32,132.274146
2024-03-27,4.35,4.32,132.289801
2024-03-28,4.35,4.32,132.305458
2024-04-02,4.35,4.32,132.383754
2024-04-03,4.35,4.32,132.399422
2024-04-04,4.35,4.32,132.415092
2024-04-05,4.35,4.32,132.430764
2024-04-08,4.35,4.32,132.477786
2024-04-09,4.35,4.32,132.493466
2024-04-10,4.35,4.32,132.509147
2024-04-11,4.35,4.32,132.524830
2024-04-12,4.35,4.32,132.540515
2024-04-15,4.35,4.32,132.587576
2024-04-16,4.35,4.32,132.603269
2024-04-17,4.35,4.32,132.618963
2024-04-18,4.35,4.32,132.634659
2024-04-19,4.35,4.32,132.650357
2024-04-22,4.35,4.32,132.697457
2024-04-23,4.35,4.32,132.713163
2024-04-24,4.35,4.32,132.728870
2024-04-26,4.35,4.32,132.760289
2024-04-29,4.35,4.32,132.807428
2024-04-30,4.35,4.32,132.823147
2024-05-01,4.35,4.32,132.838867
2024-05-02,4.35,4.32,132.854589
2024-05-03,4.35,4.32,132.870313
2024-05-06,4.35,4.32,132.917491
2024-05-07,4.35,4.32,132.933223
2024-05-08,4.35,4.32,132.948956
2024-05-09,4.35,4.32,132.964691
2024-05-10,4.35,4.32,132.980428
2024-05-13,4.35,4.32,133.027645
2024-05-14,4.35,4.32,133.043390
2024-05-15,4.35,4.32,133.059137
2024-05-16,4.35,4.32,133.074885
2024-05-17,4.35,4.32,133.090635
2024-05-20,4.35,4.32,133.137891
2024-05-21,4.35,4.32,133.153649
2024-05-22,4.35,4.32,133.169409
2024-05-23,4.35,4.32,133.185170
2024-05-24,4.35,4.33,133.200933
2024-05-27,4.35,4.33,133.248338
2024-05-28,4.35,4.32,133.264145
2024-05-29,4.35,4.32,133.279918
2024-05-30,4.35,4.32,133.295692
2024-05-31,4.35,4.32,133.311468
2024-06-03,4.35,4.32,133.358803
2024-06-04,4.35,4.32,133.374587
2024-06-05,4.35,4.32,133.390373
2024-06-06,4.35,4.32,133.406161
2024-06-07,4.35,4.32,133.421950
2024-06-11,4.35,4.32,133.485115
2024-06-12,4.35,4.32,133.500914
2024-06-13,4.35,4.32,133.516715
2024-06-14,4.35,4.32,133.532518
2024-06-17,4.35,4.32,133.579931
2024-06-18,4.35,4.32,133.595741
2024-06-19,4.35,4.33,133.611553
2024-06-20,4.35,4.33,133.627403
2024-06-21,4.35,4.33,133.643255
2024-06-24,4.35,4.33,133.690817
2024-06-25,4.35,4.33,133.706677
2024-06-26,4.35,4.33,133.722539
2024-06-27,4.35,4.33,133.738403
2024-06-28,4.35,4.33,133.754268
2024-07-01,4.35,4.33,133.801870
2024-07-02,4.35,4.33,133.817743
2024-07-03,4.35,4.33,133.833618
2024-07-04,4.35,4.33,133.849495
2024-07-05,4.35,4.34,133.865374
2024-07-08,4.35,4.34,133.913125
2024-07-09,4.35,4.34,133.929048
2024-07-10,4.35,4.34,133.944973
2024-07-11,4.35,4.34,133.960900
2024-07-12,4.35,4.34,133.976829
2024-07-15,4.35,4.34,134.024620
2024-07-16,4.35,4.34,134.040556
2024-07-17,4.35,4.34,134.056494
2024-07-18,4.35,4.34,134.072434
2024-07-19,4.35,4.34,134.088376
2024-07-22,4.35,4.34,134.136207
2024-07-23,4.35,4.34,134.152156
2024-07-24,4.35,4.34,134.168107
2024-07-25,4.35,4.34,134.184060
2024-07-26,4.35,4.34,134.200015
2024-07-29,4.35,4.34,134.247886
2024-07-30,4.35,4.34,134.263849
2024-07-31,4.35,4.34,134.279814
2024-08-01,4.35,4.34,134.295780
2024-08-02,4.35,4.34,134.311748
2024-08-05,4.35,4.34,134.359659
2024-08-06,4.35,4.34,134.375635
2024-08-07,4.35,4.34,134.391613
2024-08-08,4.35,4.34,134.407593
2024-08-09,4.35,4.34,134.423575
2024-08-12,4.35,4.34,134.471526
2024-08-13,4.35,4.34,134.487515
2024-08-14,4.35,4.34,134.503506
2024-08-15,4.35,4.34,134.519499
2024-08-16,4.35,4.34,134.535494
2024-08-19,4.35,4.34,134.583484
2024-08-20,4.35,4.34,134.599487
2024-08-21,4.35,4.34,134.615491
2024-08-22,4.35,4.34,134.631497
2024-08-23,4.35,4.34,134.647505
2024-08-26,4.35,4.34,134.695535
2024-08-27,4.35,4.34,134.711551
2024-08-28,4.35,4.34,134.727569
2024-08-29,4.35,4.34,134.743589
2024-08-30,4.35,4.34,134.759611
2024-09-02,4.35,4.34,134.807681
2024-09-03,4.35,4.34,134.823710
2024-09-04,4.35,4.34,134.839741
2024-09-05,4.35,4.34,134.855774
2024-09-06,4.35,4.34,134.871809
2024-09-09,4.35,4.34,134.919919
2024-09-10,4.35,4.34,134.935962
2024-09-11,4.35,4.34,134.952006
2024-09-12,4.35,4.34,134.968052
2024-09-13,4.35,4.34,134.984100
2024-09-16,4.35,4.34,135.032250
2024-09-17,4.35,4.34,135.048306
2024-09-18,4.35,4.34,135.064364
2024-09-19,4.35,4.34,135.080424
2024-09-20,4.35,4.34,135.096486
2024-09-23,4.35,4.34,135.144677
2024-09-24,4.35,4.34,135.160746
2024-09-25,4.35,4.34,135.176817
2024-09-26,4.35,4.34,135.192890
2024-09-27,4.35,4.34,135.208965
2024-09-30,4.35,4.34,135.257196
2024-10-01,4.35,4.34,135.273279
2024-10-02,4.35,4.34,135.289364
2024-10-03,4.35,4.34,135.305450
2024-10-04,4.35,4.34,135.321538
2024-10-07,4.35,4.34,135.369809
2024-10-08,4.35,4.34,135.385905
2024-10-09,4.35,4.34,135.402003
2024-10-10,4.35,4.34,135.418103
2024-10-11,4.35,4.34,135.434205
2024-10-14,4.35,4.34,135.482516
2024-10-15,4.35,4.34,135.498625
2024-10-16,4.35,4.34,135.514736
2024-10-17,4.35,4.34,135.530849
2024-10-18,4.35,4.34,135.546964
2024-10-21,4.35,4.34,135.595315
2024-10-22,4.35,4.34,135.611438
2024-10-23,4.35,4.34,135.627563
2024-10-24,4.35,4.34,135.643690
2024-10-25,4.35,4.34,135.659819
2024-10-28,4.35,4.34,135.708211
2024-10-29,4.35,4.34,135.724347
2024-10-30,4.35,4.34,135.740485
2024-10-31,4.35,4.34,135.756625
2024-11-01,4.35,4.34,135.772767
2024-11-04,4.35,4.34,135.821199
2024-11-05,4.35,4.34,135.837349
2024-11-06,4.35,4.34,135.853501
2024-11-07,4.35,4.34,135.869655
2024-11-08,4.35,4.34,135.885810
2024-11-11,4.35,4.34,135.934282
2024-11-12,4.35,4.34,135.950445
2024-11-13,4.35,4.34,135.966610
2024-11-14,4.35,4.34,135.982777
2024-11-15,4.35,4.34,135.998946
2024-11-18,4.35,4.34,136.047459
2024-11-19,4.35,4.34,136.063636
2024-11-20,4.35,4.34,136.079815
2024-11-21,4.35,4.34,136.095995
2024-11-22,4.35,4.34,136.112177
2024-11-25,4.35,4.34,136.160730
2024-11-26,4.35,4.34,136.176920
2024-11-27,4.35,4.34,136.193112
2024-11-28,4.35,4.34,136.209306
2024-11-29,4.35,4.34,136.225502
2024-12-02,4.35,4.34,136.274095
2024-12-03,4.35,4.34,136.290299
2024-12-04,4.35,4.34,136.306504
2024-12-05,4.35,4.34,136.322711
2024-12-06,4.35,4.34,136.338920
2024-12-09,4.35,4.34,136.387554
2024-12-10,4.35,4.34,136.403771
2024-12-11,4.35,4.34,136.419990
2024-12-12,4.35,4.34,136.436211
2024-12-13,4.35,4.34,136.452434
2024-12-16,4.35,4.34,136.501108
2024-12-17,4.35,4.34,136.517339
2024-12-18,4.35,4.34,136.533571
2024-12-19,4.35,4.34,136.549805
2024-12-20,4.35,4.34,136.566041
2024-12-23,4.35,4.34,136.614756
2024-12-24,4.35,4.34,136.631000
2024-12-27,4.35,4.34,136.679738
2024-12-30,4.35,4.34,136.728493
2024-12-31,4.35,4.34,136.744751
2025-01-02,4.35,4.34,136.777270
2025-01-03,4.35,4.34,136.793533
2025-01-06,4.35,4.34,136.842329
2025-01-07,4.35,4.34,136.858600
2025-01-08,4.35,4.34,136.874873
2025-01-09,4.35,4.34,136.891148
2025-01-10,4.35,4.34,136.907425
2025-01-13,4.35,4.34,136.956262
2025-01-14,4.35,4.34,136.972547
2025-01-15,4.35,4.34,136.988834
2025-01-16,4.35,4.34,137.005123
2025-01-17,4.35,4.34,137.021413
2025-01-20,4.35,4.34,137.070290
2025-01-21,4.35,4.34,137.086588
2025-01-22,4.35,4.34,137.102888
2025-01-23,4.35,4.34,137.119190
2025-01-24,4.35,4.34,137.135494
2025-01-28,4.35,4.34,137.200718
2025-01-29,4.35,4.34,137.217032
2025-01-30,4.35,4.34,137.233348
2025-01-31,4.35,4.34,137.249666
2025-02-03,4.35,4.34,137.298625
2025-02-04,4.35,4.34,137.314950
2025-02-05,4.35,4.34,137.331277
2025-02-06,4.35,4.34,137.347606
2025-02-07,4.35,4.34,137.363937
2025-02-10,4.35,4.34,137.412936
2025-02-11,4.35,4.34,137.429275
2025-02-12,4.35,4.34,137.445616
2025-02-13,4.35,4.34,137.461959
2025-02-14,4.35,4.34,137.478304
2025-02-17,4.35,4.34,137.527344
2025-02-18,4.35,4.34,137.543697
2025-02-19,4.10,4.09,137.560052
2025-02-20,4.10,4.09,137.575466
2025-02-21,4.10,4.09,137.590882
2025-02-24,4.10,4.09,137.637135
2025-02-25,4.10,4.09,137.652558
2025-02-26,4.10,4.09,137.667983
2025-02-27,4.10,4.09,137.683409
2025-02-28,4.10,4.09,137.698837
2025-03-03,4.10,4.09,137.745126
2025-03-04,4.10,4.09,137.760561
2025-03-05,4.10,4.09,137.775998
2025-03-06,4.10,4.09,137.791436
2025-03-07,4.10,4.09,137.806876
2025-03-10,4.10,4.09,137.853202
2025-03-11,4.10,4.09,137.868649
2025-03-12,4.10,4.09,137.884098
2025-03-13,4.10,4.09,137.899549
2025-03-14,4.10,4.09,137.915001
2025-03-17,4.10,4.09,137.961363
2025-03-18,4.10,4.09,137.976822
2025-03-19,4.10,4.09,137.992283
2025-03-20,4.10,4.09,138.007746
2025-03-21,4.10,4.09,138.023210
2025-03-24,4.10,4.09,138.069608
2025-03-25,4.10,4.09,138.085079
2025-03-26,4.10,4.09,138.100552
2025-03-27,4.10,4.09,138.116027
2025-03-28,4.10,4.09,138.131504
2025-03-31,4.10,4.09,138.177939
2025-04-01,4.10,4.09,138.193423
2025-04-02,4.10,4.09,138.208908
2025-04-03,4.10,4.09,138.224395
2025-04-04,4.10,4.09,138.239884
2025-04-07,4.10,4.09,138.286355
2025-04-08,4.10,4.09,138.301851
2025-04-09,4.10,4.09,138.317348
2025-04-10,4.10,4.09,138.332847
2025-04-11,4.10,4.09,138.348348
2025-04-14,4.10,4.09,138.394856
2025-04-15,4.10,4.09,138.410364
2025-04-16,4.10,4.09,138.425874
2025-04-17,4.10,4.09,138.441385
2025-04-22,4.10,4.09,138.518950
2025-04-23,4.10,4.09,138.534472
2025-04-24,4.10,4.09,138.549995
2025-04-28,4.10,4.09,138.612096
2025-04-29,4.10,4.09,138.627628
2025-04-30,4.10,4.09,138.643162
2025-05-01,4.10,4.09,138.658698
2025-05-02,4.10,4.09,138.674235
2025-05-05,4.10,4.09,138.720852
2025-05-06,4.10,4.09,138.736396
2025-05-07,4.10,4.09,138.751942
2025-05-08,4.10,4.09,138.767490
2025-05-09,4.10,4.09,138.783040
2025-05-12,4.10,4.09,138.829694
2025-05-13,4.10,4.09,138.845251
2025-05-14,4.10,4.09,138.860809
2025-05-15,4.10,4.09,138.876369
2025-05-16,4.10,4.09,138.891931
2025-05-19,4.10,4.09,138.938622
2025-05-20,4.10,4.09,138.954191
2025-05-21,3.85,3.84,138.969761
2025-05-22,3.85,3.84,138.984381
2025-05-23,3.85,3.84,138.999003
2025-05-26,3.85,3.84,139.042873
2025-05-27,3.85,3.84,139.057501
2025-05-28,3.85,3.84,139.072131
2025-05-29,3.85,3.84,139.086762
2025-05-30,3.85,3.84,139.101395
2025-06-02,3.85,3.84,139.145298
2025-06-03,3.85,3.84,139.159937
2025-06-04,3.85,3.84,139.174577
2025-06-05,3.85,3.84,139.189219
2025-06-06,3.85,3.84,139.203862
2025-06-10,3.85,3.84,139.262442
2025-06-11,3.85,3.84,139.277093
2025-06-12,3.85,3.84,139.291746
2025-06-13,3.85,3.84,139.306400
2025-06-16,3.85,3.84,139.350367
2025-06-17,3.85,3.84,139.365027
2025-06-18,3.85,3.84,139.379689
2025-06-19,3.85,3.84,139.394353
2025-06-20,3.85,3.84,139.409018
2025-06-23,3.85,3.84,139.453018
2025-06-24,3.85,3.84,139.467689
2025-06-25,3.85,3.84,139.482362
2025-06-26,3.85,3.84,139.497036
2025-06-27,3.85,3.84,139.511712
2025-06-30,3.85,3.84,139.555744
2025-07-01,3.85,3.84,139.570426
2025-07-02,3.85,3.84,139.585110
2025-07-03,3.85,3.84,139.599795
2025-07-04,3.85,3.84,139.614482
2025-07-07,3.85,3.84,139.658547
2025-07-08,3.85,3.84,139.673240
2025-07-09,3.85,3.84,139.687934
2025-07-10,3.85,3.84,139.702630
2025-07-11,3.85,3.84,139.717327
2025-07-14,3.85,3.84,139.761424
2025-07-15,3.85,3.84,139.776128
2025-07-16,3.85,3.84,139.790833
2025-07-17,3.85,3.84,139.805540
2025-07-18,3.85,3.84,139.820248
2025-07-21,3.85,3.84,139.864378
2025-07-22,3.85,3.84,139.879092
2025-07-23,3.85,3.84,139.893808
2025-07-24,3.85,3.84,139.908526
2025-07-25,3.85,3.84,139.923245
2025-07-28,3.85,3.84,139.967407
2025-07-29,3.85,3.84,139.982132
2025-07-30,3.85,3.84,139.996859
2025-07-31,3.85,3.84,140.011587
2025-08-01,3.85,3.84,140.026317
2025-08-04,3.85,3.84,140.070512
2025-08-05,3.85,3.84,140.085248
2025-08-06,3.85,3.84,140.099986
2025-08-07,3.85,3.84,140.114725
2025-08-08,3.85,3.84,140.129466
2025-08-11,3.85,3.84,140.173693
2025-08-12,3.85,3.84,140.188440
2025-08-13,3.60,3.59,140.203189
2025-08-14,3.60,3.59,140.216979
2025-08-15,3.60,3.59,140.230770
2025-08-18,3.60,3.59,140.272148
2025-08-19,3.60,3.59,140.285945
2025-08-20,3.60,3.59,140.299743
2025-08-21,3.60,3.59,140.313542
2025-08-22,3.60,3.59,140.327343
2025-08-25,3.60,3.59,140.368749
2025-08-26,3.60,3.59,140.382555
2025-08-27,3.60,3.59,140.396362
2025-08-28,3.60,3.59,140.410171
2025-08-29,3.60,3.59,140.423981
2025-09-01,3.60,3.59,140.465416
2025-09-02,3.60,3.59,140.479232
2025-09-03,3.60,3.59,140.493049
2025-09-04,3.60,3.59,140.506867
2025-09-05,3.60,3.59,140.520687
2025-09-08,3.60,3.60,140.562150
2025-09-09,3.60,3.60,140.576014
2025-09-10,3.60,3.60,140.589879
2025-09-11,3.60,3.60,140.603745
2025-09-12,3.60,3.60,140.617613
2025-09-15,3.60,3.60,140.659220
2025-09-16,3.60,3.60,140.673093
2025-09-17,3.60,3.60,140.686968
2025-09-18,3.60,3.60,140.700844
2025-09-19,3.60,3.60,140.714721
2025-09-22,3.60,3.60,140.756357
2025-09-23,3.60,3.60,140.770240
2025-09-24,3.60,3.60,140.784124
2025-09-25,3.60,3.60,140.798010
2025-09-26,3.60,3.60,140.811897
2025-09-29,3.60,3.60,140.853562
2025-09-30,3.60,3.59,140.867454
2025-10-01,3.60,3.60,140.881309
2025-10-02,3.60,3.60,140.895204
2025-10-03,3.60,3.60,140.909101
2025-10-06,3.60,3.60,140.950795
2025-10-07,3.60,3.60,140.964697
2025-10-08,3.60,3.60,140.978600
2025-10-09,3.60,3.60,140.992505
2025-10-10,3.60,3.60,141.006411
2025-10-13,3.60,3.60,141.048133
2025-10-14,3.60,3.60,141.062045
2025-10-15,3.60,3.60,141.075958
2025-10-16,3.60,3.60,141.089872
2025-10-17,3.60,3.60,141.103788
2025-10-20,3.60,3.60,141.145539
2025-10-21,3.60,3.60,141.159460
2025-10-22,3.60,3.60,141.173383
2025-10-23,3.60,3.60,141.187307
2025-10-24,3.60,3.60,141.201232
2025-10-27,3.60,3.60,141.243012
2025-10-28,3.60,3.60,141.256943
2025-10-29,3.60,3.60,141.270875
2025-10-30,3.60,3.60,141.284809
2025-10-31,3.60,3.60,141.298744
2025-11-03,3.60,3.60,141.340553
2025-11-04,3.60,3.60,141.354493
2025-11-05,3.60,3.60,141.368435
2025-11-06,3.60,3.60,141.382378
2025-11-07,3.60,3.60,141.396323
2025-11-10,3.60,3.60,141.438161
2025-11-11,3.60,3.60,141.452111
2025-11-12,3.60,3.60,141.466062
2025-11-13,3.60,3.60,141.480015
2025-11-14,3.60,3.60,141.493969
2025-11-17,3.60,3.60,141.535836
2025-11-18,3.60,3.60,141.549796
2025-11-19,3.60,3.60,141.563757
2025-11-20,3.60,3.60,141.577719
2025-11-21,3.60,3.60,141.591683
2025-11-24,3.60,3.60,141.633579
2025-11-25,3.60,3.60,141.647548
2025-11-26,3.60,3.60,141.661519
2025-11-27,3.60,3.60,141.675491
2025-11-28,3.60,3.60,141.689464
2025-12-01,3.60,3.60,141.731389
2025-12-02,3.60,3.60,141.745368
2025-12-03,3.60,3.60,141.759348
2025-12-04,3.60,3.60,141.773330
2025-12-05,3.60,3.60,141.787313
2025-12-08,3.60,3.60,141.829267
2025-12-09,3.60,3.60,141.843256
2025-12-10,3.60,3.60,141.857246
2025-12-11,3.60,3.60,141.871237
2025-12-12,3.60,3.60,141.885230
2025-12-15,3.60,3.60,141.927212
2025-12-16,3.60,3.60,141.941210
2025-12-17,3.60,3.60,141.955210
2025-12-18,3.60,3.60,141.969211
2025-12-19,3.60,3.60,141.983213
2025-12-22,3.60,3.60,142.025224
2025-12-23,3.60,3.60,142.039232
2025-12-24,3.60,3.60,142.053241
2025-12-29,3.60,3.60,142.123295
2025-12-30,3.60,3.60,142.137313
2025-12-31,3.60,3.60,142.151332
2026-01-02,3.60,3.60,142.179373
2026-01-05,3.60,3.60,142.221443
2026-01-06,3.60,3.60,142.235470
2026-01-07,3.60,3.60,142.249499
2026-01-08,3.60,3.60,142.263529
2026-01-09,3.60,3.60,142.277560
2026-01-12,3.60,3.60,142.319659
2026-01-13,3.60,3.60,142.333696
2026-01-14,3.60,3.60,142.347734
2026-01-15,3.60,3.60,142.361774
2026-01-16,3.60,3.60,142.375815
2026-01-19,3.60,3.60,142.417943
2026-01-20,3.60,3.60,142.431990
2026-01-21,3.60,3.60,142.446038
2026-01-22,3.60,3.60,142.460087
2026-01-23,3.60,3.60,142.474138
2026-01-27,3.60,3.60,142.530347
2026-01-28,3.60,3.60,142.544405
2026-01-29,3.60,3.60,142.558464
2026-01-30,3.60,3.60,142.572525
2026-02-02,3.60,3.60,142.614711
2026-02-03,3.60,3.60,142.628777
2026-02-04,3.85,3.85,142.642844
2026-02-05,3.85,3.85,142.657890
2026-02-06,3.85,3.85,142.672937
2026-02-09,3.85,3.85,142.718084
2026-02-10,3.85,3.85,142.733138
2026-02-11,3.85,3.85,142.748193
2026-02-12,3.85,3.85,142.763250
2026-02-13,3.85,3.85,142.778309
2026-02-16,3.85,3.85,142.823490
2026-02-17,3.85,3.85,142.838555
2026-02-18,3.85,3.85,142.853622
2026-02-19,3.85,3.85,142.868690
2026-02-20,3.85,3.85,142.883760
2026-02-23,3.85,3.85,142.928974
2026-02-24,3.85,3.85,142.944050
2026-02-25,3.85,3.85,142.959128
2026-02-26,3.85,3.85,142.974207
2026-02-27,3.85,3.85,142.989288
2026-03-02,3.85,3.85,143.034535
2026-03-03,3.85,3.85,143.049622
2026-03-04,3.85,3.85,143.064711
2026-03-05,3.85,3.85,143.079801
2026-03-06,3.85,3.85,143.094893
2026-03-09,3.85,3.85,143.140174
2026-03-10,3.85,3.85,143.155272
2026-03-11,3.85,3.85,143.170372
2026-03-12,3.85,3.85,143.185474
2026-03-13,3.85,3.85,143.200577
2026-03-16,3.85,3.85,143.245891
2026-03-17,3.85,3.85,143.261000
2026-03-18,4.10,4.10,143.276111
2026-03-19,4.10,4.10,143.292205
2026-03-20,4.10,4.10,143.308301
2026-03-23,4.10,4.10,143.356594
2026-03-24,4.10,4.10,143.372697
2026-03-25,4.10,4.10,143.388802
2026-03-26,4.10,4.10,143.404909
2026-03-27,4.10,4.10,143.421017
2026-03-30,4.10,4.10,143.469348
2026-03-31,4.10,4.10,143.485464
2026-04-01,4.10,4.10,143.501582
2026-04-02,4.10,4.10,143.517701
2026-04-07,4.10,4.10,143.598307
2026-04-08,4.10,4.10,143.614437
2026-04-09,4.10,4.10,143.630569
2026-04-10,4.10,4.10,143.646703
2026-04-13,4.10,4.10,143.695110
2026-04-14,4.10,4.10,143.711251
2026-04-15,4.10,4.10,143.727394
2026-04-16,4.10,4.10,143.743539
2026-04-17,4.10,4.10,143.759686
2026-04-20,4.10,4.10,143.808131
2026-04-21,4.10,4.10,143.824285
2026-04-22,4.10,4.10,143.840441
2026-04-23,4.10,4.10,143.856598
2026-04-24,4.10,4.10,143.872757
2026-04-27,4.10,4.10,143.921240
2026-04-28,4.10,4.10,143.937406
2026-04-29,4.10,4.10,143.953574
2026-04-30,4.10,4.10,143.969744
2026-05-01,4.10,4.10,143.985916
2026-05-04,4.10,4.10,144.034437
2026-05-05,4.10,4.10,144.050616
2026-05-06,4.35,4.35,144.066797
2026-05-07,4.35,4.35,144.083967
2026-05-08,4.35,4.35,144.101139
2026-05-11,4.35,4.35,144.152660
2026-05-12,4.35,4.35,144.169840
2026-05-13,4.35,4.35,144.187022
2026-05-14,4.35,4.35,144.204206
2026-05-15,4.35,4.35,144.221392
2026-05-18,4.35,4.35,144.272956
2026-05-19,4.35,4.35,144.290150
2026-05-20,4.35,4.35,144.307346
2026-05-21,4.35,4.35,144.324544
2026-05-22,4.35,4.35,144.341744
2026-05-25,4.35,4.35,144.393351
2026-05-26,4.35,4.35,144.410560
2026-05-27,4.35,4.35,144.427771
2026-05-28,4.35,4.35,144.444984
2026-05-29,4.35,4.35,144.462199
2026-06-01,4.35,4.35,144.513849
2026-06-02,4.35,4.35,144.531072
2026-06-03,4.35,4.35,144.548297
2026-06-04,4.35,4.35,144.565524
2026-06-05,4.35,4.35,144.582753
2026-06-09,4.35,4.35,144.651677
2026-06-10,4.35,4.35,144.668916
2026-06-11,4.35,4.35,144.686157
2026-06-12,4.35,4.35,144.703400
2026-06-15,4.35,4.35,144.755136
2026-06-16,4.35,4.35,144.772388
2026-06-17,4.35,4.35,144.789642
2026-06-18,4.35,4.35,144.806898
2026-06-19,4.35,4.35,144.824156
2026-06-22,4.35,4.35,144.875936
2026-06-23,4.35,4.35,144.893202
2026-06-24,4.35,4.35,144.910470
2026-06-25,4.35,4.35,144.927740
2026-06-26,4.35,4.35,144.945012
2026-06-29,4.35,4.35,144.996835
2026-06-30,4.35,4.35,145.014115
2026-07-01,4.35,4.35,145.031398
2026-07-02,4.35,4.35,145.048683
2026-07-03,4.35,4.35,145.065970
2026-07-06,4.35,4.35,145.117836
2026-07-07,4.35,4.35,145.135131
2026-07-08,4.35,4.35,145.152428
2026-07-09,4.35,4.35,145.169727
2026-07-10,4.35,4.35,145.187028
2026-07-13,4.35,4.35,145.238937
2026-07-14,4.35,4.35,145.256246
2026-07-15,4.35,4.35,145.273557
2026-07-16,4.35,4.35,145.290870
2026-07-17,4.35,4.35,145.308185
2026-07-20,4.35,4.35,145.360138
2026-07-21,4.35,4.35,145.377462
2026-07-22,4.35,4.35,145.394788
2026-07-23,4.35,4.35,145.412116
2026-07-24,4.35,4.35,145.429446
2026-07-27,4.35,4.35,145.481442
2026-07-28,4.35,4.35,145.498780
2026-07-29,4.35,4.35,145.516120
2026-07-30,4.35,4.35,145.533462
2026-07-31,4.35,4.35,145.550806
2026-08-03,4.35,4.35,145.602845
2026-08-04,4.35,4.35,145.620198
2026-08-05,4.35,4.35,145.637553
2026-08-06,4.35,4.35,145.654910
2026-08-07,4.35,4.35,145.672269
2026-08-10,4.35,4.35,145.724352
2026-08-11,4.35,4.35,145.741719
2026-08-12,4.35,4.35,145.759088
2026-08-13,4.35,4.35,145.776459
2026-08-14,4.35,4.35,145.793832
2026-08-17,4.35,4.35,145.845958
2026-08-18,4.35,4.35,145.863340
2026-08-19,4.35,4.35,145.880724
2026-08-20,4.35,4.35,145.898110
2026-08-21,4.35,4.35,145.915498
2026-08-24,4.35,4.35,145.967668
2026-08-25,4.35,4.35,145.985064
2026-08-26,4.35,4.35,146.002462
2026-08-27,4.35,4.35,146.019862
2026-08-28,4.35,4.35,146.037264
2026-08-31,4.35,4.35,146.089477
2026-09-01,4.35,4.35,146.106888
2026-09-02,4.35,4.35,146.124301
2026-09-03,4.35,4.35,146.141716
2026-09-04,4.35,4.35,146.159133
2026-09-07,4.35,4.35,146.211390
2026-09-08,4.35,4.35,146.228815
2026-09-09,4.35,4.35,146.246242
2026-09-10,4.35,4.35,146.263671
2026-09-11,4.35,4.35,146.281102
2026-09-14,4.35,4.35,146.333403
2026-09-15,4.35,4.35,146.350843
2026-09-16,4.35,4.35,146.368285
2026-09-17,4.35,4.35,146.385729
2026-09-18,4.35,4.35,146.403175
2026-09-21,4.35,4.35,146.455519
2026-09-22,,,146.472973
""")
print('wrote data/rba_cash_rate.csv')

In [ ]:
# Parameters (papermill overrides these)
valuation_date = "2026-09-22"
rba_csv = "data/rba_cash_rate.csv"
output_json = "build/outputs.json"

## 1. Setup

In [ ]:
import json
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import QuantLib as ql

iso = lambda d: d.ISO()
out = {"meta": {
    "episode": 1,
    "valuation_date": valuation_date,
    "quantlib_version": ql.__version__,
    "quotes_label": "RBA data (Source: RBA 2026)",
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}}
print("QuantLib", ql.__version__)

## 2. Three rates

| Rate | What it is | Who | Source |
|---|---|---|---|
| Cash rate target | The policy rate. The Monetary Policy Board decides it; changes are announced at 2:30pm Sydney time after a Board meeting and take effect the next day. | RBA | RBA *Cash Rate Target* and *Cash Rate Target Overview* |
| Cash rate (AONIA) | The interest rate on unsecured overnight loans between banks: the market outcome the RBA steers towards the target. Also known as AONIA. | calculated and published by the RBA | RBA *Cash Rate Methodology*; RBA speech, Kent (2020) |
| BBSW | Bank Bill Swap Rate: the rate on prime bank bills and NCDs for 1 to 6 months, set from trades between 8:30 and 10:00am and published by 10:30am. It includes a bank credit premium. | administered by ASX | ASX *BBSW Conventions and Methodology* (Dec 2025) |

## 3. The cash rate since 2011

In [ ]:
rba = pd.read_csv(rba_csv, parse_dates=["date"]).dropna(subset=["cash_rate"]).reset_index(drop=True)
rba["gap_bp"] = ((rba.cash_rate - rba.cash_rate_target) * 100).round(2)

changes = rba[rba.cash_rate_target.diff().fillna(0) != 0]
last = rba.iloc[-1]
hi = rba.loc[rba.cash_rate_target.idxmax()]
lo = rba.loc[rba.cash_rate_target.idxmin()]

# The 2022-23 tightening cycle: every change between the low and the next peak.
cycle = changes[(changes.date > lo.date) & (changes.date <= "2023-12-31")]
history = {
    "start": rba.date.iloc[0].date().isoformat(), "end": last.date.date().isoformat(),
    "first_target_pct": float(rba.cash_rate_target.iloc[0]),
    "latest_target_pct": float(last.cash_rate_target), "latest_aonia_pct": float(last.cash_rate),
    "max_target_pct": float(hi.cash_rate_target), "max_target_date": hi.date.date().isoformat(),
    "min_target_pct": float(lo.cash_rate_target), "min_target_date": lo.date.date().isoformat(),
    "n_changes": int(len(changes)),
    "cycle_first": cycle.date.iloc[0].date().isoformat(), "cycle_last": cycle.date.iloc[-1].date().isoformat(),
    "cycle_n_hikes": int(len(cycle)), "cycle_total_bp": float((cycle.cash_rate_target.iloc[-1] - lo.cash_rate_target) * 100),
    "cycle_peak_pct": float(cycle.cash_rate_target.iloc[-1]),
}
since = changes[changes.date > cycle.date.iloc[-1]]
history["since_cuts"] = int((since.cash_rate_target.diff().fillna(since.cash_rate_target.iloc[0] - cycle.cash_rate_target.iloc[-1]) < 0).sum())
history["since_hikes"] = int(len(since) - history["since_cuts"])
history["since_first"] = since.date.iloc[0].date().isoformat()
out["history"] = history
pd.Series(history)

For the chart we keep one point per week (the last business day of each week), which is plenty for a fifteen-year picture. The x axis is in decimal years.

In [ ]:
weekly = rba.set_index("date").resample("W-FRI").last().dropna(subset=["cash_rate"]).reset_index()
decimal_year = lambda d: d.year + (d.dayofyear - 1) / (366 if d.is_leap_year else 365)
weekly["t"] = weekly.date.map(decimal_year)
out["chart"] = {
    "t": weekly.t.round(4).tolist(),
    "target_pct": weekly.cash_rate_target.tolist(),
    "aonia_pct": weekly.cash_rate.tolist(),
    "gap_bp": weekly.gap_bp.tolist(),
    "ticks": [{"x": float(y), "label": str(y)} for y in range(2012, 2027, 2)],
    "range": [decimal_year(rba.date.iloc[0]), decimal_year(last.date)],
}

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].step(weekly.date, weekly.cash_rate_target, where="post", label="cash rate target")
ax[0].plot(weekly.date, weekly.cash_rate, "--", label="cash rate (AONIA)")
ax[0].set(title="Cash rate, % (Source: RBA 2026)", ylabel="%")
ax[1].step(weekly.date, weekly.gap_bp, where="post", color="C2")
ax[1].set(title="AONIA minus target, basis points", ylabel="bp")
for a in ax:
    a.grid(alpha=.3)
ax[0].legend()
plt.tight_layout()

## 4. AONIA vs the target

Before March 2020 the RBA supplied just enough reserves for the cash rate to sit on the target, and it almost always did. From 2020 the banking system held far more reserves than it needed, and the cash rate traded a little below the target (RBA speech, Kent 2024). In 2026 it is back on the target.

In [ ]:
def gap_stats(frame):
    return {"mean_bp": float(frame.gap_bp.mean()), "min_bp": float(frame.gap_bp.min()), "max_bp": float(frame.gap_bp.max()),
            "share_on_target": float((frame.gap_bp == 0).mean())}

pre = rba[rba.date < "2020-03-01"]
post = rba[(rba.date >= "2020-03-01") & (rba.date < "2026-01-01")]
now = rba[rba.date >= "2026-01-01"]
off = rba[rba.gap_bp != 0]
out["gap"] = {"pre_2020": gap_stats(pre), "2020_2025": gap_stats(post), "2026": gap_stats(now),
              "widest_bp": float(rba.gap_bp.min()), "widest_date": rba.loc[rba.gap_bp.idxmin(), "date"].date().isoformat(),
              "first_off_target": off.date.iloc[0].date().isoformat(),
              "back_since": rba[rba.date > off.date.iloc[-1]].date.iloc[0].date().isoformat()}
pd.DataFrame({k: v for k, v in out["gap"].items() if isinstance(v, dict)})

## 5. Dates: T+1 and Modified Following

AFMA's conventions (§3.3) work through three date examples. We reproduce them with QuantLib's Australian calendar.

* **AUD swaps**: start on the next Sydney business day (T+1); dates roll *Modified Following*; **no** end-of-month rule.
* **Cross-currency swaps** (for comparison): spot is T+2 in Sydney and New York, and the end-of-month rule applies.
* **Forward-starting swaps**: start and roll dates are counted from the trade's spot date.

Modified Following: a date on a weekend or holiday moves forward to the next business day, unless that crosses into the next month, in which case it moves back instead.

In [ ]:
cal = ql.Australia(ql.Australia.Settlement)
# QuantLib 1.43 misses NSW's additional Anzac Day holidays when 25 April falls on a weekend.
for d in [ql.Date(27, 4, 2026), ql.Date(26, 4, 2027)]:
    cal.addHoliday(d)
sydney_ny = ql.JointCalendar(cal, ql.UnitedStates(ql.UnitedStates.FederalReserve))

def forward_dates(calendar, spot, months, eom):
    return [calendar.advance(spot, ql.Period(m, ql.Months), ql.ModifiedFollowing, eom) for m in months]

# AFMA example 1: AUD swap traded Monday 29 April 2024
trade1 = ql.Date(29, 4, 2024)
spot1 = cal.advance(trade1, 1, ql.Days)
aud = forward_dates(cal, spot1, [1, 2, 3], eom=False)
# AFMA example 2: cross-currency swap traded Friday 26 April 2024, spot T+2, end-of-month rule
trade2 = ql.Date(26, 4, 2024)
spot2 = sydney_ny.advance(trade2, 2, ql.Days)
xccy = forward_dates(sydney_ny, spot2, [1, 2, 3], eom=True)
# AFMA example 3: one-year swap starting in three months (3m/1y), traded 20 June 2024
trade3 = ql.Date(20, 6, 2024)
spot3 = cal.advance(trade3, 1, ql.Days)
start3 = cal.adjust(spot3 + ql.Period(3, ql.Months), ql.ModifiedFollowing)
end3 = cal.adjust(spot3 + ql.Period(15, ql.Months), ql.ModifiedFollowing)

AFMA_PRINTED = {  # dates as printed in AFMA §3.3
    "aud_spot": "2024-04-30", "aud": ["2024-05-30", "2024-06-28", "2024-07-30"],
    "xccy_spot": "2024-04-30", "xccy": ["2024-05-31", "2024-06-28", "2024-07-31"],
    "fwd_spot": "2024-06-21", "fwd_start": "2024-09-23", "fwd_end": "2025-09-22",
}
ql_dates = {
    "aud_spot": iso(spot1), "aud": [iso(d) for d in aud],
    "xccy_spot": iso(spot2), "xccy": [iso(d) for d in xccy],
    "fwd_spot": iso(spot3), "fwd_start": iso(start3), "fwd_end": iso(end3),
}
straight = [iso(spot1 + ql.Period(m, ql.Months)) for m in (1, 2, 3)]
rows = [{"tenor": f"{m}M", "straight_run": s, "straight_run_weekday": dt.date.fromisoformat(s).strftime("%A"),
         "aud_swap": a, "xccy_eom": x}
        for m, s, a, x in zip((1, 2, 3), straight, ql_dates["aud"], ql_dates["xccy"])]
out["dates"] = {"trade": iso(trade1), "spot": iso(spot1), "rows": rows,
                "fwd": {"trade": iso(trade3), "spot": iso(spot3), "start": iso(start3), "end": iso(end3)},
                "all_match_afma": ql_dates == AFMA_PRINTED}
print("QuantLib reproduces AFMA's examples:", out["dates"]["all_match_afma"])
pd.DataFrame(rows)

## 6. Which product uses which rate

| Product | Floating rate | Payments | Source |
|---|---|---|---|
| AONIA OIS | AONIA, compounded daily | once at maturity up to 12 months, then annual | AFMA §2.2, §3.7 |
| BBSW swap, up to 3 years | 3-month BBSW | quarterly | AFMA §2.2, §2.3, §3.7 |
| BBSW swap, 4 years and longer | 6-month BBSW | semi-annual | AFMA §2.2, §2.3, §3.7 |
| 3s6s basis swap | 3-month vs 6-month BBSW | quarterly vs semi-annual | AFMA §2.3 |
| Cash/BBSW basis swap (BOB) | AONIA vs BBSW | quarterly | AFMA §2.3, §3.17 |
| FRA | BBSW for the period | once, at the start of the period | AFMA §2.1, §3.9 |

In [ ]:
out["products"] = [
    {"product": "AONIA OIS", "rate": "AONIA, compounded daily"},
    {"product": "BBSW swap, up to 3 years", "rate": "3-month BBSW, quarterly"},
    {"product": "BBSW swap, 4 years and longer", "rate": "6-month BBSW, semi-annual"},
    {"product": "3s6s basis swap", "rate": "3-month vs 6-month BBSW"},
    {"product": "Cash/BBSW basis swap", "rate": "AONIA vs BBSW"},
    {"product": "FRA", "rate": "BBSW for the period"},
]
pd.DataFrame(out["products"])

## 7. Export for the video

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")